# Notebook 3 — Locked External Validation on MC-MED

This notebook performs external validation of the locked MIMIC-IV ESI-3 abdominal-pain disposition model on MC-MED.

The binary outcome is HOME = 0 versus ADMITTED = 1.

Model development was completed in Notebook 2. The final model consists of a training-fitted XGBoost pipeline, validation-fitted Platt probability calibrator and validation-derived decision threshold of 0.282.

Notebook 3 is strictly an external-validation notebook. MC-MED must not be used to retrain the model, refit preprocessing, tune hyperparameters, select features, recalibrate probabilities or change the decision threshold.

The external-validation workflow is: verify the frozen Notebook 2 hand-off, load and validate the MC-MED external cohort, align its 2,495 raw features to the locked feature universe, apply the saved preprocessing and XGBoost model, apply the saved Platt calibrator and threshold, quantify external discrimination/calibration/classification performance with bootstrap confidence intervals, and subsequently perform matched cross-dataset SHAP analyses.

The 2,495 grouped raw features form the clinically interpretable feature universe. The 2,498 prepared MIMIC-IV model columns reflect the training-fitted one-hot encoding of sex/gender and arrival transport and are retained only as a technical model-input representation.

In [ ]:



from datetime import datetime, timezone
from pathlib import Path

import gc
import hashlib
import json
import os
import platform
import subprocess
import sys
import time
import warnings


warnings.filterwarnings(
    "ignore"
)
from scipy.optimize import minimize


import numpy as np
import pandas as pd



from scipy import sparse



import joblib



import sklearn

from sklearn.calibration import calibration_curve

from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    log_loss,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder



import xgboost as xgb

from xgboost import XGBClassifier



import matplotlib.pyplot as plt



import shap



import plotly
import plotly.graph_objects as go



from IPython.display import display



GLOBAL_RANDOM_SEED = 42

np.random.seed(
    GLOBAL_RANDOM_SEED
)


print(
    "Notebook 3 libraries imported successfully."
)

print(
    "Python:",
    platform.python_version(),
)

print(
    "pandas:",
    pd.__version__,
)

print(
    "NumPy:",
    np.__version__,
)

print(
    "scikit-learn:",
    sklearn.__version__,
)

print(
    "XGBoost:",
    xgb.__version__,
)

print(
    "SHAP:",
    shap.__version__,
)

print(
    "Plotly:",
    plotly.__version__,
)

print(
    "joblib:",
    joblib.__version__,
)

In [ ]:



PROJECT_ROOT = Path(
    os.environ.get(
        "ABDOMINAL_PAIN_ED_PROJECT_ROOT",
        Path.cwd(),
    )
).expanduser().resolve()


EXPECTED_PYTHON = Path(
    os.environ.get(
        "ABDOMINAL_PAIN_ED_PYTHON",
        sys.executable,
    )
).expanduser().resolve()


EXPECTED_TMUX_SESSION = (
    "External_val_paper"
)



CURRENT_WORKING_DIRECTORY = Path.cwd().resolve()


CURRENT_PYTHON = Path(
    sys.executable
).resolve()


TMUX_ENVIRONMENT = os.environ.get(
    "TMUX"
)



current_tmux_session = None


if TMUX_ENVIRONMENT:

    tmux_result = subprocess.run(
        [
            "tmux",
            "display-message",
            "-p",
            "#S",
        ],
        capture_output=True,
        text=True,
        check=False,
    )


    if (
        tmux_result.returncode
        ==
        0
    ):

        current_tmux_session = (
            tmux_result.stdout.strip()
        )



runtime_audit = pd.DataFrame(
    [
        {
            "check": (
                "Project root exists"
            ),
            "observed": str(
                PROJECT_ROOT
            ),
            "expected": str(
                PROJECT_ROOT
            ),
            "passed": (
                PROJECT_ROOT.exists()
            ),
        },
        {
            "check": (
                "Working directory"
            ),
            "observed": str(
                CURRENT_WORKING_DIRECTORY
            ),
            "expected": str(
                PROJECT_ROOT
            ),
            "passed": (
                CURRENT_WORKING_DIRECTORY
                ==
                PROJECT_ROOT
            ),
        },
        {
            "check": (
                "Python interpreter"
            ),
            "observed": str(
                CURRENT_PYTHON
            ),
            "expected": str(
                EXPECTED_PYTHON
            ),
            "passed": (
                CURRENT_PYTHON
                ==
                EXPECTED_PYTHON
            ),
        },
        {
            "check": (
                "Running inside tmux"
            ),
            "observed": (
                "YES"
                if TMUX_ENVIRONMENT
                else "NO"
            ),
            "expected": (
                "YES"
            ),
            "passed": bool(
                TMUX_ENVIRONMENT
            ),
        },
        {
            "check": (
                "tmux session"
            ),
            "observed": (
                current_tmux_session
            ),
            "expected": (
                EXPECTED_TMUX_SESSION
            ),
            "passed": (
                current_tmux_session
                ==
                EXPECTED_TMUX_SESSION
            ),
        },
    ]
)


display(
    runtime_audit
)



assert PROJECT_ROOT.exists(), (
    "Project root does not exist:\n"
    f"{PROJECT_ROOT}"
)


assert (
    CURRENT_WORKING_DIRECTORY
    ==
    PROJECT_ROOT
), (
    "Notebook is running from the wrong working directory.\n\n"
    f"Observed:\n{CURRENT_WORKING_DIRECTORY}\n\n"
    f"Expected:\n{PROJECT_ROOT}"
)


assert (
    CURRENT_PYTHON
    ==
    EXPECTED_PYTHON
), (
    "Notebook is using the wrong Python interpreter.\n\n"
    f"Observed:\n{CURRENT_PYTHON}\n\n"
    f"Expected:\n{EXPECTED_PYTHON}"
)


assert TMUX_ENVIRONMENT, (
    "This Jupyter kernel does not appear to have been launched "
    "from inside tmux."
)


assert (
    current_tmux_session
    ==
    EXPECTED_TMUX_SESSION
), (
    "Unexpected tmux session.\n\n"
    f"Observed: {current_tmux_session}\n"
    f"Expected: {EXPECTED_TMUX_SESSION}"
)



print(
    "NOTEBOOK 3 RUNTIME VERIFIED"
)

print(
    "Project root:",
    PROJECT_ROOT,
)

print(
    "Python interpreter:",
    CURRENT_PYTHON,
)

print(
    "tmux session:",
    current_tmux_session,
)

print(
    "Kernel environment correct: YES"
)

## Frozen Notebook 2 model hand-off

The completed Notebook 2 artifacts are loaded and integrity-checked before any MC-MED external data are accessed.

The saved training-fitted XGBoost pipeline, validation-fitted Platt calibrator, validation-derived threshold, raw 2,495-feature order and reproducibility configuration constitute the immutable external-validation model package.

SHA-256 checksums are verified against the Notebook 2 artifact manifest. The external cohort is not loaded in this section.

In [ ]:



NOTEBOOK2_MODEL_DIR = (
    PROJECT_ROOT
    / "models"
    / "notebook2_locked"
)


NOTEBOOK2_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook2"
)


NOTEBOOK2_SHAP_DIR = (
    PROJECT_ROOT
    / "results"
    / "shap"
    / "notebook2"
)


NOTEBOOK2_PREDICTION_DIR = (
    PROJECT_ROOT
    / "results"
    / "predictions"
    / "notebook2"
)


notebook2_handoff_paths = {
    "locked_pipeline": (
        NOTEBOOK2_MODEL_DIR
        / "locked_xgboost_pipeline.joblib"
    ),
    "platt_calibrator": (
        NOTEBOOK2_MODEL_DIR
        / "platt_calibrator.joblib"
    ),
    "locked_model_package": (
        NOTEBOOK2_MODEL_DIR
        / "locked_model_package.joblib"
    ),
    "model_config": (
        NOTEBOOK2_MODEL_DIR
        / "notebook2_model_config.json"
    ),
    "core_artifact_manifest": (
        NOTEBOOK2_MODEL_DIR
        / "notebook2_artifact_manifest.csv"
    ),
    "shap_handoff_config": (
        NOTEBOOK2_SHAP_DIR
        / "notebook2_shap_handoff_config.json"
    ),
    "shap_handoff_manifest": (
        NOTEBOOK2_TABLE_DIR
        / "notebook2_shap_handoff_manifest.csv"
    ),
    "clinician_feature_universe": (
        NOTEBOOK2_TABLE_DIR
        / "clinician_comparison_feature_universe.csv"
    ),
    "mimic_grouped_shap_bundle": (
        NOTEBOOK2_SHAP_DIR
        / "internal_test_grouped_shap_analysis_bundle.npz"
    ),
}



handoff_file_audit = pd.DataFrame(
    [
        {
            "artifact": artifact_name,
            "exists": artifact_path.exists(),
            "path": str(
                artifact_path
            ),
        }
        for artifact_name, artifact_path
        in notebook2_handoff_paths.items()
    ]
)


display(
    handoff_file_audit
)


assert handoff_file_audit[
    "exists"
].all(), (
    "One or more required Notebook 2 hand-off artifacts "
    "are missing."
)



def file_sha256(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(
        path
    )


    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                chunk_size
            ),
            b"",
        ):

            digest.update(
                chunk
            )


    return digest.hexdigest()



notebook2_core_artifact_manifest = pd.read_csv(
    notebook2_handoff_paths[
        "core_artifact_manifest"
    ]
)


required_manifest_columns = {
    "artifact",
    "filename",
    "directory",
    "size_bytes",
    "sha256",
}


assert required_manifest_columns.issubset(
    notebook2_core_artifact_manifest.columns
)


core_checksum_records = []


for manifest_row in (
    notebook2_core_artifact_manifest
    .itertuples(
        index=False
    )
):

    artifact_path = (
        Path(
            manifest_row.directory
        )
        /
        manifest_row.filename
    )


    artifact_exists = (
        artifact_path.exists()
    )


    observed_sha256 = (
        file_sha256(
            artifact_path
        )
        if artifact_exists
        else None
    )


    checksum_match = (
        artifact_exists
        and
        observed_sha256
        ==
        manifest_row.sha256
    )


    core_checksum_records.append(
        {
            "artifact": (
                manifest_row.artifact
            ),
            "exists": (
                artifact_exists
            ),
            "checksum_match": (
                checksum_match
            ),
            "expected_sha256": (
                manifest_row.sha256
            ),
            "observed_sha256": (
                observed_sha256
            ),
            "path": str(
                artifact_path
            ),
        }
    )


notebook2_core_checksum_audit = pd.DataFrame(
    core_checksum_records
)


assert notebook2_core_checksum_audit[
    "exists"
].all(), (
    "At least one Notebook 2 artifact referenced by the "
    "manifest is missing."
)


assert notebook2_core_checksum_audit[
    "checksum_match"
].all(), (
    "At least one Notebook 2 artifact failed SHA-256 "
    "verification."
)



notebook2_shap_handoff_manifest = pd.read_csv(
    notebook2_handoff_paths[
        "shap_handoff_manifest"
    ]
)


shap_checksum_records = []


for manifest_row in (
    notebook2_shap_handoff_manifest
    .itertuples(
        index=False
    )
):

    artifact_path = (
        Path(
            manifest_row.directory
        )
        /
        manifest_row.filename
    )


    artifact_exists = (
        artifact_path.exists()
    )


    observed_sha256 = (
        file_sha256(
            artifact_path
        )
        if artifact_exists
        else None
    )


    checksum_match = (
        artifact_exists
        and
        observed_sha256
        ==
        manifest_row.sha256
    )


    shap_checksum_records.append(
        {
            "artifact": (
                manifest_row.artifact
            ),
            "exists": (
                artifact_exists
            ),
            "checksum_match": (
                checksum_match
            ),
            "expected_sha256": (
                manifest_row.sha256
            ),
            "observed_sha256": (
                observed_sha256
            ),
            "path": str(
                artifact_path
            ),
        }
    )


notebook2_shap_checksum_audit = pd.DataFrame(
    shap_checksum_records
)


assert notebook2_shap_checksum_audit[
    "exists"
].all()


assert notebook2_shap_checksum_audit[
    "checksum_match"
].all(), (
    "At least one final Notebook 2 SHAP hand-off artifact "
    "failed SHA-256 verification."
)



locked_external_pipeline = joblib.load(
    notebook2_handoff_paths[
        "locked_pipeline"
    ]
)


locked_external_calibrator = joblib.load(
    notebook2_handoff_paths[
        "platt_calibrator"
    ]
)


locked_external_model_package = joblib.load(
    notebook2_handoff_paths[
        "locked_model_package"
    ]
)


with notebook2_handoff_paths[
    "model_config"
].open(
    "r",
    encoding="utf-8",
) as file:

    notebook2_model_config = json.load(
        file
    )


with notebook2_handoff_paths[
    "shap_handoff_config"
].open(
    "r",
    encoding="utf-8",
) as file:

    notebook2_shap_handoff_config = (
        json.load(
            file
        )
    )



LOCKED_ALGORITHM = (
    locked_external_model_package[
        "algorithm"
    ]
)


LOCKED_RAW_FEATURE_ORDER = list(
    locked_external_model_package[
        "raw_feature_order"
    ]
)


LOCKED_RAW_FEATURE_COUNT = int(
    locked_external_model_package[
        "raw_feature_count"
    ]
)


LOCKED_FEATURE_ORDER_SHA256 = (
    locked_external_model_package[
        "feature_order_sha256"
    ]
)


LOCKED_DECISION_THRESHOLD = float(
    locked_external_model_package[
        "decision_threshold"
    ]
)


LOCKED_CALIBRATION_METHOD = (
    locked_external_model_package[
        "calibration_method"
    ]
)


locked_external_preprocessor = (
    locked_external_pipeline
    .named_steps[
        "preprocessor"
    ]
)


locked_external_xgboost = (
    locked_external_pipeline
    .named_steps[
        "model"
    ]
)



calculated_feature_order_sha256 = hashlib.sha256(
    "\n".join(
        LOCKED_RAW_FEATURE_ORDER
    ).encode(
        "utf-8"
    )
).hexdigest()



EXPECTED_FEATURE_ORDER_SHA256 = (
    "9a4a106d770e2ce6edc5527a7b0f1b58"
    "e18e6db126a5f44a028d73eb54ae0587"
)


assert (
    LOCKED_ALGORITHM
    ==
    "XGBoost"
)


assert isinstance(
    locked_external_pipeline,
    Pipeline,
)


assert isinstance(
    locked_external_xgboost,
    XGBClassifier,
)


assert hasattr(
    locked_external_preprocessor,
    "transform"
)


assert hasattr(
    locked_external_xgboost,
    "predict_proba"
)


assert hasattr(
    locked_external_calibrator,
    "predict_proba"
)


assert LOCKED_RAW_FEATURE_COUNT == 2495


assert len(
    LOCKED_RAW_FEATURE_ORDER
) == 2495


assert len(
    set(
        LOCKED_RAW_FEATURE_ORDER
    )
) == 2495


assert (
    LOCKED_FEATURE_ORDER_SHA256
    ==
    EXPECTED_FEATURE_ORDER_SHA256
)


assert (
    calculated_feature_order_sha256
    ==
    EXPECTED_FEATURE_ORDER_SHA256
)


assert (
    notebook2_model_config[
        "feature_order_sha256"
    ]
    ==
    EXPECTED_FEATURE_ORDER_SHA256
)


assert (
    notebook2_shap_handoff_config[
        "feature_order_sha256"
    ]
    ==
    EXPECTED_FEATURE_ORDER_SHA256
)


assert np.isclose(
    LOCKED_DECISION_THRESHOLD,
    0.282,
    rtol=0.0,
    atol=1e-12,
)


assert (
    LOCKED_CALIBRATION_METHOD
    ==
    "Platt scaling"
)


assert (
    notebook2_model_config[
        "development_boundaries"
    ][
        "model_development_locked"
    ]
    is True
)


assert (
    notebook2_model_config[
        "development_boundaries"
    ][
        "mcmed_used"
    ]
    is False
)


assert (
    notebook2_shap_handoff_config[
        "grouped_raw_features"
    ]
    ==
    2495
)


assert (
    notebook2_shap_handoff_config[
        "prepared_model_columns"
    ]
    ==
    2498
)



LOCKED_CALIBRATOR_INTERCEPT = float(
    locked_external_calibrator
    .intercept_[
        0
    ]
)


LOCKED_CALIBRATOR_SLOPE = float(
    locked_external_calibrator
    .coef_[
        0
    ][
        0
    ]
)


assert np.isclose(
    LOCKED_CALIBRATOR_INTERCEPT,
    float(
        notebook2_model_config[
            "calibration"
        ][
            "final_intercept"
        ]
    ),
    rtol=0.0,
    atol=1e-10,
)


assert np.isclose(
    LOCKED_CALIBRATOR_SLOPE,
    float(
        notebook2_model_config[
            "calibration"
        ][
            "final_slope"
        ]
    ),
    rtol=0.0,
    atol=1e-10,
)



clinician_comparison_feature_universe = pd.read_csv(
    notebook2_handoff_paths[
        "clinician_feature_universe"
    ]
)


assert len(
    clinician_comparison_feature_universe
) == 2495


assert (
    clinician_comparison_feature_universe[
        "source_raw_feature"
    ]
    .tolist()
    ==
    LOCKED_RAW_FEATURE_ORDER
)



notebook3_frozen_handoff_audit = pd.DataFrame(
    [
        {
            "algorithm": (
                LOCKED_ALGORITHM
            ),
            "raw_feature_count": (
                LOCKED_RAW_FEATURE_COUNT
            ),
            "prepared_mimic_model_columns": (
                notebook2_shap_handoff_config[
                    "prepared_model_columns"
                ]
            ),
            "calibration": (
                LOCKED_CALIBRATION_METHOD
            ),
            "calibrator_intercept": (
                LOCKED_CALIBRATOR_INTERCEPT
            ),
            "calibrator_slope": (
                LOCKED_CALIBRATOR_SLOPE
            ),
            "locked_threshold": (
                LOCKED_DECISION_THRESHOLD
            ),
            "feature_order_sha256": (
                LOCKED_FEATURE_ORDER_SHA256
            ),
            "core_checksums_passed": bool(
                notebook2_core_checksum_audit[
                    "checksum_match"
                ].all()
            ),
            "shap_handoff_checksums_passed": bool(
                notebook2_shap_checksum_audit[
                    "checksum_match"
                ].all()
            ),
            "model_development_locked": True,
            "mcmed_loaded": False,
        }
    ]
)


display(
    notebook3_frozen_handoff_audit
)


print(
    "\nNOTEBOOK 2 FROZEN HAND-OFF VERIFIED"
)

print(
    "Algorithm:",
    LOCKED_ALGORITHM,
)

print(
    "Raw features:",
    LOCKED_RAW_FEATURE_COUNT,
)

print(
    "Prepared MIMIC model columns:",
    notebook2_shap_handoff_config[
        "prepared_model_columns"
    ],
)

print(
    "Calibration:",
    LOCKED_CALIBRATION_METHOD,
)

print(
    "Calibration intercept:",
    round(
        LOCKED_CALIBRATOR_INTERCEPT,
        6,
    ),
)

print(
    "Calibration slope:",
    round(
        LOCKED_CALIBRATOR_SLOPE,
        6,
    ),
)

print(
    "Locked threshold:",
    round(
        LOCKED_DECISION_THRESHOLD,
        6,
    ),
)

print(
    "Feature-order SHA-256:",
    LOCKED_FEATURE_ORDER_SHA256,
)

print(
    "\nCore Notebook 2 checksums: PASSED"
)

print(
    "Grouped-SHAP hand-off checksums: PASSED"
)

print(
    "Model refitted: NO"
)

print(
    "Preprocessor refitted: NO"
)

print(
    "Calibrator refitted: NO"
)

print(
    "Threshold changed: NO"
)

print(
    "MC-MED loaded: NO"
)

## Frozen MC-MED external cohort loading and integrity verification

The external MC-MED cohort frozen in Notebook 1 is now loaded for the first time in Notebook 3.

Before any preprocessing or model prediction, the saved Parquet file is verified against its Notebook 1 SHA-256 checksum, expected cohort size, binary outcome support, encounter uniqueness and the locked 2,495-feature model-input universe.

This section performs data-integrity checks only. The training-fitted preprocessing pipeline, XGBoost model, Platt calibrator and decision threshold remain untouched, and no external predictions are generated.

In [ ]:



def locate_unique_project_file(
    filename,
):
    """
    Locate exactly one project file by filename.

    This avoids hard-coding the Notebook 1 processed-data
    subdirectory while still requiring an unambiguous artifact.
    """

    matches = [
        path.resolve()
        for path
        in PROJECT_ROOT.rglob(
            filename
        )
        if ".git" not in path.parts
    ]


    if len(
        matches
    ) == 0:

        raise FileNotFoundError(
            "Could not locate the required project artifact:\n"
            f"{filename}"
        )


    if len(
        matches
    ) > 1:

        raise RuntimeError(
            "More than one project artifact with this filename "
            "was found. The external-validation source must be "
            "unambiguous.\n\n"
            +
            "\n".join(
                str(
                    path
                )
                for path
                in matches
            )
        )


    return matches[0]



MCMED_EXTERNAL_PATH = (
    locate_unique_project_file(
        "mcmed_external.parquet"
    )
)


NOTEBOOK1_CHECKSUM_PATH = (
    locate_unique_project_file(
        "artifact_checksums.csv"
    )
)


print(
    "MC-MED external cohort:"
)

print(
    MCMED_EXTERNAL_PATH
)


print(
    "\nNotebook 1 checksum manifest:"
)

print(
    NOTEBOOK1_CHECKSUM_PATH
)



notebook1_artifact_checksums = pd.read_csv(
    NOTEBOOK1_CHECKSUM_PATH
)


print(
    "\nNotebook 1 checksum-manifest columns:"
)

print(
    notebook1_artifact_checksums.columns.tolist()
)



possible_filename_columns = [
    "filename",
    "file_name",
    "artifact",
    "path",
]


possible_sha_columns = [
    "sha256",
    "sha_256",
    "checksum",
    "sha256_checksum",
]


notebook1_filename_column = next(
    (
        column_name
        for column_name
        in possible_filename_columns
        if column_name
        in notebook1_artifact_checksums.columns
    ),
    None,
)


notebook1_sha_column = next(
    (
        column_name
        for column_name
        in possible_sha_columns
        if column_name
        in notebook1_artifact_checksums.columns
    ),
    None,
)


if notebook1_filename_column is None:

    raise RuntimeError(
        "Could not identify the filename column in the "
        "Notebook 1 artifact checksum manifest."
    )


if notebook1_sha_column is None:

    raise RuntimeError(
        "Could not identify the SHA-256 column in the "
        "Notebook 1 artifact checksum manifest."
    )



mcmed_checksum_matches = (
    notebook1_artifact_checksums.loc[
        notebook1_artifact_checksums[
            notebook1_filename_column
        ]
        .astype(str)
        .map(
            lambda value: Path(
                value
            ).name
        )
        .eq(
            "mcmed_external.parquet"
        )
    ]
    .copy()
)


assert len(
    mcmed_checksum_matches
) == 1, (
    "Expected exactly one checksum-manifest entry for "
    "mcmed_external.parquet."
)


expected_mcmed_sha256 = str(
    mcmed_checksum_matches[
        notebook1_sha_column
    ]
    .iloc[0]
)


observed_mcmed_sha256 = file_sha256(
    MCMED_EXTERNAL_PATH
)


assert (
    observed_mcmed_sha256
    ==
    expected_mcmed_sha256
), (
    "MC-MED external cohort failed Notebook 1 SHA-256 "
    "verification.\n\n"
    f"Expected:\n{expected_mcmed_sha256}\n\n"
    f"Observed:\n{observed_mcmed_sha256}"
)



mcmed_external = pd.read_parquet(
    MCMED_EXTERNAL_PATH,
    engine="pyarrow",
)


assert isinstance(
    mcmed_external,
    pd.DataFrame,
)



EXPECTED_MCMED_ROWS = 6707

EXPECTED_MCMED_HOME = 4872

EXPECTED_MCMED_ADMITTED = 1835


assert len(
    mcmed_external
) == EXPECTED_MCMED_ROWS, (
    "Unexpected MC-MED external cohort size.\n\n"
    f"Observed: {len(mcmed_external):,}\n"
    f"Expected: {EXPECTED_MCMED_ROWS:,}"
)



possible_outcome_columns = [
    "outcome_binary",
    "outcome",
]


mcmed_outcome_column = next(
    (
        column_name
        for column_name
        in possible_outcome_columns
        if column_name
        in mcmed_external.columns
    ),
    None,
)


if mcmed_outcome_column is None:

    raise RuntimeError(
        "Could not identify the frozen MC-MED outcome column."
    )


if (
    mcmed_outcome_column
    ==
    "outcome_binary"
):

    mcmed_external_outcome = pd.to_numeric(
        mcmed_external[
            mcmed_outcome_column
        ],
        errors="raise",
    ).astype(
        "int8"
    )


else:

    observed_outcome_values = (
        mcmed_external[
            mcmed_outcome_column
        ]
        .astype(str)
        .str.upper()
    )


    frozen_outcome_mapping = {
        "HOME": 0,
        "ADMITTED": 1,
    }


    unexpected_outcomes = sorted(
        set(
            observed_outcome_values.unique()
        )
        -
        set(
            frozen_outcome_mapping.keys()
        )
    )


    if unexpected_outcomes:

        raise RuntimeError(
            "Unexpected values found in the frozen external "
            "outcome column:\n"
            f"{unexpected_outcomes}"
        )


    mcmed_external_outcome = (
        observed_outcome_values
        .map(
            frozen_outcome_mapping
        )
        .astype(
            "int8"
        )
    )


assert set(
    mcmed_external_outcome.unique()
) == {
    0,
    1,
}


observed_mcmed_home = int(
    (
        mcmed_external_outcome
        ==
        0
    ).sum()
)


observed_mcmed_admitted = int(
    (
        mcmed_external_outcome
        ==
        1
    ).sum()
)


assert (
    observed_mcmed_home
    ==
    EXPECTED_MCMED_HOME
), (
    "Unexpected MC-MED HOME support.\n\n"
    f"Observed: {observed_mcmed_home:,}\n"
    f"Expected: {EXPECTED_MCMED_HOME:,}"
)


assert (
    observed_mcmed_admitted
    ==
    EXPECTED_MCMED_ADMITTED
), (
    "Unexpected MC-MED ADMITTED support.\n\n"
    f"Observed: {observed_mcmed_admitted:,}\n"
    f"Expected: {EXPECTED_MCMED_ADMITTED:,}"
)


assert (
    observed_mcmed_home
    +
    observed_mcmed_admitted
    ==
    EXPECTED_MCMED_ROWS
)



missing_locked_mcmed_features = [
    feature_name
    for feature_name
    in LOCKED_RAW_FEATURE_ORDER
    if feature_name
    not in mcmed_external.columns
]


assert not missing_locked_mcmed_features, (
    "MC-MED is missing one or more features required by the "
    "locked MIMIC-IV model.\n\n"
    f"Missing feature count: "
    f"{len(missing_locked_mcmed_features)}\n\n"
    +
    "\n".join(
        missing_locked_mcmed_features[
            :50
        ]
    )
)


mcmed_locked_feature_count = sum(
    feature_name
    in mcmed_external.columns
    for feature_name
    in LOCKED_RAW_FEATURE_ORDER
)


assert (
    mcmed_locked_feature_count
    ==
    2495
)



X_mcmed_external_raw = (
    mcmed_external.loc[
        :,
        LOCKED_RAW_FEATURE_ORDER,
    ]
    .copy()
)


y_mcmed_external = (
    mcmed_external_outcome
    .copy()
)


assert X_mcmed_external_raw.shape == (
    6707,
    2495,
)


assert len(
    y_mcmed_external
) == 6707


assert X_mcmed_external_raw.index.equals(
    y_mcmed_external.index
)


assert X_mcmed_external_raw.columns.tolist() == (
    LOCKED_RAW_FEATURE_ORDER
)



possible_mcmed_identifier_columns = [
    "CSN",
    "csn",
    "Visit_no",
    "visit_no",
    "MRN",
    "mrn",
]


present_mcmed_identifier_columns = [
    column_name
    for column_name
    in possible_mcmed_identifier_columns
    if column_name
    in mcmed_external.columns
]


mcmed_identifier_audit_records = []


for identifier_column in (
    present_mcmed_identifier_columns
):

    identifier_series = (
        mcmed_external[
            identifier_column
        ]
    )


    mcmed_identifier_audit_records.append(
        {
            "identifier": (
                identifier_column
            ),
            "rows": int(
                len(
                    identifier_series
                )
            ),
            "non_null": int(
                identifier_series
                .notna()
                .sum()
            ),
            "unique_non_null": int(
                identifier_series
                .nunique(
                    dropna=True
                )
            ),
            "duplicated_non_null_rows": int(
                identifier_series[
                    identifier_series.notna()
                ]
                .duplicated(
                    keep=False
                )
                .sum()
            ),
        }
    )


mcmed_identifier_audit = pd.DataFrame(
    mcmed_identifier_audit_records
)



assert not mcmed_external.index.duplicated().any(), (
    "The frozen MC-MED external DataFrame index contains "
    "duplicate rows."
)



mcmed_feature_structure_audit = pd.DataFrame(
    [
        {
            "external_rows": int(
                len(
                    mcmed_external
                )
            ),
            "home": (
                observed_mcmed_home
            ),
            "admitted": (
                observed_mcmed_admitted
            ),
            "admitted_prevalence": float(
                y_mcmed_external.mean()
            ),
            "locked_raw_features_expected": (
                LOCKED_RAW_FEATURE_COUNT
            ),
            "locked_raw_features_present": (
                mcmed_locked_feature_count
            ),
            "ordered_raw_input_columns": int(
                X_mcmed_external_raw.shape[
                    1
                ]
            ),
            "checksum_verified": True,
            "model_prediction_generated": False,
        }
    ]
)



notebook3_mcmed_loading_audit = pd.DataFrame(
    [
        {
            "external_dataset": (
                "MC-MED"
            ),
            "source_file": (
                MCMED_EXTERNAL_PATH.name
            ),
            "rows": (
                len(
                    mcmed_external
                )
            ),
            "home": (
                observed_mcmed_home
            ),
            "admitted": (
                observed_mcmed_admitted
            ),
            "prevalence": float(
                y_mcmed_external.mean()
            ),
            "raw_features": (
                X_mcmed_external_raw.shape[
                    1
                ]
            ),
            "source_sha256": (
                observed_mcmed_sha256
            ),
            "source_checksum_passed": True,
            "preprocessing_applied": False,
            "model_prediction_generated": False,
            "calibration_applied": False,
            "threshold_applied": False,
            "model_refitted": False,
            "preprocessor_refitted": False,
            "calibrator_refitted": False,
        }
    ]
)



print(
    "MC-MED external cohort integrity summary:"
)

display(
    mcmed_feature_structure_audit
)


if not mcmed_identifier_audit.empty:

    print(
        "\nMC-MED identifier audit:"
    )

    display(
        mcmed_identifier_audit
    )


print(
    "\nExternal-validation boundary audit:"
)

display(
    notebook3_mcmed_loading_audit
)


print(
    "\nFROZEN MC-MED EXTERNAL COHORT LOADED AND VERIFIED"
)

print(
    "Rows:",
    f"{len(mcmed_external):,}",
)

print(
    "HOME:",
    f"{observed_mcmed_home:,}",
)

print(
    "ADMITTED:",
    f"{observed_mcmed_admitted:,}",
)

print(
    "Admission prevalence:",
    round(
        float(
            y_mcmed_external.mean()
        ),
        6,
    ),
)

print(
    "Locked raw features present:",
    mcmed_locked_feature_count,
)

print(
    "Raw feature order aligned:",
    "YES",
)

print(
    "Notebook 1 source checksum:",
    "PASSED",
)

print(
    "\nPreprocessing applied: NO"
)

print(
    "Model probabilities generated: NO"
)

print(
    "Calibration applied: NO"
)

print(
    "Threshold applied: NO"
)

print(
    "Model refitted: NO"
)

print(
    "Preprocessor refitted: NO"
)

print(
    "Calibrator refitted: NO"
)

### MC-MED feature compatibility audit before external prediction

Before applying the locked XGBoost model, the frozen MC-MED raw feature matrix is checked for compatibility with the training-fitted MIMIC-IV preprocessing pipeline.

The audit verifies numeric coercibility, categorical levels, unseen categories, binary-feature domains, missingness and the exact output dimensionality of the frozen preprocessor. Only deterministic dtype standardisation is performed; no statistics, categories, imputation values or model parameters are learned from MC-MED.

Unseen categorical levels are documented and handled by the already-fitted `OneHotEncoder(handle_unknown="ignore")`. Missingness differences are treated as external dataset characteristics rather than reasons to modify the locked model.

No model probabilities, calibrated probabilities or classifications are generated in this section.

In [ ]:



required_compatibility_objects = [
    "PROJECT_ROOT",
    "X_mcmed_external_raw",
    "y_mcmed_external",
    "locked_external_preprocessor",
    "locked_external_xgboost",
    "LOCKED_RAW_FEATURE_ORDER",
    "LOCKED_RAW_FEATURE_COUNT",
]


missing_compatibility_objects = [
    object_name
    for object_name in required_compatibility_objects
    if object_name not in globals()
]


if missing_compatibility_objects:

    raise RuntimeError(
        "Load and verify the frozen MC-MED cohort and "
        "Notebook 2 hand-off first.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_compatibility_objects
        )
    )


assert X_mcmed_external_raw.shape == (
    6707,
    2495,
)


assert X_mcmed_external_raw.columns.tolist() == (
    LOCKED_RAW_FEATURE_ORDER
)


assert len(
    y_mcmed_external
) == 6707



frozen_transformer_lookup = {
    transformer_name: (
        transformer_object,
        transformer_columns,
    )
    for (
        transformer_name,
        transformer_object,
        transformer_columns,
    )
    in locked_external_preprocessor.transformers_
    if transformer_name != "remainder"
}


required_transformer_names = {
    "s0_numeric",
    "s0_categorical",
    "binary_features",
}


assert required_transformer_names.issubset(
    frozen_transformer_lookup.keys()
), (
    "The frozen preprocessor does not contain the expected "
    "feature-group transformers."
)


LOCKED_S0_NUMERIC_FEATURES = list(
    frozen_transformer_lookup[
        "s0_numeric"
    ][
        1
    ]
)


LOCKED_S0_CATEGORICAL_FEATURES = list(
    frozen_transformer_lookup[
        "s0_categorical"
    ][
        1
    ]
)


LOCKED_BINARY_FEATURES = list(
    frozen_transformer_lookup[
        "binary_features"
    ][
        1
    ]
)


assert len(
    LOCKED_S0_NUMERIC_FEATURES
) == 7


assert len(
    LOCKED_S0_CATEGORICAL_FEATURES
) == 2


assert len(
    LOCKED_BINARY_FEATURES
) == 2486


assert set(
    LOCKED_S0_CATEGORICAL_FEATURES
) == {
    "sex_gender",
    "arrival_transport_grouped",
}



LOCKED_PREPROCESSOR_RAW_FEATURES = (
    LOCKED_S0_NUMERIC_FEATURES
    +
    LOCKED_S0_CATEGORICAL_FEATURES
    +
    LOCKED_BINARY_FEATURES
)


assert len(
    LOCKED_PREPROCESSOR_RAW_FEATURES
) == 2495


assert len(
    set(
        LOCKED_PREPROCESSOR_RAW_FEATURES
    )
) == 2495, (
    "A raw feature appears in more than one frozen "
    "preprocessing group."
)


assert set(
    LOCKED_PREPROCESSOR_RAW_FEATURES
) == set(
    LOCKED_RAW_FEATURE_ORDER
), (
    "The frozen preprocessing groups do not contain exactly "
    "the same 2,495 features as the locked raw-feature universe."
)


missing_from_preprocessor = sorted(
    set(
        LOCKED_RAW_FEATURE_ORDER
    )
    -
    set(
        LOCKED_PREPROCESSOR_RAW_FEATURES
    )
)


unexpected_in_preprocessor = sorted(
    set(
        LOCKED_PREPROCESSOR_RAW_FEATURES
    )
    -
    set(
        LOCKED_RAW_FEATURE_ORDER
    )
)


assert not missing_from_preprocessor


assert not unexpected_in_preprocessor


print(
    "Frozen preprocessing feature partition: VERIFIED"
)

print(
    "Raw locked feature order:",
    len(
        LOCKED_RAW_FEATURE_ORDER
    ),
)

print(
    "Features assigned to frozen transformers:",
    len(
        LOCKED_PREPROCESSOR_RAW_FEATURES
    ),
)

print(
    "Missing from frozen preprocessor:",
    len(
        missing_from_preprocessor
    ),
)

print(
    "Unexpected in frozen preprocessor:",
    len(
        unexpected_in_preprocessor
    ),
)

print(
    "Exact set match:",
    "YES",
)


print(
    "Frozen preprocessing groups recovered:"
)

print(
    "Numeric S0:",
    len(
        LOCKED_S0_NUMERIC_FEATURES
    ),
)

print(
    "Categorical S0:",
    len(
        LOCKED_S0_CATEGORICAL_FEATURES
    ),
)

print(
    "Binary S1/S2:",
    len(
        LOCKED_BINARY_FEATURES
    ),
)



frozen_numeric_imputer = (
    locked_external_preprocessor
    .named_transformers_[
        "s0_numeric"
    ]
    .named_steps[
        "median_imputer"
    ]
)


frozen_categorical_pipeline = (
    locked_external_preprocessor
    .named_transformers_[
        "s0_categorical"
    ]
)


frozen_categorical_imputer = (
    frozen_categorical_pipeline
    .named_steps[
        "missing_category_imputer"
    ]
)


frozen_categorical_encoder = (
    frozen_categorical_pipeline
    .named_steps[
        "one_hot_encoder"
    ]
)


frozen_numeric_statistics_before = (
    np.asarray(
        frozen_numeric_imputer.statistics_
    )
    .copy()
)


frozen_categories_before = [
    np.asarray(
        category_levels
    ).copy()
    for category_levels
    in frozen_categorical_encoder.categories_
]


LOCKED_PREPARED_FEATURE_COUNT = (
    len(
        LOCKED_S0_NUMERIC_FEATURES
    )
    +
    sum(
        len(
            category_levels
        )
        for category_levels
        in frozen_categories_before
    )
    +
    len(
        LOCKED_BINARY_FEATURES
    )
)


assert LOCKED_PREPARED_FEATURE_COUNT == 2498


assert getattr(
    locked_external_xgboost,
    "n_features_in_",
    LOCKED_PREPARED_FEATURE_COUNT,
) == LOCKED_PREPARED_FEATURE_COUNT



numeric_compatibility_records = []


for feature_name in (
    LOCKED_S0_NUMERIC_FEATURES
):

    original_series = (
        X_mcmed_external_raw[
            feature_name
        ]
    )


    numeric_series = pd.to_numeric(
        original_series,
        errors="coerce",
    )


    originally_non_missing = (
        original_series.notna()
    )


    coercion_failure_mask = (
        originally_non_missing
        &
        numeric_series.isna()
    )


    numeric_compatibility_records.append(
        {
            "feature": (
                feature_name
            ),
            "original_dtype": str(
                original_series.dtype
            ),
            "rows": int(
                len(
                    original_series
                )
            ),
            "original_missing": int(
                original_series
                .isna()
                .sum()
            ),
            "non_missing_values": int(
                originally_non_missing.sum()
            ),
            "numeric_coercion_failures": int(
                coercion_failure_mask.sum()
            ),
            "minimum_numeric": (
                float(
                    numeric_series.min()
                )
                if numeric_series.notna().any()
                else np.nan
            ),
            "maximum_numeric": (
                float(
                    numeric_series.max()
                )
                if numeric_series.notna().any()
                else np.nan
            ),
            "compatible": (
                coercion_failure_mask.sum()
                ==
                0
            ),
        }
    )


mcmed_numeric_compatibility_audit = pd.DataFrame(
    numeric_compatibility_records
)


assert mcmed_numeric_compatibility_audit[
    "compatible"
].all(), (
    "At least one MC-MED numeric feature contains non-missing "
    "values that cannot be converted to numeric form."
)



categorical_compatibility_records = []


for feature_name, training_categories in zip(
    LOCKED_S0_CATEGORICAL_FEATURES,
    frozen_categories_before,
):

    external_series = (
        X_mcmed_external_raw[
            feature_name
        ]
        .astype(
            "string"
        )
    )


    external_non_missing_values = (
        external_series[
            external_series.notna()
        ]
        .astype(str)
    )


    observed_external_categories = sorted(
        external_non_missing_values
        .unique()
        .tolist()
    )


    training_category_strings = [
        str(
            category
        )
        for category
        in training_categories
    ]


    unseen_external_categories = sorted(
        set(
            observed_external_categories
        )
        -
        set(
            training_category_strings
        )
    )


    for external_category in (
        observed_external_categories
    ):

        category_count = int(
            (
                external_non_missing_values
                ==
                external_category
            ).sum()
        )


        categorical_compatibility_records.append(
            {
                "feature": (
                    feature_name
                ),
                "category": (
                    external_category
                ),
                "external_count": (
                    category_count
                ),
                "external_percent": float(
                    100
                    *
                    category_count
                    /
                    len(
                        external_series
                    )
                ),
                "seen_in_mimic_training": (
                    external_category
                    in training_category_strings
                ),
                "handled_by_encoder": True,
            }
        )


    categorical_compatibility_records.append(
        {
            "feature": (
                feature_name
            ),
            "category": (
                "__MISSING__"
            ),
            "external_count": int(
                external_series
                .isna()
                .sum()
            ),
            "external_percent": float(
                100
                *
                external_series
                .isna()
                .mean()
            ),
            "seen_in_mimic_training": (
                "__MISSING__"
                in training_category_strings
            ),
            "handled_by_encoder": True,
        }
    )


mcmed_categorical_compatibility_audit = (
    pd.DataFrame(
        categorical_compatibility_records
    )
)


mcmed_unseen_category_summary = (
    mcmed_categorical_compatibility_audit.loc[
        (
            mcmed_categorical_compatibility_audit[
                "category"
            ]
            !=
            "__MISSING__"
        )
        &
        (
            ~mcmed_categorical_compatibility_audit[
                "seen_in_mimic_training"
            ]
        )
    ]
    .copy()
)



binary_domain_records = []


binary_domain_violation_examples = []


for feature_name in (
    LOCKED_BINARY_FEATURES
):

    original_series = (
        X_mcmed_external_raw[
            feature_name
        ]
    )


    numeric_series = pd.to_numeric(
        original_series,
        errors="coerce",
    )


    originally_non_missing = (
        original_series.notna()
    )


    coercion_failure_mask = (
        originally_non_missing
        &
        numeric_series.isna()
    )


    valid_binary_mask = (
        numeric_series.isna()
        |
        numeric_series.isin(
            [
                0,
                1,
            ]
        )
    )


    invalid_domain_mask = (
        ~valid_binary_mask
    )


    invalid_count = int(
        invalid_domain_mask.sum()
        +
        coercion_failure_mask.sum()
    )


    unique_non_missing_values = (
        numeric_series[
            numeric_series.notna()
        ]
        .unique()
    )


    binary_domain_records.append(
        {
            "feature": (
                feature_name
            ),
            "original_dtype": str(
                original_series.dtype
            ),
            "missing_count": int(
                numeric_series
                .isna()
                .sum()
            ),
            "zero_count": int(
                (
                    numeric_series
                    ==
                    0
                ).sum()
            ),
            "one_count": int(
                (
                    numeric_series
                    ==
                    1
                ).sum()
            ),
            "coercion_failures": int(
                coercion_failure_mask.sum()
            ),
            "invalid_binary_values": int(
                invalid_domain_mask.sum()
            ),
            "domain_valid": (
                invalid_count
                ==
                0
            ),
        }
    )


    if invalid_count > 0:

        invalid_values = (
            original_series.loc[
                (
                    coercion_failure_mask
                    |
                    invalid_domain_mask
                )
            ]
            .drop_duplicates()
            .astype(str)
            .head(
                10
            )
            .tolist()
        )


        binary_domain_violation_examples.append(
            {
                "feature": (
                    feature_name
                ),
                "examples": (
                    " | ".join(
                        invalid_values
                    )
                ),
            }
        )


mcmed_binary_domain_audit = pd.DataFrame(
    binary_domain_records
)


mcmed_binary_domain_violations = pd.DataFrame(
    binary_domain_violation_examples
)


assert mcmed_binary_domain_audit[
    "domain_valid"
].all(), (
    "At least one locked binary MC-MED feature contains "
    "values outside {0, 1, missing}."
)



mcmed_missingness_compatibility = pd.DataFrame(
    {
        "feature": (
            LOCKED_RAW_FEATURE_ORDER
        ),
        "missing_count": [
            int(
                X_mcmed_external_raw[
                    feature_name
                ]
                .isna()
                .sum()
            )
            for feature_name
            in LOCKED_RAW_FEATURE_ORDER
        ],
    }
)


mcmed_missingness_compatibility[
    "missing_percent"
] = (
    100
    *
    mcmed_missingness_compatibility[
        "missing_count"
    ]
    /
    len(
        X_mcmed_external_raw
    )
)


feature_type_lookup = {}


for feature_name in (
    LOCKED_S0_NUMERIC_FEATURES
):

    feature_type_lookup[
        feature_name
    ] = "numeric_s0"


for feature_name in (
    LOCKED_S0_CATEGORICAL_FEATURES
):

    feature_type_lookup[
        feature_name
    ] = "categorical_s0"


for feature_name in (
    LOCKED_BINARY_FEATURES
):

    feature_type_lookup[
        feature_name
    ] = "binary_s1_s2"


mcmed_missingness_compatibility[
    "feature_type"
] = (
    mcmed_missingness_compatibility[
        "feature"
    ]
    .map(
        feature_type_lookup
    )
)


assert mcmed_missingness_compatibility[
    "feature_type"
].notna().all()



X_mcmed_external_standardised = (
    X_mcmed_external_raw
    .copy()
)


for feature_name in (
    LOCKED_S0_NUMERIC_FEATURES
):

    X_mcmed_external_standardised[
        feature_name
    ] = pd.to_numeric(
        X_mcmed_external_standardised[
            feature_name
        ],
        errors="raise",
    ).astype(
        np.float64
    )


for feature_name in (
    LOCKED_S0_CATEGORICAL_FEATURES
):

    categorical_series = (
        X_mcmed_external_standardised[
            feature_name
        ]
        .astype(
            "string"
        )
    )


    X_mcmed_external_standardised[
        feature_name
    ] = (
        categorical_series
        .astype(
            object
        )
        .where(
            categorical_series.notna(),
            np.nan,
        )
    )


for feature_name in (
    LOCKED_BINARY_FEATURES
):

    X_mcmed_external_standardised[
        feature_name
    ] = pd.to_numeric(
        X_mcmed_external_standardised[
            feature_name
        ],
        errors="raise",
    ).astype(
        np.float32
    )


assert X_mcmed_external_standardised.shape == (
    6707,
    2495,
)


assert X_mcmed_external_standardised.columns.tolist() == (
    LOCKED_RAW_FEATURE_ORDER
)


assert X_mcmed_external_standardised.index.equals(
    y_mcmed_external.index
)



preprocessor_transform_start = (
    time.perf_counter()
)


X_mcmed_external_transformed = (
    locked_external_preprocessor
    .transform(
        X_mcmed_external_standardised
    )
)


preprocessor_transform_seconds = (
    time.perf_counter()
    -
    preprocessor_transform_start
)


if sparse.issparse(
    X_mcmed_external_transformed
):

    X_mcmed_external_transformed = (
        X_mcmed_external_transformed
        .tocsr()
        .astype(
            np.float32
        )
    )


    transformed_values_finite = (
        np.isfinite(
            X_mcmed_external_transformed.data
        ).all()
    )

else:

    X_mcmed_external_transformed = np.asarray(
        X_mcmed_external_transformed,
        dtype=np.float32,
    )


    transformed_values_finite = (
        np.isfinite(
            X_mcmed_external_transformed
        ).all()
    )


assert X_mcmed_external_transformed.shape == (
    6707,
    2498,
), (
    "Frozen preprocessor produced an unexpected external "
    "feature width.\n\n"
    f"Observed: {X_mcmed_external_transformed.shape}\n"
    "Expected: (6707, 2498)"
)


assert transformed_values_finite, (
    "The frozen preprocessor produced NaN or infinite values "
    "for MC-MED."
)



frozen_numeric_statistics_after = (
    np.asarray(
        frozen_numeric_imputer.statistics_
    )
)


frozen_categories_after = [
    np.asarray(
        category_levels
    )
    for category_levels
    in frozen_categorical_encoder.categories_
]


assert np.array_equal(
    frozen_numeric_statistics_before,
    frozen_numeric_statistics_after,
    equal_nan=True,
), (
    "Frozen numeric-imputer statistics changed after MC-MED "
    "transform."
)


assert len(
    frozen_categories_before
) == len(
    frozen_categories_after
)


for (
    categories_before,
    categories_after,
) in zip(
    frozen_categories_before,
    frozen_categories_after,
):

    assert np.array_equal(
        categories_before,
        categories_after,
    ), (
        "Frozen OneHotEncoder categories changed after "
        "MC-MED transform."
    )



frozen_categorical_width_records = []


for feature_name, category_levels in zip(
    LOCKED_S0_CATEGORICAL_FEATURES,
    frozen_categories_after,
):

    frozen_categorical_width_records.append(
        {
            "feature": (
                feature_name
            ),
            "training_categories": int(
                len(
                    category_levels
                )
            ),
            "training_category_levels": (
                " | ".join(
                    [
                        str(
                            category
                        )
                        for category
                        in category_levels
                    ]
                )
            ),
        }
    )


mcmed_prepared_categorical_audit = pd.DataFrame(
    frozen_categorical_width_records
)


assert (
    mcmed_prepared_categorical_audit[
        "training_categories"
    ].sum()
    ==
    5
)



mcmed_feature_compatibility_summary = pd.DataFrame(
    [
        {
            "external_rows": 6707,
            "raw_features": 2495,
            "numeric_s0_features": int(
                len(
                    LOCKED_S0_NUMERIC_FEATURES
                )
            ),
            "categorical_s0_features": int(
                len(
                    LOCKED_S0_CATEGORICAL_FEATURES
                )
            ),
            "binary_s1_s2_features": int(
                len(
                    LOCKED_BINARY_FEATURES
                )
            ),
            "numeric_coercion_failures": int(
                mcmed_numeric_compatibility_audit[
                    "numeric_coercion_failures"
                ].sum()
            ),
            "binary_domain_violations": int(
                (
                    ~mcmed_binary_domain_audit[
                        "domain_valid"
                    ]
                ).sum()
            ),
            "unseen_categorical_levels": int(
                len(
                    mcmed_unseen_category_summary
                )
            ),
            "prepared_features_after_frozen_transform": int(
                X_mcmed_external_transformed.shape[
                    1
                ]
            ),
            "transformed_values_finite": bool(
                transformed_values_finite
            ),
            "numeric_imputer_unchanged": True,
            "categorical_encoder_unchanged": True,
            "preprocessor_refitted": False,
            "model_prediction_generated": False,
        }
    ]
)



NOTEBOOK3_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook3"
)


NOTEBOOK3_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


compatibility_output_paths = {
    "summary": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_feature_compatibility_summary.csv"
    ),
    "numeric": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_numeric_compatibility_audit.csv"
    ),
    "categorical": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_categorical_compatibility_audit.csv"
    ),
    "unseen_categories": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_unseen_category_summary.csv"
    ),
    "binary_domain": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_binary_domain_audit.csv"
    ),
    "missingness": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_missingness_compatibility.csv"
    ),
    "prepared_categorical": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_prepared_categorical_audit.csv"
    ),
}


mcmed_feature_compatibility_summary.to_csv(
    compatibility_output_paths[
        "summary"
    ],
    index=False,
)


mcmed_numeric_compatibility_audit.to_csv(
    compatibility_output_paths[
        "numeric"
    ],
    index=False,
)


mcmed_categorical_compatibility_audit.to_csv(
    compatibility_output_paths[
        "categorical"
    ],
    index=False,
)


mcmed_unseen_category_summary.to_csv(
    compatibility_output_paths[
        "unseen_categories"
    ],
    index=False,
)


mcmed_binary_domain_audit.to_csv(
    compatibility_output_paths[
        "binary_domain"
    ],
    index=False,
)


mcmed_missingness_compatibility.to_csv(
    compatibility_output_paths[
        "missingness"
    ],
    index=False,
)


mcmed_prepared_categorical_audit.to_csv(
    compatibility_output_paths[
        "prepared_categorical"
    ],
    index=False,
)



print(
    "MC-MED feature compatibility summary:"
)

display(
    mcmed_feature_compatibility_summary
)


print(
    "\nNumeric S0 compatibility:"
)

display(
    mcmed_numeric_compatibility_audit
)


print(
    "\nCategorical compatibility:"
)

display(
    mcmed_categorical_compatibility_audit
)


print(
    "\nUnseen MC-MED categorical levels:"
)

if mcmed_unseen_category_summary.empty:

    print(
        "None"
    )

else:

    display(
        mcmed_unseen_category_summary
    )


print(
    "\nFrozen categorical encoder structure:"
)

display(
    mcmed_prepared_categorical_audit
)


print(
    "\nBinary-domain summary:"
)

display(
    pd.DataFrame(
        [
            {
                "binary_features": len(
                    mcmed_binary_domain_audit
                ),
                "domain_valid_features": int(
                    mcmed_binary_domain_audit[
                        "domain_valid"
                    ].sum()
                ),
                "features_with_missing_values": int(
                    (
                        mcmed_binary_domain_audit[
                            "missing_count"
                        ]
                        >
                        0
                    ).sum()
                ),
                "domain_violations": int(
                    (
                        ~mcmed_binary_domain_audit[
                            "domain_valid"
                        ]
                    ).sum()
                ),
            }
        ]
    )
)


print(
    "\nTop 20 MC-MED features by missingness:"
)

display(
    mcmed_missingness_compatibility
    .sort_values(
        [
            "missing_percent",
            "feature",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .head(
        20
    )
    .reset_index(
        drop=True
    )
    .round(
        4
    )
)



print(
    "\nMC-MED FEATURE COMPATIBILITY AUDIT PASSED"
)

print(
    "External rows:",
    f"{len(X_mcmed_external_standardised):,}",
)

print(
    "Raw feature columns:",
    X_mcmed_external_standardised.shape[
        1
    ],
)

print(
    "Prepared columns after frozen transform:",
    X_mcmed_external_transformed.shape[
        1
    ],
)

print(
    "Expected prepared columns:",
    LOCKED_PREPARED_FEATURE_COUNT,
)

print(
    "Numeric coercion failures:",
    int(
        mcmed_numeric_compatibility_audit[
            "numeric_coercion_failures"
        ].sum()
    ),
)

print(
    "Binary domain violations:",
    int(
        (
            ~mcmed_binary_domain_audit[
                "domain_valid"
            ]
        ).sum()
    ),
)

print(
    "Unseen categorical levels:",
    len(
        mcmed_unseen_category_summary
    ),
)

print(
    "Frozen numeric imputer unchanged: YES"
)

print(
    "Frozen categorical encoder unchanged: YES"
)

print(
    "Preprocessor refitted: NO"
)

print(
    "Model probabilities generated: NO"
)

print(
    "Calibration applied: NO"
)

print(
    "Threshold applied: NO"
)

## Locked MC-MED external prediction

The verified MC-MED external cohort is now passed through the frozen MIMIC-IV model without any model-development changes.

The already-transformed 2,498-column MC-MED matrix is evaluated using the locked XGBoost classifier to obtain raw admission probabilities. These probabilities are converted to logits and passed through the validation-fitted Platt calibrator. The previously locked calibrated-probability threshold of 0.282 is then applied to generate HOME versus ADMITTED classifications.

Raw and calibrated probabilities, classifications and point-estimate external-validation metrics are saved. No preprocessing, model fitting, calibration fitting, threshold optimisation or feature selection is performed using MC-MED.

In [ ]:



required_external_prediction_objects = [
    "PROJECT_ROOT",
    "mcmed_external",
    "X_mcmed_external_standardised",
    "X_mcmed_external_transformed",
    "y_mcmed_external",
    "locked_external_pipeline",
    "locked_external_xgboost",
    "locked_external_calibrator",
    "LOCKED_DECISION_THRESHOLD",
    "LOCKED_RAW_FEATURE_ORDER",
]


missing_external_prediction_objects = [
    object_name
    for object_name
    in required_external_prediction_objects
    if object_name not in globals()
]


if missing_external_prediction_objects:

    raise RuntimeError(
        "Complete the MC-MED loading and compatibility "
        "sections before external prediction.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_external_prediction_objects
        )
    )


assert X_mcmed_external_standardised.shape == (
    6707,
    2495,
)


assert X_mcmed_external_transformed.shape == (
    6707,
    2498,
)


assert len(
    y_mcmed_external
) == 6707


assert set(
    y_mcmed_external.unique()
) == {
    0,
    1,
}


assert np.isclose(
    LOCKED_DECISION_THRESHOLD,
    0.282,
    rtol=0.0,
    atol=1e-12,
)



external_prediction_start_time = (
    time.perf_counter()
)


mcmed_external_raw_probability = (
    locked_external_xgboost
    .predict_proba(
        X_mcmed_external_transformed
    )[
        :,
        1,
    ]
)


mcmed_external_raw_probability = np.asarray(
    mcmed_external_raw_probability,
    dtype=float,
)


assert mcmed_external_raw_probability.shape == (
    6707,
)


assert np.isfinite(
    mcmed_external_raw_probability
).all()


assert (
    (
        mcmed_external_raw_probability
        >=
        0.0
    )
    &
    (
        mcmed_external_raw_probability
        <=
        1.0
    )
).all()



mcmed_external_pipeline_probability = (
    locked_external_pipeline
    .predict_proba(
        X_mcmed_external_standardised
    )[
        :,
        1,
    ]
)


mcmed_external_pipeline_probability = np.asarray(
    mcmed_external_pipeline_probability,
    dtype=float,
)


external_raw_probability_difference = np.abs(
    mcmed_external_raw_probability
    -
    mcmed_external_pipeline_probability
)


assert np.allclose(
    mcmed_external_raw_probability,
    mcmed_external_pipeline_probability,
    rtol=1e-7,
    atol=1e-7,
), (
    "The direct transformed-matrix XGBoost prediction does "
    "not reproduce the saved full-pipeline prediction.\n\n"
    f"Maximum absolute difference: "
    f"{external_raw_probability_difference.max()}"
)



EXTERNAL_PROBABILITY_EPSILON = 1e-6


mcmed_external_raw_probability_clipped = np.clip(
    mcmed_external_raw_probability,
    EXTERNAL_PROBABILITY_EPSILON,
    1.0
    -
    EXTERNAL_PROBABILITY_EPSILON,
)


mcmed_external_raw_logit = np.log(
    mcmed_external_raw_probability_clipped
    /
    (
        1.0
        -
        mcmed_external_raw_probability_clipped
    )
)


assert np.isfinite(
    mcmed_external_raw_logit
).all()


mcmed_external_calibrated_probability = (
    locked_external_calibrator
    .predict_proba(
        mcmed_external_raw_logit.reshape(
            -1,
            1,
        )
    )[
        :,
        1,
    ]
)


mcmed_external_calibrated_probability = np.asarray(
    mcmed_external_calibrated_probability,
    dtype=float,
)


assert mcmed_external_calibrated_probability.shape == (
    6707,
)


assert np.isfinite(
    mcmed_external_calibrated_probability
).all()


assert (
    (
        mcmed_external_calibrated_probability
        >=
        0.0
    )
    &
    (
        mcmed_external_calibrated_probability
        <=
        1.0
    )
).all()



mcmed_external_prediction = (
    mcmed_external_calibrated_probability
    >=
    LOCKED_DECISION_THRESHOLD
).astype(
    np.int8
)


assert mcmed_external_prediction.shape == (
    6707,
)


assert set(
    np.unique(
        mcmed_external_prediction
    )
).issubset(
    {
        0,
        1,
    }
)


external_prediction_elapsed_seconds = (
    time.perf_counter()
    -
    external_prediction_start_time
)



external_tn, external_fp, external_fn, external_tp = (
    confusion_matrix(
        y_mcmed_external,
        mcmed_external_prediction,
        labels=[
            0,
            1,
        ],
    )
    .ravel()
)


assert (
    external_tn
    +
    external_fp
    +
    external_fn
    +
    external_tp
    ==
    6707
)



external_prevalence = float(
    y_mcmed_external.mean()
)


external_accuracy = float(
    accuracy_score(
        y_mcmed_external,
        mcmed_external_prediction,
    )
)


external_balanced_accuracy = float(
    balanced_accuracy_score(
        y_mcmed_external,
        mcmed_external_prediction,
    )
)


external_sensitivity = float(
    recall_score(
        y_mcmed_external,
        mcmed_external_prediction,
        pos_label=1,
    )
)


external_specificity = float(
    external_tn
    /
    (
        external_tn
        +
        external_fp
    )
)


external_ppv = float(
    precision_score(
        y_mcmed_external,
        mcmed_external_prediction,
        pos_label=1,
        zero_division=0,
    )
)


external_npv = float(
    external_tn
    /
    (
        external_tn
        +
        external_fn
    )
)


external_f1 = float(
    f1_score(
        y_mcmed_external,
        mcmed_external_prediction,
        pos_label=1,
        zero_division=0,
    )
)


external_auroc = float(
    roc_auc_score(
        y_mcmed_external,
        mcmed_external_calibrated_probability,
    )
)


external_auprc = float(
    average_precision_score(
        y_mcmed_external,
        mcmed_external_calibrated_probability,
    )
)


external_raw_log_loss = float(
    log_loss(
        y_mcmed_external,
        mcmed_external_raw_probability,
        labels=[
            0,
            1,
        ],
    )
)


external_calibrated_log_loss = float(
    log_loss(
        y_mcmed_external,
        mcmed_external_calibrated_probability,
        labels=[
            0,
            1,
        ],
    )
)


external_raw_brier = float(
    brier_score_loss(
        y_mcmed_external,
        mcmed_external_raw_probability,
    )
)


external_calibrated_brier = float(
    brier_score_loss(
        y_mcmed_external,
        mcmed_external_calibrated_probability,
    )
)



mcmed_external_point_metrics = pd.DataFrame(
    [
        {
            "metric": "AUPRC",
            "value": external_auprc,
        },
        {
            "metric": "AUROC",
            "value": external_auroc,
        },
        {
            "metric": "Calibrated log loss",
            "value": external_calibrated_log_loss,
        },
        {
            "metric": "Calibrated Brier score",
            "value": external_calibrated_brier,
        },
        {
            "metric": "Accuracy",
            "value": external_accuracy,
        },
        {
            "metric": "Balanced accuracy",
            "value": external_balanced_accuracy,
        },
        {
            "metric": "Sensitivity",
            "value": external_sensitivity,
        },
        {
            "metric": "Specificity",
            "value": external_specificity,
        },
        {
            "metric": "PPV",
            "value": external_ppv,
        },
        {
            "metric": "NPV",
            "value": external_npv,
        },
        {
            "metric": "F1",
            "value": external_f1,
        },
    ]
)



mcmed_external_probability_summary = pd.DataFrame(
    [
        {
            "external_rows": 6707,
            "home": int(
                (
                    y_mcmed_external
                    ==
                    0
                ).sum()
            ),
            "admitted": int(
                (
                    y_mcmed_external
                    ==
                    1
                ).sum()
            ),
            "observed_admission_prevalence": (
                external_prevalence
            ),
            "mean_raw_probability": float(
                mcmed_external_raw_probability.mean()
            ),
            "mean_calibrated_probability": float(
                mcmed_external_calibrated_probability.mean()
            ),
            "predicted_admitted_fraction": float(
                mcmed_external_prediction.mean()
            ),
            "locked_threshold": float(
                LOCKED_DECISION_THRESHOLD
            ),
        }
    ]
)


mcmed_external_probability_comparison = pd.DataFrame(
    [
        {
            "probability_type": "Raw XGBoost",
            "mean_probability": float(
                mcmed_external_raw_probability.mean()
            ),
            "log_loss": (
                external_raw_log_loss
            ),
            "brier_score": (
                external_raw_brier
            ),
        },
        {
            "probability_type": "Platt calibrated",
            "mean_probability": float(
                mcmed_external_calibrated_probability.mean()
            ),
            "log_loss": (
                external_calibrated_log_loss
            ),
            "brier_score": (
                external_calibrated_brier
            ),
        },
    ]
)



mcmed_external_confusion_matrix = pd.DataFrame(
    [
        {
            "actual": "HOME",
            "predicted_HOME": int(
                external_tn
            ),
            "predicted_ADMITTED": int(
                external_fp
            ),
        },
        {
            "actual": "ADMITTED",
            "predicted_HOME": int(
                external_fn
            ),
            "predicted_ADMITTED": int(
                external_tp
            ),
        },
    ]
)



mcmed_external_predictions = pd.DataFrame(
    index=mcmed_external.index
)


for identifier_column in [
    "CSN",
    "csn",
    "MRN",
    "mrn",
    "Visit_no",
    "visit_no",
]:

    if (
        identifier_column
        in mcmed_external.columns
        and
        identifier_column
        not in mcmed_external_predictions.columns
    ):

        mcmed_external_predictions[
            identifier_column
        ] = (
            mcmed_external[
                identifier_column
            ]
        )


mcmed_external_predictions[
    "outcome_binary"
] = (
    y_mcmed_external
    .astype(
        np.int8
    )
)


mcmed_external_predictions[
    "raw_probability"
] = (
    mcmed_external_raw_probability
    .astype(
        np.float32
    )
)


mcmed_external_predictions[
    "calibrated_probability"
] = (
    mcmed_external_calibrated_probability
    .astype(
        np.float32
    )
)


mcmed_external_predictions[
    "prediction_binary"
] = (
    mcmed_external_prediction
)


mcmed_external_predictions[
    "prediction_label"
] = np.where(
    mcmed_external_prediction
    ==
    1,
    "ADMITTED",
    "HOME",
)


assert len(
    mcmed_external_predictions
) == 6707


assert (
    mcmed_external_predictions[
        "outcome_binary"
    ]
    .to_numpy()
    ==
    y_mcmed_external.to_numpy()
).all()



mcmed_external_prediction_audit = pd.DataFrame(
    [
        {
            "dataset": "MC-MED",
            "external_rows": 6707,
            "algorithm": "XGBoost",
            "raw_features": 2495,
            "prepared_features": 2498,
            "calibration": "Platt scaling",
            "calibrator_intercept": float(
                locked_external_calibrator
                .intercept_[
                    0
                ]
            ),
            "calibrator_slope": float(
                locked_external_calibrator
                .coef_[
                    0
                ][
                    0
                ]
            ),
            "threshold": float(
                LOCKED_DECISION_THRESHOLD
            ),
            "max_pipeline_probability_difference": float(
                external_raw_probability_difference.max()
            ),
            "prediction_seconds": float(
                external_prediction_elapsed_seconds
            ),
            "preprocessor_refitted": False,
            "model_refitted": False,
            "calibrator_refitted": False,
            "threshold_changed": False,
            "external_labels_used_for_prediction": False,
        }
    ]
)



NOTEBOOK3_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook3"
)


NOTEBOOK3_PREDICTION_DIR = (
    PROJECT_ROOT
    / "results"
    / "predictions"
    / "notebook3"
)


for directory in [
    NOTEBOOK3_TABLE_DIR,
    NOTEBOOK3_PREDICTION_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


external_prediction_paths = {
    "point_metrics": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_external_point_metrics.csv"
    ),
    "probability_summary": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_external_probability_summary.csv"
    ),
    "probability_comparison": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_external_probability_comparison.csv"
    ),
    "confusion_matrix": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_external_confusion_matrix.csv"
    ),
    "prediction_audit": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_external_prediction_audit.csv"
    ),
    "patient_predictions": (
        NOTEBOOK3_PREDICTION_DIR
        / "mcmed_external_predictions.parquet"
    ),
}


mcmed_external_point_metrics.to_csv(
    external_prediction_paths[
        "point_metrics"
    ],
    index=False,
)


mcmed_external_probability_summary.to_csv(
    external_prediction_paths[
        "probability_summary"
    ],
    index=False,
)


mcmed_external_probability_comparison.to_csv(
    external_prediction_paths[
        "probability_comparison"
    ],
    index=False,
)


mcmed_external_confusion_matrix.to_csv(
    external_prediction_paths[
        "confusion_matrix"
    ],
    index=False,
)


mcmed_external_prediction_audit.to_csv(
    external_prediction_paths[
        "prediction_audit"
    ],
    index=False,
)


mcmed_external_predictions.to_parquet(
    external_prediction_paths[
        "patient_predictions"
    ],
    index=True,
    engine="pyarrow",
)



reloaded_mcmed_external_predictions = pd.read_parquet(
    external_prediction_paths[
        "patient_predictions"
    ],
    engine="pyarrow",
)


assert len(
    reloaded_mcmed_external_predictions
) == 6707


assert np.allclose(
    reloaded_mcmed_external_predictions[
        "raw_probability"
    ],
    mcmed_external_raw_probability,
    rtol=1e-6,
    atol=1e-6,
)


assert np.allclose(
    reloaded_mcmed_external_predictions[
        "calibrated_probability"
    ],
    mcmed_external_calibrated_probability,
    rtol=1e-6,
    atol=1e-6,
)


assert (
    reloaded_mcmed_external_predictions[
        "prediction_binary"
    ]
    .to_numpy(
        dtype=np.int8
    )
    ==
    mcmed_external_prediction
).all()



print(
    "MC-MED external point-estimate metrics:"
)

display(
    mcmed_external_point_metrics.round(
        6
    )
)


print(
    "\nMC-MED probability summary:"
)

display(
    mcmed_external_probability_summary.round(
        6
    )
)


print(
    "\nRaw versus calibrated probability performance:"
)

display(
    mcmed_external_probability_comparison.round(
        6
    )
)


print(
    "\nMC-MED confusion matrix:"
)

display(
    mcmed_external_confusion_matrix
)


print(
    "\nLocked external-prediction audit:"
)

display(
    mcmed_external_prediction_audit.round(
        8
    )
)



print(
    "\nLOCKED MC-MED EXTERNAL PREDICTION COMPLETED"
)

print(
    "External rows:",
    f"{len(y_mcmed_external):,}",
)

print(
    "Observed HOME:",
    f"{int((y_mcmed_external == 0).sum()):,}",
)

print(
    "Observed ADMITTED:",
    f"{int((y_mcmed_external == 1).sum()):,}",
)

print(
    "Observed admission prevalence:",
    round(
        external_prevalence,
        6,
    ),
)

print(
    "\nMean raw probability:",
    round(
        float(
            mcmed_external_raw_probability.mean()
        ),
        6,
    ),
)

print(
    "Mean calibrated probability:",
    round(
        float(
            mcmed_external_calibrated_probability.mean()
        ),
        6,
    ),
)

print(
    "Predicted ADMITTED fraction:",
    round(
        float(
            mcmed_external_prediction.mean()
        ),
        6,
    ),
)

print(
    "\nLocked threshold:",
    round(
        LOCKED_DECISION_THRESHOLD,
        6,
    ),
)

print(
    "Full-pipeline probability reproduction: PASSED"
)

print(
    "\nPreprocessor refitted: NO"
)

print(
    "XGBoost refitted: NO"
)

print(
    "Platt calibrator refitted: NO"
)

print(
    "Threshold changed: NO"
)

print(
    "MC-MED labels used to generate predictions: NO"
)

print(
    "MC-MED labels used only for evaluation metrics: YES"
)

print(
    "\nPatient-level external predictions saved: YES"
)

### MC-MED external performance with bootstrap 95% confidence intervals

Uncertainty in the locked external-validation performance is quantified using 2,000 bootstrap resamples with replacement and a fixed random seed.

Each bootstrap replicate resamples the external evaluation observations and recalculates discrimination, probability and threshold-dependent classification metrics using the already-generated locked probabilities and classifications. No preprocessing, model fitting, recalibration or threshold optimisation occurs during bootstrapping.

The reported 95% confidence intervals are percentile bootstrap intervals based on the 2.5th and 97.5th percentiles. The metric set matches the internal temporal-test evaluation to support direct comparison between internal and external validation.

In [ ]:



required_bootstrap_objects = [
    "PROJECT_ROOT",
    "y_mcmed_external",
    "mcmed_external_raw_probability",
    "mcmed_external_calibrated_probability",
    "mcmed_external_prediction",
    "mcmed_external_point_metrics",
    "LOCKED_DECISION_THRESHOLD",
]


missing_bootstrap_objects = [
    object_name
    for object_name
    in required_bootstrap_objects
    if object_name not in globals()
]


if missing_bootstrap_objects:

    raise RuntimeError(
        "Complete the locked MC-MED external prediction "
        "section before bootstrapping.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_bootstrap_objects
        )
    )



EXTERNAL_BOOTSTRAP_RANDOM_SEED = 42

N_EXTERNAL_BOOTSTRAP_REPLICATES = 2000

EXTERNAL_BOOTSTRAP_CI_LEVEL = 0.95


external_bootstrap_rng = np.random.default_rng(
    EXTERNAL_BOOTSTRAP_RANDOM_SEED
)



external_bootstrap_y = (
    y_mcmed_external
    .to_numpy(
        dtype=np.int8
    )
)


external_bootstrap_probability = np.asarray(
    mcmed_external_calibrated_probability,
    dtype=np.float64,
)


external_bootstrap_prediction = np.asarray(
    mcmed_external_prediction,
    dtype=np.int8,
)


external_bootstrap_raw_probability = np.asarray(
    mcmed_external_raw_probability,
    dtype=np.float64,
)


N_EXTERNAL_ROWS = len(
    external_bootstrap_y
)


assert N_EXTERNAL_ROWS == 6707


assert external_bootstrap_probability.shape == (
    N_EXTERNAL_ROWS,
)


assert external_bootstrap_prediction.shape == (
    N_EXTERNAL_ROWS,
)


assert external_bootstrap_raw_probability.shape == (
    N_EXTERNAL_ROWS,
)


assert set(
    np.unique(
        external_bootstrap_y
    )
) == {
    0,
    1,
}


assert np.isfinite(
    external_bootstrap_probability
).all()


assert np.isclose(
    LOCKED_DECISION_THRESHOLD,
    0.282,
    rtol=0.0,
    atol=1e-12,
)



external_bootstrap_sampling_unit = (
    "frozen external cohort row"
)


external_patient_identifier = None


for candidate_identifier in [
    "MRN",
    "mrn",
]:

    if (
        candidate_identifier
        in mcmed_external.columns
    ):

        external_patient_identifier = (
            candidate_identifier
        )

        break


if external_patient_identifier is not None:

    external_patient_ids = (
        mcmed_external[
            external_patient_identifier
        ]
    )


    non_missing_patient_ids = (
        external_patient_ids[
            external_patient_ids.notna()
        ]
    )


    duplicate_patient_rows = int(
        non_missing_patient_ids
        .duplicated(
            keep=False
        )
        .sum()
    )


    print(
        "Patient identifier used for bootstrap audit:",
        external_patient_identifier,
    )

    print(
        "Duplicated non-missing patient rows:",
        duplicate_patient_rows,
    )


    if duplicate_patient_rows == 0:

        external_bootstrap_sampling_unit = (
            "patient / frozen external cohort row"
        )

    else:

        raise RuntimeError(
            "The frozen MC-MED external cohort contains "
            "multiple rows for at least one patient. "
            "A clustered patient-level bootstrap would be "
            "required instead of simple row resampling."
        )


else:

    print(
        "No MRN field available for an independent patient-"
        "uniqueness audit."
    )

    print(
        "Bootstrap will use the frozen external cohort row, "
        "consistent with the final cohort definition."
    )



def calculate_external_metrics(
    outcome,
    calibrated_probability,
    prediction,
):
    """
    Calculate the metric set used for the locked internal
    temporal test and MC-MED external validation.
    """

    tn, fp, fn, tp = (
        confusion_matrix(
            outcome,
            prediction,
            labels=[
                0,
                1,
            ],
        )
        .ravel()
    )


    sensitivity = (
        tp
        /
        (
            tp
            +
            fn
        )
        if (
            tp
            +
            fn
        )
        >
        0
        else np.nan
    )


    specificity = (
        tn
        /
        (
            tn
            +
            fp
        )
        if (
            tn
            +
            fp
        )
        >
        0
        else np.nan
    )


    ppv = (
        tp
        /
        (
            tp
            +
            fp
        )
        if (
            tp
            +
            fp
        )
        >
        0
        else np.nan
    )


    npv = (
        tn
        /
        (
            tn
            +
            fn
        )
        if (
            tn
            +
            fn
        )
        >
        0
        else np.nan
    )


    return {
        "AUPRC": float(
            average_precision_score(
                outcome,
                calibrated_probability,
            )
        ),
        "AUROC": float(
            roc_auc_score(
                outcome,
                calibrated_probability,
            )
        ),
        "Calibrated log loss": float(
            log_loss(
                outcome,
                calibrated_probability,
                labels=[
                    0,
                    1,
                ],
            )
        ),
        "Calibrated Brier score": float(
            brier_score_loss(
                outcome,
                calibrated_probability,
            )
        ),
        "Accuracy": float(
            accuracy_score(
                outcome,
                prediction,
            )
        ),
        "Balanced accuracy": float(
            balanced_accuracy_score(
                outcome,
                prediction,
            )
        ),
        "Sensitivity": float(
            sensitivity
        ),
        "Specificity": float(
            specificity
        ),
        "PPV": float(
            ppv
        ),
        "NPV": float(
            npv
        ),
        "F1": float(
            f1_score(
                outcome,
                prediction,
                pos_label=1,
                zero_division=0,
            )
        ),
    }



external_bootstrap_point_metrics = (
    calculate_external_metrics(
        external_bootstrap_y,
        external_bootstrap_probability,
        external_bootstrap_prediction,
    )
)


previous_external_metric_lookup = (
    mcmed_external_point_metrics
    .set_index(
        "metric"
    )[
        "value"
    ]
    .to_dict()
)


for metric_name, metric_value in (
    external_bootstrap_point_metrics.items()
):

    assert metric_name in (
        previous_external_metric_lookup
    )


    assert np.isclose(
        metric_value,
        float(
            previous_external_metric_lookup[
                metric_name
            ]
        ),
        rtol=1e-10,
        atol=1e-10,
    ), (
        "Bootstrap point-estimate reconstruction failed for "
        f"{metric_name}."
    )


print(
    "External point-estimate reproduction: PASSED"
)



external_bootstrap_records = []


external_bootstrap_start_time = (
    time.perf_counter()
)


for bootstrap_number in range(
    1,
    N_EXTERNAL_BOOTSTRAP_REPLICATES
    +
    1,
):

    sampled_positions = (
        external_bootstrap_rng.integers(
            low=0,
            high=N_EXTERNAL_ROWS,
            size=N_EXTERNAL_ROWS,
        )
    )


    bootstrap_outcome = (
        external_bootstrap_y[
            sampled_positions
        ]
    )


    if np.unique(
        bootstrap_outcome
    ).size < 2:

        continue


    bootstrap_probability = (
        external_bootstrap_probability[
            sampled_positions
        ]
    )


    bootstrap_prediction = (
        external_bootstrap_prediction[
            sampled_positions
        ]
    )


    replicate_metrics = (
        calculate_external_metrics(
            bootstrap_outcome,
            bootstrap_probability,
            bootstrap_prediction,
        )
    )


    external_bootstrap_records.append(
        {
            "bootstrap_replicate": (
                bootstrap_number
            ),
            "admitted_prevalence": float(
                bootstrap_outcome.mean()
            ),
            **replicate_metrics,
        }
    )


external_bootstrap_elapsed_seconds = (
    time.perf_counter()
    -
    external_bootstrap_start_time
)


mcmed_external_bootstrap_distribution = (
    pd.DataFrame(
        external_bootstrap_records
    )
)


valid_external_bootstrap_replicates = len(
    mcmed_external_bootstrap_distribution
)


assert valid_external_bootstrap_replicates == (
    N_EXTERNAL_BOOTSTRAP_REPLICATES
), (
    "One or more bootstrap replicates were invalid because "
    "they did not contain both outcome classes."
)



bootstrap_lower_percentile = (
    100
    *
    (
        1.0
        -
        EXTERNAL_BOOTSTRAP_CI_LEVEL
    )
    /
    2.0
)


bootstrap_upper_percentile = (
    100
    -
    bootstrap_lower_percentile
)


external_metric_order = [
    "AUPRC",
    "AUROC",
    "Calibrated log loss",
    "Calibrated Brier score",
    "Accuracy",
    "Balanced accuracy",
    "Sensitivity",
    "Specificity",
    "PPV",
    "NPV",
    "F1",
]


external_ci_records = []


for metric_name in (
    external_metric_order
):

    bootstrap_metric_values = (
        mcmed_external_bootstrap_distribution[
            metric_name
        ]
        .to_numpy(
            dtype=float
        )
    )


    lower_ci = float(
        np.percentile(
            bootstrap_metric_values,
            bootstrap_lower_percentile,
        )
    )


    upper_ci = float(
        np.percentile(
            bootstrap_metric_values,
            bootstrap_upper_percentile,
        )
    )


    external_ci_records.append(
        {
            "metric": (
                metric_name
            ),
            "value": float(
                external_bootstrap_point_metrics[
                    metric_name
                ]
            ),
            "ci_lower_95": (
                lower_ci
            ),
            "ci_upper_95": (
                upper_ci
            ),
            "bootstrap_replicates": (
                valid_external_bootstrap_replicates
            ),
            "bootstrap_method": (
                "Percentile"
            ),
            "bootstrap_unit": (
                external_bootstrap_sampling_unit
            ),
        }
    )


mcmed_external_metrics_with_ci = pd.DataFrame(
    external_ci_records
)



mcmed_external_metrics_with_ci[
    "ci_width"
] = (
    mcmed_external_metrics_with_ci[
        "ci_upper_95"
    ]
    -
    mcmed_external_metrics_with_ci[
        "ci_lower_95"
    ]
)


external_auprc_value = float(
    mcmed_external_metrics_with_ci.loc[
        mcmed_external_metrics_with_ci[
            "metric"
        ]
        ==
        "AUPRC",
        "value",
    ]
    .iloc[
        0
    ]
)


external_auprc_prevalence_ratio = (
    external_auprc_value
    /
    float(
        external_bootstrap_y.mean()
    )
)



mcmed_external_bootstrap_audit = pd.DataFrame(
    [
        {
            "dataset": (
                "MC-MED"
            ),
            "rows": (
                N_EXTERNAL_ROWS
            ),
            "home": int(
                (
                    external_bootstrap_y
                    ==
                    0
                ).sum()
            ),
            "admitted": int(
                (
                    external_bootstrap_y
                    ==
                    1
                ).sum()
            ),
            "observed_prevalence": float(
                external_bootstrap_y.mean()
            ),
            "bootstrap_replicates_requested": (
                N_EXTERNAL_BOOTSTRAP_REPLICATES
            ),
            "bootstrap_replicates_valid": (
                valid_external_bootstrap_replicates
            ),
            "bootstrap_seed": (
                EXTERNAL_BOOTSTRAP_RANDOM_SEED
            ),
            "confidence_level": (
                EXTERNAL_BOOTSTRAP_CI_LEVEL
            ),
            "interval_method": (
                "Percentile"
            ),
            "sampling_unit": (
                external_bootstrap_sampling_unit
            ),
            "locked_threshold": float(
                LOCKED_DECISION_THRESHOLD
            ),
            "bootstrap_seconds": float(
                external_bootstrap_elapsed_seconds
            ),
            "model_refitted": False,
            "preprocessor_refitted": False,
            "calibrator_refitted": False,
            "threshold_changed": False,
        }
    ]
)



NOTEBOOK3_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook3"
)


NOTEBOOK3_BOOTSTRAP_DIR = (
    PROJECT_ROOT
    / "results"
    / "bootstrap"
    / "notebook3"
)


for directory in [
    NOTEBOOK3_TABLE_DIR,
    NOTEBOOK3_BOOTSTRAP_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


external_bootstrap_output_paths = {
    "metrics_with_ci": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_external_metrics_with_ci.csv"
    ),
    "bootstrap_audit": (
        NOTEBOOK3_TABLE_DIR
        / "mcmed_external_bootstrap_audit.csv"
    ),
    "bootstrap_distribution": (
        NOTEBOOK3_BOOTSTRAP_DIR
        / "mcmed_external_bootstrap_distribution.parquet"
    ),
}


mcmed_external_metrics_with_ci.to_csv(
    external_bootstrap_output_paths[
        "metrics_with_ci"
    ],
    index=False,
)


mcmed_external_bootstrap_audit.to_csv(
    external_bootstrap_output_paths[
        "bootstrap_audit"
    ],
    index=False,
)


mcmed_external_bootstrap_distribution.to_parquet(
    external_bootstrap_output_paths[
        "bootstrap_distribution"
    ],
    index=False,
    engine="pyarrow",
)



reloaded_external_metrics_with_ci = (
    pd.read_csv(
        external_bootstrap_output_paths[
            "metrics_with_ci"
        ]
    )
)


reloaded_external_bootstrap_distribution = (
    pd.read_parquet(
        external_bootstrap_output_paths[
            "bootstrap_distribution"
        ],
        engine="pyarrow",
    )
)


assert len(
    reloaded_external_metrics_with_ci
) == 11


assert len(
    reloaded_external_bootstrap_distribution
) == 2000


assert (
    reloaded_external_metrics_with_ci[
        "metric"
    ]
    .tolist()
    ==
    external_metric_order
)



print(
    "MC-MED external performance with 95% bootstrap CIs:"
)


display(
    mcmed_external_metrics_with_ci[
        [
            "metric",
            "value",
            "ci_lower_95",
            "ci_upper_95",
        ]
    ]
    .round(
        6
    )
)


print(
    "\nExternal bootstrap audit:"
)


display(
    mcmed_external_bootstrap_audit.round(
        6
    )
)


print(
    "\nAUPRC / prevalence ratio:",
    round(
        external_auprc_prevalence_ratio,
        3,
    ),
)



print(
    "\nMC-MED EXTERNAL BOOTSTRAP EVALUATION COMPLETED"
)


print(
    "Bootstrap replicates:",
    valid_external_bootstrap_replicates,
)


print(
    "Bootstrap seed:",
    EXTERNAL_BOOTSTRAP_RANDOM_SEED,
)


print(
    "Confidence interval:",
    "95% percentile bootstrap",
)


print(
    "Sampling unit:",
    external_bootstrap_sampling_unit,
)


print(
    "Runtime minutes:",
    round(
        external_bootstrap_elapsed_seconds
        /
        60,
        3,
    ),
)


print(
    "\nModel refitted during bootstrap: NO"
)


print(
    "Preprocessor refitted during bootstrap: NO"
)


print(
    "Calibrator refitted during bootstrap: NO"
)


print(
    "Threshold changed during bootstrap: NO"
)


print(
    "External predictions regenerated per replicate: NO"
)


print(
    "Only patient-level evaluation rows resampled: YES"
)


print(
    "\nBootstrap distribution saved: YES"
)


print(
    "External metrics with 95% CI saved: YES"
)

### Internal temporal-test versus MC-MED external-validation performance

Performance of the locked model is compared between the MIMIC-IV internal temporal-test cohort and the independent MC-MED external-validation cohort.

Both evaluations use the same locked XGBoost model, Platt calibrator and calibrated-probability threshold of 0.282. Point estimates and patient-level percentile bootstrap 95% confidence intervals are presented using the same metric definitions.

Differences are calculated as MC-MED external performance minus MIMIC-IV internal temporal-test performance. These comparisons are descriptive and do not trigger model retraining, recalibration or threshold revision.

In [ ]:



required_comparison_objects = [
    "PROJECT_ROOT",
    "mcmed_external_metrics_with_ci",
    "mcmed_external_probability_summary",
    "LOCKED_DECISION_THRESHOLD",
]


missing_comparison_objects = [
    object_name
    for object_name
    in required_comparison_objects
    if object_name not in globals()
]


if missing_comparison_objects:

    raise RuntimeError(
        "Complete the MC-MED bootstrap evaluation before "
        "creating the internal-versus-external comparison.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_comparison_objects
        )
    )



INTERNAL_TEST_METRICS_PATH = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook2"
    / "internal_test_metrics_with_ci.csv"
)


INTERNAL_TEST_PROBABILITY_SUMMARY_PATH = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook2"
    / "internal_test_probability_summary.csv"
)


assert INTERNAL_TEST_METRICS_PATH.exists(), (
    "Frozen Notebook 2 internal-test metric file is missing:\n"
    f"{INTERNAL_TEST_METRICS_PATH}"
)


assert INTERNAL_TEST_PROBABILITY_SUMMARY_PATH.exists(), (
    "Frozen Notebook 2 internal-test probability-summary "
    "file is missing:\n"
    f"{INTERNAL_TEST_PROBABILITY_SUMMARY_PATH}"
)


mimic_internal_metrics = pd.read_csv(
    INTERNAL_TEST_METRICS_PATH
)


mimic_internal_probability_summary = pd.read_csv(
    INTERNAL_TEST_PROBABILITY_SUMMARY_PATH
)



def standardise_metric_ci_table(
    metric_table,
    dataset_name,
):

    table = metric_table.copy()


    print(
        f"\n{dataset_name} source columns:"
    )

    print(
        table.columns.tolist()
    )



    if "metric" not in table.columns:

        raise RuntimeError(
            f"{dataset_name}: no 'metric' column found.\n\n"
            f"Available columns:\n"
            f"{table.columns.tolist()}"
        )



    value_candidates = [
        "value",
        "point_estimate",
        "estimate",
        "metric_value",
        "point",
    ]


    value_column = next(
        (
            column_name
            for column_name
            in value_candidates
            if column_name
            in table.columns
        ),
        None,
    )



    lower_candidates = [
        "ci_lower_95",
        "ci_lower",
        "lower_95",
        "lower_ci",
        "lower_95_ci",
        "ci_95_lower",
        "bootstrap_ci_lower",
        "bootstrap_lower",
        "ci_2.5",
        "ci_2_5",
        "percentile_2.5",
        "percentile_2_5",
        "lower",
    ]


    lower_column = next(
        (
            column_name
            for column_name
            in lower_candidates
            if column_name
            in table.columns
        ),
        None,
    )



    upper_candidates = [
        "ci_upper_95",
        "ci_upper",
        "upper_95",
        "upper_ci",
        "upper_95_ci",
        "ci_95_upper",
        "bootstrap_ci_upper",
        "bootstrap_upper",
        "ci_97.5",
        "ci_97_5",
        "percentile_97.5",
        "percentile_97_5",
        "upper",
    ]


    upper_column = next(
        (
            column_name
            for column_name
            in upper_candidates
            if column_name
            in table.columns
        ),
        None,
    )



    if lower_column is None:

        lower_matches = [
            column_name
            for column_name
            in table.columns
            if (
                "lower"
                in column_name.lower()
                or
                "2.5"
                in column_name.lower()
                or
                "2_5"
                in column_name.lower()
            )
        ]


        if len(
            lower_matches
        ) == 1:

            lower_column = (
                lower_matches[0]
            )


    if upper_column is None:

        upper_matches = [
            column_name
            for column_name
            in table.columns
            if (
                "upper"
                in column_name.lower()
                or
                "97.5"
                in column_name.lower()
                or
                "97_5"
                in column_name.lower()
            )
        ]


        if len(
            upper_matches
        ) == 1:

            upper_column = (
                upper_matches[0]
            )



    if value_column is None:

        excluded_columns = {
            "metric",
            lower_column,
            upper_column,
            "bootstrap_replicates",
            "bootstrap_method",
            "bootstrap_unit",
            "ci_width",
        }


        possible_value_columns = [
            column_name
            for column_name
            in table.columns
            if (
                column_name
                not in excluded_columns
                and
                pd.api.types.is_numeric_dtype(
                    table[
                        column_name
                    ]
                )
            )
        ]


        if len(
            possible_value_columns
        ) == 1:

            value_column = (
                possible_value_columns[0]
            )



    if value_column is None:

        raise RuntimeError(
            f"{dataset_name}: could not identify the "
            "point-estimate column.\n\n"
            f"Available columns:\n"
            f"{table.columns.tolist()}"
        )


    if lower_column is None:

        raise RuntimeError(
            f"{dataset_name}: could not identify the lower "
            "95% CI column.\n\n"
            f"Available columns:\n"
            f"{table.columns.tolist()}"
        )


    if upper_column is None:

        raise RuntimeError(
            f"{dataset_name}: could not identify the upper "
            "95% CI column.\n\n"
            f"Available columns:\n"
            f"{table.columns.tolist()}"
        )


    print(
        "Detected point-estimate column:",
        value_column,
    )

    print(
        "Detected lower-CI column:",
        lower_column,
    )

    print(
        "Detected upper-CI column:",
        upper_column,
    )



    table = table[
        [
            "metric",
            value_column,
            lower_column,
            upper_column,
        ]
    ].copy()


    table = table.rename(
        columns={
            value_column: "value",
            lower_column: "ci_lower_95",
            upper_column: "ci_upper_95",
        }
    )


    table[
        "value"
    ] = pd.to_numeric(
        table[
            "value"
        ],
        errors="raise",
    )


    table[
        "ci_lower_95"
    ] = pd.to_numeric(
        table[
            "ci_lower_95"
        ],
        errors="raise",
    )


    table[
        "ci_upper_95"
    ] = pd.to_numeric(
        table[
            "ci_upper_95"
        ],
        errors="raise",
    )


    assert (
        table[
            "ci_lower_95"
        ]
        <=
        table[
            "ci_upper_95"
        ]
    ).all()


    table[
        "dataset"
    ] = dataset_name


    return table


mimic_internal_metrics_standardised = (
    standardise_metric_ci_table(
        mimic_internal_metrics,
        "MIMIC-IV internal temporal test",
    )
)


mcmed_external_metrics_standardised = (
    standardise_metric_ci_table(
        mcmed_external_metrics_with_ci,
        "MC-MED external validation",
    )
)



def canonicalise_metric_name(
    metric_name,
):

    cleaned_name = (
        str(
            metric_name
        )
        .strip()
        .lower()
        .replace("-", "_")
        .replace(" ", "_")
    )


    while "__" in cleaned_name:

        cleaned_name = cleaned_name.replace(
            "__",
            "_",
        )


    metric_alias_lookup = {

        "auprc": "AUPRC",
        "average_precision": "AUPRC",
        "average_precision_score": "AUPRC",
        "pr_auc": "AUPRC",
        "prauc": "AUPRC",

        "auroc": "AUROC",
        "roc_auc": "AUROC",
        "roc_auc_score": "AUROC",
        "auc_roc": "AUROC",

        "log_loss": "Calibrated log loss",
        "calibrated_log_loss": "Calibrated log loss",
        "calibrated_logloss": "Calibrated log loss",

        "brier": "Calibrated Brier score",
        "brier_score": "Calibrated Brier score",
        "brier_score_loss": "Calibrated Brier score",
        "calibrated_brier": "Calibrated Brier score",
        "calibrated_brier_score": "Calibrated Brier score",

        "accuracy": "Accuracy",
        "accuracy_score": "Accuracy",

        "balanced_accuracy": "Balanced accuracy",
        "balanced_accuracy_score": "Balanced accuracy",

        "sensitivity": "Sensitivity",
        "recall": "Sensitivity",
        "recall_admitted": "Sensitivity",
        "admitted_recall": "Sensitivity",
        "true_positive_rate": "Sensitivity",
        "tpr": "Sensitivity",

        "specificity": "Specificity",
        "true_negative_rate": "Specificity",
        "tnr": "Specificity",

        "ppv": "PPV",
        "precision": "PPV",
        "positive_predictive_value": "PPV",

        "npv": "NPV",
        "negative_predictive_value": "NPV",

        "f1": "F1",
        "f1_score": "F1",
        "admitted_f1": "F1",
    }


    if cleaned_name not in metric_alias_lookup:

        raise RuntimeError(
            "Unrecognised metric label encountered:\n\n"
            f"Original label: {metric_name}\n"
            f"Normalised label: {cleaned_name}"
        )


    return metric_alias_lookup[
        cleaned_name
    ]



print(
    "Original MIMIC-IV internal metric labels:"
)

print(
    mimic_internal_metrics_standardised[
        "metric"
    ].tolist()
)


print(
    "\nOriginal MC-MED external metric labels:"
)

print(
    mcmed_external_metrics_standardised[
        "metric"
    ].tolist()
)



mimic_internal_metrics_standardised[
    "metric_original"
] = (
    mimic_internal_metrics_standardised[
        "metric"
    ]
)


mcmed_external_metrics_standardised[
    "metric_original"
] = (
    mcmed_external_metrics_standardised[
        "metric"
    ]
)


mimic_internal_metrics_standardised[
    "metric"
] = (
    mimic_internal_metrics_standardised[
        "metric"
    ]
    .map(
        canonicalise_metric_name
    )
)


mcmed_external_metrics_standardised[
    "metric"
] = (
    mcmed_external_metrics_standardised[
        "metric"
    ]
    .map(
        canonicalise_metric_name
    )
)



expected_metric_order = [
    "AUPRC",
    "AUROC",
    "Calibrated log loss",
    "Calibrated Brier score",
    "Accuracy",
    "Balanced accuracy",
    "Sensitivity",
    "Specificity",
    "PPV",
    "NPV",
    "F1",
]



assert not mimic_internal_metrics_standardised[
    "metric"
].duplicated().any(), (
    "MIMIC-IV contains duplicate metrics after metric-name "
    "canonicalisation."
)


assert not mcmed_external_metrics_standardised[
    "metric"
].duplicated().any(), (
    "MC-MED contains duplicate metrics after metric-name "
    "canonicalisation."
)



mimic_metric_set = set(
    mimic_internal_metrics_standardised[
        "metric"
    ]
)


mcmed_metric_set = set(
    mcmed_external_metrics_standardised[
        "metric"
    ]
)


expected_metric_set = set(
    expected_metric_order
)


missing_from_mimic = sorted(
    expected_metric_set
    -
    mimic_metric_set
)


extra_in_mimic = sorted(
    mimic_metric_set
    -
    expected_metric_set
)


missing_from_mcmed = sorted(
    expected_metric_set
    -
    mcmed_metric_set
)


extra_in_mcmed = sorted(
    mcmed_metric_set
    -
    expected_metric_set
)


print(
    "\nCanonical MIMIC-IV metric labels:"
)

print(
    mimic_internal_metrics_standardised[
        "metric"
    ].tolist()
)


print(
    "\nCanonical MC-MED metric labels:"
)

print(
    mcmed_external_metrics_standardised[
        "metric"
    ].tolist()
)


print(
    "\nMissing from MIMIC-IV:",
    missing_from_mimic,
)


print(
    "Extra in MIMIC-IV:",
    extra_in_mimic,
)


print(
    "Missing from MC-MED:",
    missing_from_mcmed,
)


print(
    "Extra in MC-MED:",
    extra_in_mcmed,
)


assert not missing_from_mimic, (
    "MIMIC-IV internal-test results are missing required "
    f"metrics:\n{missing_from_mimic}"
)


assert not extra_in_mimic, (
    "MIMIC-IV internal-test results contain unexpected "
    f"metrics:\n{extra_in_mimic}"
)


assert not missing_from_mcmed, (
    "MC-MED external results are missing required "
    f"metrics:\n{missing_from_mcmed}"
)


assert not extra_in_mcmed, (
    "MC-MED external results contain unexpected "
    f"metrics:\n{extra_in_mcmed}"
)


assert (
    mimic_metric_set
    ==
    mcmed_metric_set
    ==
    expected_metric_set
)


print(
    "\nMetric universe alignment: PASSED"
)

print(
    "Metrics matched:",
    len(
        expected_metric_order
    ),
)

print(
    "Metric values changed: NO"
)

print(
    "Confidence intervals changed: NO"
)

print(
    "Only metric labels standardised: YES"
)

mimic_wide = (
    mimic_internal_metrics_standardised
    .drop(
        columns=[
            "dataset",
        ]
    )
    .rename(
        columns={
            "value": (
                "mimic_internal_value"
            ),
            "ci_lower_95": (
                "mimic_internal_ci_lower_95"
            ),
            "ci_upper_95": (
                "mimic_internal_ci_upper_95"
            ),
        }
    )
)


mcmed_wide = (
    mcmed_external_metrics_standardised
    .drop(
        columns=[
            "dataset",
        ]
    )
    .rename(
        columns={
            "value": (
                "mcmed_external_value"
            ),
            "ci_lower_95": (
                "mcmed_external_ci_lower_95"
            ),
            "ci_upper_95": (
                "mcmed_external_ci_upper_95"
            ),
        }
    )
)


internal_external_performance_comparison = (
    mimic_wide
    .merge(
        mcmed_wide,
        on="metric",
        how="inner",
        validate="one_to_one",
    )
)


metric_order_lookup = {
    metric_name: metric_position
    for metric_position, metric_name
    in enumerate(
        expected_metric_order
    )
}


internal_external_performance_comparison[
    "metric_order"
] = (
    internal_external_performance_comparison[
        "metric"
    ]
    .map(
        metric_order_lookup
    )
)


internal_external_performance_comparison = (
    internal_external_performance_comparison
    .sort_values(
        "metric_order"
    )
    .drop(
        columns=[
            "metric_order",
        ]
    )
    .reset_index(
        drop=True
    )
)



internal_external_performance_comparison[
    "external_minus_internal"
] = (
    internal_external_performance_comparison[
        "mcmed_external_value"
    ]
    -
    internal_external_performance_comparison[
        "mimic_internal_value"
    ]
)


internal_external_performance_comparison[
    "absolute_difference"
] = (
    internal_external_performance_comparison[
        "external_minus_internal"
    ]
    .abs()
)



internal_external_performance_comparison[
    "mimic_internal_result"
] = (
    internal_external_performance_comparison.apply(
        lambda row: (
            f"{row['mimic_internal_value']:.3f} "
            f"({row['mimic_internal_ci_lower_95']:.3f}–"
            f"{row['mimic_internal_ci_upper_95']:.3f})"
        ),
        axis=1,
    )
)


internal_external_performance_comparison[
    "mcmed_external_result"
] = (
    internal_external_performance_comparison.apply(
        lambda row: (
            f"{row['mcmed_external_value']:.3f} "
            f"({row['mcmed_external_ci_lower_95']:.3f}–"
            f"{row['mcmed_external_ci_upper_95']:.3f})"
        ),
        axis=1,
    )
)



internal_external_publication_table = (
    internal_external_performance_comparison[
        [
            "metric",
            "mimic_internal_result",
            "mcmed_external_result",
            "external_minus_internal",
        ]
    ]
    .rename(
        columns={
            "metric": "Metric",
            "mimic_internal_result": (
                "MIMIC-IV internal test, value (95% CI)"
            ),
            "mcmed_external_result": (
                "MC-MED external, value (95% CI)"
            ),
            "external_minus_internal": (
                "External − internal"
            ),
        }
    )
)



def find_first_numeric_value(
    dataframe,
    candidate_columns,
):

    for column_name in candidate_columns:

        if column_name in dataframe.columns:

            return float(
                pd.to_numeric(
                    dataframe[
                        column_name
                    ],
                    errors="raise",
                )
                .iloc[
                    0
                ]
            )


    raise RuntimeError(
        "Could not find any of the requested columns:\n"
        f"{candidate_columns}"
    )


mimic_observed_prevalence = (
    find_first_numeric_value(
        mimic_internal_probability_summary,
        [
            "observed_prevalence",
            "observed_admission_prevalence",
            "prevalence",
        ],
    )
)


mimic_mean_raw_probability = (
    find_first_numeric_value(
        mimic_internal_probability_summary,
        [
            "mean_raw_probability",
        ],
    )
)


mimic_mean_calibrated_probability = (
    find_first_numeric_value(
        mimic_internal_probability_summary,
        [
            "mean_calibrated_probability",
        ],
    )
)


mcmed_observed_prevalence = float(
    mcmed_external_probability_summary[
        "observed_admission_prevalence"
    ]
    .iloc[
        0
    ]
)


mcmed_mean_raw_probability = float(
    mcmed_external_probability_summary[
        "mean_raw_probability"
    ]
    .iloc[
        0
    ]
)


mcmed_mean_calibrated_probability = float(
    mcmed_external_probability_summary[
        "mean_calibrated_probability"
    ]
    .iloc[
        0
    ]
)



internal_external_probability_transport = pd.DataFrame(
    [
        {
            "dataset": (
                "MIMIC-IV internal temporal test"
            ),
            "rows": 1851,
            "observed_admission_prevalence": (
                mimic_observed_prevalence
            ),
            "mean_raw_probability": (
                mimic_mean_raw_probability
            ),
            "mean_calibrated_probability": (
                mimic_mean_calibrated_probability
            ),
            "calibration_gap_mean_probability_minus_prevalence": (
                mimic_mean_calibrated_probability
                -
                mimic_observed_prevalence
            ),
        },
        {
            "dataset": (
                "MC-MED external validation"
            ),
            "rows": 6707,
            "observed_admission_prevalence": (
                mcmed_observed_prevalence
            ),
            "mean_raw_probability": (
                mcmed_mean_raw_probability
            ),
            "mean_calibrated_probability": (
                mcmed_mean_calibrated_probability
            ),
            "calibration_gap_mean_probability_minus_prevalence": (
                mcmed_mean_calibrated_probability
                -
                mcmed_observed_prevalence
            ),
        },
    ]
)



internal_auprc = float(
    internal_external_performance_comparison.loc[
        internal_external_performance_comparison[
            "metric"
        ].eq(
            "AUPRC"
        ),
        "mimic_internal_value",
    ]
    .iloc[
        0
    ]
)


external_auprc = float(
    internal_external_performance_comparison.loc[
        internal_external_performance_comparison[
            "metric"
        ].eq(
            "AUPRC"
        ),
        "mcmed_external_value",
    ]
    .iloc[
        0
    ]
)


internal_external_auprc_context = pd.DataFrame(
    [
        {
            "dataset": (
                "MIMIC-IV internal temporal test"
            ),
            "prevalence": (
                mimic_observed_prevalence
            ),
            "AUPRC": (
                internal_auprc
            ),
            "AUPRC_divided_by_prevalence": (
                internal_auprc
                /
                mimic_observed_prevalence
            ),
        },
        {
            "dataset": (
                "MC-MED external validation"
            ),
            "prevalence": (
                mcmed_observed_prevalence
            ),
            "AUPRC": (
                external_auprc
            ),
            "AUPRC_divided_by_prevalence": (
                external_auprc
                /
                mcmed_observed_prevalence
            ),
        },
    ]
)



internal_external_comparison_audit = pd.DataFrame(
    [
        {
            "algorithm": "XGBoost",
            "calibration": "Platt scaling",
            "threshold": float(
                LOCKED_DECISION_THRESHOLD
            ),
            "mimic_internal_rows": 1851,
            "mcmed_external_rows": 6707,
            "metrics_compared": len(
                expected_metric_order
            ),
            "same_metric_definitions": True,
            "same_bootstrap_replicates": 2000,
            "same_bootstrap_seed": 42,
            "model_refitted": False,
            "preprocessor_refitted": False,
            "calibrator_refitted": False,
            "threshold_changed": False,
        }
    ]
)



NOTEBOOK3_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook3"
)


NOTEBOOK3_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


comparison_output_paths = {
    "full_comparison": (
        NOTEBOOK3_TABLE_DIR
        / "internal_vs_external_performance_comparison.csv"
    ),
    "publication_table": (
        NOTEBOOK3_TABLE_DIR
        / "internal_vs_external_publication_table.csv"
    ),
    "probability_transport": (
        NOTEBOOK3_TABLE_DIR
        / "internal_vs_external_probability_transport.csv"
    ),
    "auprc_context": (
        NOTEBOOK3_TABLE_DIR
        / "internal_vs_external_auprc_context.csv"
    ),
    "comparison_audit": (
        NOTEBOOK3_TABLE_DIR
        / "internal_vs_external_comparison_audit.csv"
    ),
}


internal_external_performance_comparison.to_csv(
    comparison_output_paths[
        "full_comparison"
    ],
    index=False,
)


internal_external_publication_table.to_csv(
    comparison_output_paths[
        "publication_table"
    ],
    index=False,
)


internal_external_probability_transport.to_csv(
    comparison_output_paths[
        "probability_transport"
    ],
    index=False,
)


internal_external_auprc_context.to_csv(
    comparison_output_paths[
        "auprc_context"
    ],
    index=False,
)


internal_external_comparison_audit.to_csv(
    comparison_output_paths[
        "comparison_audit"
    ],
    index=False,
)



print(
    "MIMIC-IV internal temporal test versus "
    "MC-MED external validation:"
)


display(
    internal_external_publication_table.round(
        {
            "External − internal": 4,
        }
    )
)


print(
    "\nProbability transport:"
)


display(
    internal_external_probability_transport.round(
        6
    )
)


print(
    "\nAUPRC relative to admission prevalence:"
)


display(
    internal_external_auprc_context.round(
        6
    )
)


print(
    "\nComparison audit:"
)


display(
    internal_external_comparison_audit
)



print(
    "\nINTERNAL VERSUS EXTERNAL PERFORMANCE COMPARISON COMPLETED"
)


print(
    "MIMIC-IV internal temporal-test rows:",
    "1,851",
)


print(
    "MC-MED external-validation rows:",
    "6,707",
)


print(
    "Metrics compared:",
    len(
        expected_metric_order
    ),
)


print(
    "Both evaluations include 95% bootstrap CI: YES"
)


print(
    "Same locked algorithm: YES"
)


print(
    "Same Platt calibrator: YES"
)


print(
    "Same threshold 0.282: YES"
)


print(
    "\nModel refitted: NO"
)


print(
    "Calibration changed: NO"
)


print(
    "Threshold changed: NO"
)

### Locked external-validation figures and reproducibility record

Core external-validation figures are generated from the already-computed MC-MED probabilities, classifications and bootstrap confidence intervals.

The figures document discrimination, precision-recall performance, calibration, classification performance, probability transport and internal-versus-external performance. No model predictions are regenerated for model-development purposes and no fitting, recalibration or threshold optimisation occurs.

Notebook 3 does not create a new model artifact. Instead, the canonical Notebook 2 XGBoost pipeline, Platt calibrator and locked model package are referenced by their original paths and SHA-256 checksums to preserve model provenance.

In [ ]:



required_figure_objects = [
    "PROJECT_ROOT",
    "y_mcmed_external",
    "mcmed_external_raw_probability",
    "mcmed_external_calibrated_probability",
    "mcmed_external_prediction",
    "mcmed_external_metrics_with_ci",
    "mcmed_external_confusion_matrix",
    "internal_external_performance_comparison",
    "internal_external_probability_transport",
    "LOCKED_DECISION_THRESHOLD",
    "notebook2_handoff_paths",
]


missing_figure_objects = [
    object_name
    for object_name
    in required_figure_objects
    if object_name not in globals()
]


if missing_figure_objects:

    raise RuntimeError(
        "Complete the external prediction, bootstrap and "
        "internal-versus-external comparison sections first.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_figure_objects
        )
    )



NOTEBOOK3_FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "notebook3"
)


NOTEBOOK3_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook3"
)


NOTEBOOK3_CONFIG_DIR = (
    PROJECT_ROOT
    / "results"
    / "config"
    / "notebook3"
)


for directory in [
    NOTEBOOK3_FIGURE_DIR,
    NOTEBOOK3_TABLE_DIR,
    NOTEBOOK3_CONFIG_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )



notebook3_figure_paths = {
    "external_roc_png": (
        NOTEBOOK3_FIGURE_DIR
        / "01_mcmed_external_roc_curve.png"
    ),
    "external_roc_pdf": (
        NOTEBOOK3_FIGURE_DIR
        / "01_mcmed_external_roc_curve.pdf"
    ),

    "external_pr_png": (
        NOTEBOOK3_FIGURE_DIR
        / "02_mcmed_external_precision_recall_curve.png"
    ),
    "external_pr_pdf": (
        NOTEBOOK3_FIGURE_DIR
        / "02_mcmed_external_precision_recall_curve.pdf"
    ),

    "external_reliability_png": (
        NOTEBOOK3_FIGURE_DIR
        / "03_mcmed_external_reliability_curve.png"
    ),
    "external_reliability_pdf": (
        NOTEBOOK3_FIGURE_DIR
        / "03_mcmed_external_reliability_curve.pdf"
    ),

    "external_confusion_png": (
        NOTEBOOK3_FIGURE_DIR
        / "04_mcmed_external_confusion_matrix.png"
    ),
    "external_confusion_pdf": (
        NOTEBOOK3_FIGURE_DIR
        / "04_mcmed_external_confusion_matrix.pdf"
    ),

    "internal_external_metrics_png": (
        NOTEBOOK3_FIGURE_DIR
        / "05_internal_vs_external_performance.png"
    ),
    "internal_external_metrics_pdf": (
        NOTEBOOK3_FIGURE_DIR
        / "05_internal_vs_external_performance.pdf"
    ),

    "probability_transport_png": (
        NOTEBOOK3_FIGURE_DIR
        / "06_internal_external_probability_transport.png"
    ),
    "probability_transport_pdf": (
        NOTEBOOK3_FIGURE_DIR
        / "06_internal_external_probability_transport.pdf"
    ),
}



external_fpr, external_tpr, _ = roc_curve(
    y_mcmed_external,
    mcmed_external_calibrated_probability,
)


external_roc_auc = roc_auc_score(
    y_mcmed_external,
    mcmed_external_calibrated_probability,
)


figure, axis = plt.subplots(
    figsize=(
        8,
        7,
    )
)


axis.plot(
    external_fpr,
    external_tpr,
    linewidth=2,
    label=(
        f"MC-MED AUROC = "
        f"{external_roc_auc:.3f}"
    ),
)


axis.plot(
    [
        0,
        1,
    ],
    [
        0,
        1,
    ],
    linestyle="--",
    linewidth=1,
)


axis.set_xlim(
    0,
    1,
)


axis.set_ylim(
    0,
    1,
)


axis.set_xlabel(
    "False positive rate"
)


axis.set_ylabel(
    "True positive rate"
)


axis.set_title(
    "MC-MED external validation ROC curve"
)


axis.legend(
    frameon=False
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


figure.savefig(
    notebook3_figure_paths[
        "external_roc_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


figure.savefig(
    notebook3_figure_paths[
        "external_roc_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    figure
)



external_precision_curve, external_recall_curve, _ = (
    precision_recall_curve(
        y_mcmed_external,
        mcmed_external_calibrated_probability,
    )
)


external_auprc_value = average_precision_score(
    y_mcmed_external,
    mcmed_external_calibrated_probability,
)


external_prevalence_value = float(
    y_mcmed_external.mean()
)


figure, axis = plt.subplots(
    figsize=(
        8,
        7,
    )
)


axis.plot(
    external_recall_curve,
    external_precision_curve,
    linewidth=2,
    label=(
        f"MC-MED AUPRC = "
        f"{external_auprc_value:.3f}"
    ),
)


axis.axhline(
    y=external_prevalence_value,
    linestyle="--",
    linewidth=1,
    label=(
        "Admission prevalence = "
        f"{external_prevalence_value:.3f}"
    ),
)


axis.set_xlim(
    0,
    1,
)


axis.set_ylim(
    0,
    1,
)


axis.set_xlabel(
    "Recall / sensitivity"
)


axis.set_ylabel(
    "Precision / PPV"
)


axis.set_title(
    "MC-MED external validation precision-recall curve"
)


axis.legend(
    frameon=False
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


figure.savefig(
    notebook3_figure_paths[
        "external_pr_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


figure.savefig(
    notebook3_figure_paths[
        "external_pr_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    figure
)



external_raw_fraction_positive, external_raw_mean_predicted = (
    calibration_curve(
        y_mcmed_external,
        mcmed_external_raw_probability,
        n_bins=10,
        strategy="quantile",
    )
)


(
    external_calibrated_fraction_positive,
    external_calibrated_mean_predicted,
) = calibration_curve(
    y_mcmed_external,
    mcmed_external_calibrated_probability,
    n_bins=10,
    strategy="quantile",
)


figure, axis = plt.subplots(
    figsize=(
        8,
        7,
    )
)


axis.plot(
    [
        0,
        1,
    ],
    [
        0,
        1,
    ],
    linestyle="--",
    linewidth=1,
    label="Perfect calibration",
)


axis.plot(
    external_raw_mean_predicted,
    external_raw_fraction_positive,
    marker="o",
    linewidth=2,
    label="Raw XGBoost",
)


axis.plot(
    external_calibrated_mean_predicted,
    external_calibrated_fraction_positive,
    marker="o",
    linewidth=2,
    label="Platt calibrated",
)


axis.set_xlim(
    0,
    1,
)


axis.set_ylim(
    0,
    1,
)


axis.set_xlabel(
    "Mean predicted admission probability"
)


axis.set_ylabel(
    "Observed admission proportion"
)


axis.set_title(
    "MC-MED external validation reliability curve"
)


axis.legend(
    frameon=False
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


figure.savefig(
    notebook3_figure_paths[
        "external_reliability_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


figure.savefig(
    notebook3_figure_paths[
        "external_reliability_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    figure
)



external_confusion_array = confusion_matrix(
    y_mcmed_external,
    mcmed_external_prediction,
    labels=[
        0,
        1,
    ],
)


figure, axis = plt.subplots(
    figsize=(
        7,
        6,
    )
)


confusion_image = axis.imshow(
    external_confusion_array
)


axis.set_xticks(
    [
        0,
        1,
    ]
)


axis.set_xticklabels(
    [
        "HOME",
        "ADMITTED",
    ]
)


axis.set_yticks(
    [
        0,
        1,
    ]
)


axis.set_yticklabels(
    [
        "HOME",
        "ADMITTED",
    ]
)


axis.set_xlabel(
    "Predicted"
)


axis.set_ylabel(
    "Observed"
)


axis.set_title(
    "MC-MED external validation confusion matrix"
)


for row_position in range(
    2
):

    for column_position in range(
        2
    ):

        axis.text(
            column_position,
            row_position,
            f"{external_confusion_array[row_position, column_position]:,}",
            ha="center",
            va="center",
            fontsize=13,
        )


figure.colorbar(
    confusion_image,
    ax=axis,
    label="Encounters",
)


figure.tight_layout()


figure.savefig(
    notebook3_figure_paths[
        "external_confusion_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


figure.savefig(
    notebook3_figure_paths[
        "external_confusion_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    figure
)



comparison_plot_metrics = [
    "AUPRC",
    "AUROC",
    "Balanced accuracy",
    "Sensitivity",
    "Specificity",
    "PPV",
    "NPV",
    "F1",
]


comparison_plot_table = (
    internal_external_performance_comparison.loc[
        internal_external_performance_comparison[
            "metric"
        ].isin(
            comparison_plot_metrics
        )
    ]
    .copy()
)


comparison_metric_order = {
    metric_name: position
    for position, metric_name
    in enumerate(
        comparison_plot_metrics
    )
}


comparison_plot_table[
    "plot_order"
] = (
    comparison_plot_table[
        "metric"
    ]
    .map(
        comparison_metric_order
    )
)


comparison_plot_table = (
    comparison_plot_table
    .sort_values(
        "plot_order"
    )
    .reset_index(
        drop=True
    )
)


plot_positions = np.arange(
    len(
        comparison_plot_table
    )
)


plot_width = 0.36


mimic_lower_error = (
    comparison_plot_table[
        "mimic_internal_value"
    ]
    -
    comparison_plot_table[
        "mimic_internal_ci_lower_95"
    ]
)


mimic_upper_error = (
    comparison_plot_table[
        "mimic_internal_ci_upper_95"
    ]
    -
    comparison_plot_table[
        "mimic_internal_value"
    ]
)


mcmed_lower_error = (
    comparison_plot_table[
        "mcmed_external_value"
    ]
    -
    comparison_plot_table[
        "mcmed_external_ci_lower_95"
    ]
)


mcmed_upper_error = (
    comparison_plot_table[
        "mcmed_external_ci_upper_95"
    ]
    -
    comparison_plot_table[
        "mcmed_external_value"
    ]
)


figure, axis = plt.subplots(
    figsize=(
        13,
        7,
    )
)


axis.errorbar(
    plot_positions
    -
    plot_width
    /
    2,
    comparison_plot_table[
        "mimic_internal_value"
    ],
    yerr=np.vstack(
        [
            mimic_lower_error,
            mimic_upper_error,
        ]
    ),
    fmt="o",
    capsize=4,
    label="MIMIC-IV internal test",
)


axis.errorbar(
    plot_positions
    +
    plot_width
    /
    2,
    comparison_plot_table[
        "mcmed_external_value"
    ],
    yerr=np.vstack(
        [
            mcmed_lower_error,
            mcmed_upper_error,
        ]
    ),
    fmt="o",
    capsize=4,
    label="MC-MED external validation",
)


axis.set_xticks(
    plot_positions
)


axis.set_xticklabels(
    comparison_plot_table[
        "metric"
    ],
    rotation=35,
    ha="right",
)


axis.set_ylim(
    0,
    1,
)


axis.set_ylabel(
    "Performance"
)


axis.set_title(
    "Locked model: internal versus external validation"
)


axis.legend(
    frameon=False
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


figure.savefig(
    notebook3_figure_paths[
        "internal_external_metrics_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


figure.savefig(
    notebook3_figure_paths[
        "internal_external_metrics_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    figure
)



probability_transport_plot = (
    internal_external_probability_transport
    .copy()
)


transport_positions = np.arange(
    len(
        probability_transport_plot
    )
)


transport_width = 0.24


figure, axis = plt.subplots(
    figsize=(
        10,
        7,
    )
)


axis.bar(
    transport_positions
    -
    transport_width,
    probability_transport_plot[
        "observed_admission_prevalence"
    ],
    width=transport_width,
    label="Observed prevalence",
)


axis.bar(
    transport_positions,
    probability_transport_plot[
        "mean_raw_probability"
    ],
    width=transport_width,
    label="Mean raw probability",
)


axis.bar(
    transport_positions
    +
    transport_width,
    probability_transport_plot[
        "mean_calibrated_probability"
    ],
    width=transport_width,
    label="Mean calibrated probability",
)


axis.set_xticks(
    transport_positions
)


axis.set_xticklabels(
    [
        "MIMIC-IV\ninternal test",
        "MC-MED\nexternal validation",
    ]
)


axis.set_ylim(
    0,
    0.5,
)


axis.set_ylabel(
    "Proportion / mean probability"
)


axis.set_title(
    "Probability transport across internal and external validation"
)


axis.legend(
    frameon=False
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


figure.savefig(
    notebook3_figure_paths[
        "probability_transport_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


figure.savefig(
    notebook3_figure_paths[
        "probability_transport_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    figure
)



locked_model_reference_records = []


for artifact_name in [
    "locked_pipeline",
    "platt_calibrator",
    "locked_model_package",
    "model_config",
]:

    artifact_path = (
        notebook2_handoff_paths[
            artifact_name
        ]
    )


    assert artifact_path.exists()


    locked_model_reference_records.append(
        {
            "artifact": artifact_name,
            "canonical_path": str(
                artifact_path
            ),
            "size_bytes": int(
                artifact_path
                .stat()
                .st_size
            ),
            "sha256": file_sha256(
                artifact_path
            ),
            "canonical_source_notebook": (
                "Notebook 2"
            ),
            "duplicated_in_notebook3": False,
        }
    )


notebook3_locked_model_reference = pd.DataFrame(
    locked_model_reference_records
)


LOCKED_MODEL_REFERENCE_PATH = (
    NOTEBOOK3_TABLE_DIR
    / "notebook3_locked_model_reference.csv"
)


notebook3_locked_model_reference.to_csv(
    LOCKED_MODEL_REFERENCE_PATH,
    index=False,
)



notebook3_external_validation_config = {
    "notebook": (
        "03_external_validation.ipynb"
    ),
    "development_dataset": (
        "MIMIC-IV"
    ),
    "external_dataset": (
        "MC-MED"
    ),
    "external_rows": 6707,
    "external_home": 4872,
    "external_admitted": 1835,
    "external_prevalence": float(
        y_mcmed_external.mean()
    ),
    "algorithm": "XGBoost",
    "raw_features": 2495,
    "prepared_model_columns": 2498,
    "feature_order_sha256": (
        LOCKED_FEATURE_ORDER_SHA256
    ),
    "calibration": "Platt scaling",
    "calibrator_intercept": float(
        LOCKED_CALIBRATOR_INTERCEPT
    ),
    "calibrator_slope": float(
        LOCKED_CALIBRATOR_SLOPE
    ),
    "decision_threshold": float(
        LOCKED_DECISION_THRESHOLD
    ),
    "bootstrap": {
        "replicates": 2000,
        "seed": 42,
        "confidence_level": 0.95,
        "interval_method": "Percentile",
    },
    "model_provenance": {
        row[
            "artifact"
        ]: {
            "path": (
                row[
                    "canonical_path"
                ]
            ),
            "sha256": (
                row[
                    "sha256"
                ]
            ),
        }
        for row
        in notebook3_locked_model_reference.to_dict(
            orient="records"
        )
    },
    "external_validation_boundaries": {
        "model_refitted": False,
        "preprocessor_refitted": False,
        "calibrator_refitted": False,
        "threshold_changed": False,
        "external_labels_used_for_prediction": False,
    },
    "software_versions": {
        "python": platform.python_version(),
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__,
        "shap": shap.__version__,
    },
    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}


NOTEBOOK3_CONFIG_PATH = (
    NOTEBOOK3_CONFIG_DIR
    / "notebook3_external_validation_config.json"
)


with NOTEBOOK3_CONFIG_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        notebook3_external_validation_config,
        file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )



figure_manifest_records = []


for figure_name, figure_path in (
    notebook3_figure_paths.items()
):

    assert figure_path.exists()


    figure_manifest_records.append(
        {
            "artifact": figure_name,
            "filename": figure_path.name,
            "directory": str(
                figure_path.parent
            ),
            "size_bytes": int(
                figure_path
                .stat()
                .st_size
            ),
            "sha256": file_sha256(
                figure_path
            ),
        }
    )


notebook3_figure_manifest = pd.DataFrame(
    figure_manifest_records
)


NOTEBOOK3_FIGURE_MANIFEST_PATH = (
    NOTEBOOK3_FIGURE_DIR
    / "notebook3_figure_manifest.csv"
)


notebook3_figure_manifest.to_csv(
    NOTEBOOK3_FIGURE_MANIFEST_PATH,
    index=False,
)



print(
    "Locked Notebook 2 model references:"
)


display(
    notebook3_locked_model_reference
)


print(
    "\nNotebook 3 figure manifest:"
)


display(
    notebook3_figure_manifest
)


print(
    "\nEXTERNAL VALIDATION FIGURES AND MODEL REFERENCE SAVED"
)


print(
    "Figures saved:",
    len(
        notebook3_figure_manifest
    ),
)


print(
    "PNG resolution:",
    "600 dpi",
)


print(
    "Vector PDF versions:",
    "YES",
)


print(
    "\nExternal result tables already saved: YES"
)


print(
    "Patient-level MC-MED predictions saved: YES"
)


print(
    "Bootstrap distribution saved: YES"
)


print(
    "Internal-vs-external publication table saved: YES"
)


print(
    "\nNotebook 2 model duplicated: NO"
)


print(
    "Canonical locked-model references saved: YES"
)


print(
    "Canonical locked-model SHA-256 checksums saved: YES"
)


print(
    "Notebook 3 reproducibility configuration saved: YES"
)


print(
    "\nModel refitted: NO"
)


print(
    "Calibration changed: NO"
)


print(
    "Threshold changed: NO"
)

## MC-MED external grouped SHAP interpretation

SHAP contributions are calculated for all MC-MED external-validation encounters using the unchanged locked XGBoost model and XGBoost's native `pred_contribs` implementation.

The 2,498 prepared model-input contributions are mapped back to the locked 2,495 raw-feature universe. One-hot contributions for `sex_gender` and `arrival_transport_grouped` are summed patient by patient before calculating feature importance or producing clinically interpretable SHAP plots.

Global feature importance is defined as the mean absolute grouped SHAP value across all external encounters. A grouped raw-feature beeswarm visualises both the magnitude and direction of patient-level model contributions.

The complete patient-by-feature grouped SHAP matrix and corresponding feature-value matrix are retained for subsequent robust distance-from-zero, minimum/maximum distance, stability, cross-dataset and clinician-comparison analyses.

No model fitting, preprocessing fitting, recalibration, threshold modification or feature selection is performed.

In [ ]:



required_mcmed_shap_objects = [
    "PROJECT_ROOT",
    "mcmed_external",
    "X_mcmed_external_standardised",
    "X_mcmed_external_transformed",
    "y_mcmed_external",
    "mcmed_external_raw_probability",
    "locked_external_xgboost",
    "locked_external_preprocessor",
    "LOCKED_RAW_FEATURE_ORDER",
    "LOCKED_S0_NUMERIC_FEATURES",
    "LOCKED_S0_CATEGORICAL_FEATURES",
    "LOCKED_BINARY_FEATURES",
    "clinician_comparison_feature_universe",
]


missing_mcmed_shap_objects = [
    object_name
    for object_name in required_mcmed_shap_objects
    if object_name not in globals()
]


if missing_mcmed_shap_objects:

    raise RuntimeError(
        "Complete the MC-MED external-validation prediction "
        "and feature-compatibility sections first.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_mcmed_shap_objects
        )
    )


assert X_mcmed_external_standardised.shape == (
    6707,
    2495,
)


assert X_mcmed_external_transformed.shape == (
    6707,
    2498,
)


assert len(
    LOCKED_RAW_FEATURE_ORDER
) == 2495


assert len(
    LOCKED_S0_NUMERIC_FEATURES
) == 7


assert len(
    LOCKED_S0_CATEGORICAL_FEATURES
) == 2


assert len(
    LOCKED_BINARY_FEATURES
) == 2486


assert set(
    LOCKED_S0_CATEGORICAL_FEATURES
) == {
    "sex_gender",
    "arrival_transport_grouped",
}



mcmed_shap_categorical_encoder = (
    locked_external_preprocessor
    .named_transformers_[
        "s0_categorical"
    ]
    .named_steps[
        "one_hot_encoder"
    ]
)


mcmed_shap_training_categories = (
    mcmed_shap_categorical_encoder
    .categories_
)


assert len(
    mcmed_shap_training_categories
) == 2


mcmed_encoded_categorical_feature_names = (
    mcmed_shap_categorical_encoder
    .get_feature_names_out(
        LOCKED_S0_CATEGORICAL_FEATURES
    )
    .tolist()
)


MC_MED_ENCODED_CATEGORICAL_COUNT = len(
    mcmed_encoded_categorical_feature_names
)


assert MC_MED_ENCODED_CATEGORICAL_COUNT == 5


MC_MED_PREPARED_FEATURE_COUNT = (
    len(
        LOCKED_S0_NUMERIC_FEATURES
    )
    +
    MC_MED_ENCODED_CATEGORICAL_COUNT
    +
    len(
        LOCKED_BINARY_FEATURES
    )
)


assert MC_MED_PREPARED_FEATURE_COUNT == 2498



mcmed_prepared_mapping_records = []

prepared_position = 0


for feature_name in (
    LOCKED_S0_NUMERIC_FEATURES
):

    mcmed_prepared_mapping_records.append(
        {
            "prepared_position": prepared_position,
            "prepared_feature_name": feature_name,
            "source_raw_feature": feature_name,
            "category_level": pd.NA,
            "grouped_categorical": False,
        }
    )


    prepared_position += 1


encoded_offset = 0


for (
    raw_categorical_feature,
    category_levels,
) in zip(
    LOCKED_S0_CATEGORICAL_FEATURES,
    mcmed_shap_training_categories,
):

    n_categories = len(
        category_levels
    )


    encoded_names_for_feature = (
        mcmed_encoded_categorical_feature_names[
            encoded_offset:
            encoded_offset
            +
            n_categories
        ]
    )


    assert len(
        encoded_names_for_feature
    ) == n_categories


    for (
        encoded_feature_name,
        category_level,
    ) in zip(
        encoded_names_for_feature,
        category_levels,
    ):

        mcmed_prepared_mapping_records.append(
            {
                "prepared_position": prepared_position,
                "prepared_feature_name": (
                    encoded_feature_name
                ),
                "source_raw_feature": (
                    raw_categorical_feature
                ),
                "category_level": str(
                    category_level
                ),
                "grouped_categorical": True,
            }
        )


        prepared_position += 1


    encoded_offset += n_categories


assert encoded_offset == 5


for feature_name in (
    LOCKED_BINARY_FEATURES
):

    mcmed_prepared_mapping_records.append(
        {
            "prepared_position": prepared_position,
            "prepared_feature_name": feature_name,
            "source_raw_feature": feature_name,
            "category_level": pd.NA,
            "grouped_categorical": False,
        }
    )


    prepared_position += 1


mcmed_shap_prepared_feature_mapping = pd.DataFrame(
    mcmed_prepared_mapping_records
)


assert len(
    mcmed_shap_prepared_feature_mapping
) == 2498


assert (
    mcmed_shap_prepared_feature_mapping[
        "prepared_position"
    ].tolist()
    ==
    list(
        range(
            2498
        )
    )
)


assert set(
    mcmed_shap_prepared_feature_mapping[
        "source_raw_feature"
    ]
) == set(
    LOCKED_RAW_FEATURE_ORDER
)


mcmed_prepared_feature_order = (
    mcmed_shap_prepared_feature_mapping[
        "prepared_feature_name"
    ]
    .tolist()
)



locked_feature_metadata = (
    clinician_comparison_feature_universe[
        [
            "source_raw_feature",
            "family",
            "timing_stage",
            "storage_type",
        ]
    ]
    .drop_duplicates(
        subset=[
            "source_raw_feature",
        ]
    )
)


assert len(
    locked_feature_metadata
) == 2495


mcmed_shap_prepared_feature_mapping = (
    mcmed_shap_prepared_feature_mapping
    .merge(
        locked_feature_metadata,
        on="source_raw_feature",
        how="left",
        validate="many_to_one",
    )
    .sort_values(
        "prepared_position"
    )
    .reset_index(
        drop=True
    )
)


assert mcmed_shap_prepared_feature_mapping[
    "family"
].notna().all()



mcmed_shap_start_time = (
    time.perf_counter()
)


locked_mcmed_booster = (
    locked_external_xgboost
    .get_booster()
)


mcmed_shap_dmatrix = xgb.DMatrix(
    X_mcmed_external_transformed
)


mcmed_native_prediction_kwargs = {
    "validate_features": False,
}


try:

    mcmed_locked_best_iteration = (
        locked_external_xgboost
        .best_iteration
    )

except Exception:

    mcmed_locked_best_iteration = None


if mcmed_locked_best_iteration is not None:

    mcmed_native_prediction_kwargs[
        "iteration_range"
    ] = (
        0,
        int(
            mcmed_locked_best_iteration
        )
        +
        1,
    )


mcmed_native_contributions = (
    locked_mcmed_booster
    .predict(
        mcmed_shap_dmatrix,
        pred_contribs=True,
        approx_contribs=False,
        **mcmed_native_prediction_kwargs,
    )
)


mcmed_native_contributions = np.asarray(
    mcmed_native_contributions,
    dtype=np.float32,
)


if mcmed_native_contributions.ndim == 3:

    if mcmed_native_contributions.shape[
        1
    ] == 1:

        mcmed_native_contributions = (
            mcmed_native_contributions[
                :,
                0,
                :,
            ]
        )

    elif mcmed_native_contributions.shape[
        2
    ] == 1:

        mcmed_native_contributions = (
            mcmed_native_contributions[
                :,
                :,
                0,
            ]
        )

    else:

        raise RuntimeError(
            "Unexpected XGBoost SHAP contribution shape:\n"
            f"{mcmed_native_contributions.shape}"
        )


assert mcmed_native_contributions.shape == (
    6707,
    2499,
), (
    "Expected 2,498 feature contributions plus one bias "
    "column."
)


mcmed_prepared_shap_values = (
    mcmed_native_contributions[
        :,
        :-1,
    ]
)


mcmed_shap_bias = (
    mcmed_native_contributions[
        :,
        -1,
    ]
)


assert mcmed_prepared_shap_values.shape == (
    6707,
    2498,
)


assert mcmed_shap_bias.shape == (
    6707,
)


assert np.isfinite(
    mcmed_prepared_shap_values
).all()


assert np.isfinite(
    mcmed_shap_bias
).all()



mcmed_native_margin = (
    locked_mcmed_booster
    .predict(
        mcmed_shap_dmatrix,
        output_margin=True,
        **mcmed_native_prediction_kwargs,
    )
)


mcmed_native_margin = np.asarray(
    mcmed_native_margin,
    dtype=np.float32,
).reshape(
    -1
)


mcmed_reconstructed_margin = (
    mcmed_shap_bias
    +
    mcmed_prepared_shap_values.sum(
        axis=1
    )
)


mcmed_shap_additivity_error = np.abs(
    mcmed_native_margin
    -
    mcmed_reconstructed_margin
)


assert np.allclose(
    mcmed_native_margin,
    mcmed_reconstructed_margin,
    rtol=1e-5,
    atol=1e-4,
), (
    "MC-MED native SHAP values failed the additivity audit.\n"
    f"Maximum absolute error: "
    f"{mcmed_shap_additivity_error.max()}"
)



mcmed_native_probability = (
    locked_mcmed_booster
    .predict(
        mcmed_shap_dmatrix,
        **mcmed_native_prediction_kwargs,
    )
)


mcmed_native_probability = np.asarray(
    mcmed_native_probability,
    dtype=float,
).reshape(
    -1
)


mcmed_shap_probability_difference = np.abs(
    mcmed_native_probability
    -
    np.asarray(
        mcmed_external_raw_probability,
        dtype=float,
    )
)


assert np.allclose(
    mcmed_native_probability,
    mcmed_external_raw_probability,
    rtol=1e-6,
    atol=1e-6,
), (
    "Native XGBoost SHAP prediction does not reproduce the "
    "locked MC-MED raw probabilities.\n"
    f"Maximum difference: "
    f"{mcmed_shap_probability_difference.max()}"
)



mcmed_raw_feature_position_lookup = {
    feature_name: feature_position
    for feature_position, feature_name
    in enumerate(
        LOCKED_RAW_FEATURE_ORDER
    )
}


mcmed_grouped_raw_shap_values = np.zeros(
    (
        6707,
        2495,
    ),
    dtype=np.float32,
)


for mapping_row in (
    mcmed_shap_prepared_feature_mapping
    .itertuples(
        index=False
    )
):

    raw_position = (
        mcmed_raw_feature_position_lookup[
            mapping_row.source_raw_feature
        ]
    )


    mcmed_grouped_raw_shap_values[
        :,
        raw_position,
    ] += mcmed_prepared_shap_values[
        :,
        mapping_row.prepared_position,
    ]


assert mcmed_grouped_raw_shap_values.shape == (
    6707,
    2495,
)


assert np.isfinite(
    mcmed_grouped_raw_shap_values
).all()


mcmed_grouping_error = np.abs(
    mcmed_prepared_shap_values.sum(
        axis=1
    )
    -
    mcmed_grouped_raw_shap_values.sum(
        axis=1
    )
)


assert np.allclose(
    mcmed_prepared_shap_values.sum(
        axis=1
    ),
    mcmed_grouped_raw_shap_values.sum(
        axis=1
    ),
    rtol=1e-5,
    atol=1e-5,
), (
    "Patient-level SHAP totals changed during categorical "
    "grouping."
)



mcmed_grouped_raw_feature_values = np.full(
    (
        6707,
        2495,
    ),
    np.nan,
    dtype=np.float32,
)


n_numeric = len(
    LOCKED_S0_NUMERIC_FEATURES
)


if sparse.issparse(
    X_mcmed_external_transformed
):

    mcmed_numeric_values = (
        X_mcmed_external_transformed[
            :,
            :n_numeric,
        ]
        .toarray()
        .astype(
            np.float32
        )
    )

else:

    mcmed_numeric_values = np.asarray(
        X_mcmed_external_transformed[
            :,
            :n_numeric,
        ],
        dtype=np.float32,
    )


numeric_raw_positions = [
    mcmed_raw_feature_position_lookup[
        feature_name
    ]
    for feature_name
    in LOCKED_S0_NUMERIC_FEATURES
]


mcmed_grouped_raw_feature_values[
    :,
    numeric_raw_positions,
] = mcmed_numeric_values



mcmed_categorical_code_records = []


for (
    feature_name,
    training_categories,
) in zip(
    LOCKED_S0_CATEGORICAL_FEATURES,
    mcmed_shap_training_categories,
):

    category_strings = [
        str(
            category
        )
        for category
        in training_categories
    ]


    category_to_code = {
        category_value: category_code
        for category_code, category_value
        in enumerate(
            category_strings
        )
    }


    external_category_values = (
        X_mcmed_external_standardised[
            feature_name
        ]
        .astype(
            "string"
        )
        .fillna(
            "__MISSING__"
        )
        .astype(str)
    )


    external_category_codes = (
        external_category_values
        .map(
            category_to_code
        )
        .fillna(
            -1
        )
        .to_numpy(
            dtype=np.float32
        )
    )


    raw_position = (
        mcmed_raw_feature_position_lookup[
            feature_name
        ]
    )


    mcmed_grouped_raw_feature_values[
        :,
        raw_position,
    ] = external_category_codes


    for (
        category_value,
        category_code,
    ) in category_to_code.items():

        mcmed_categorical_code_records.append(
            {
                "raw_categorical_feature": (
                    feature_name
                ),
                "beeswarm_category_code": int(
                    category_code
                ),
                "category_level": (
                    category_value
                ),
            }
        )


mcmed_categorical_beeswarm_code_mapping = (
    pd.DataFrame(
        mcmed_categorical_code_records
    )
)


binary_prepared_start = (
    len(
        LOCKED_S0_NUMERIC_FEATURES
    )
    +
    MC_MED_ENCODED_CATEGORICAL_COUNT
)


binary_prepared_end = (
    binary_prepared_start
    +
    len(
        LOCKED_BINARY_FEATURES
    )
)


if sparse.issparse(
    X_mcmed_external_transformed
):

    mcmed_binary_values = (
        X_mcmed_external_transformed[
            :,
            binary_prepared_start:
            binary_prepared_end,
        ]
        .toarray()
        .astype(
            np.float32
        )
    )

else:

    mcmed_binary_values = np.asarray(
        X_mcmed_external_transformed[
            :,
            binary_prepared_start:
            binary_prepared_end,
        ],
        dtype=np.float32,
    )


binary_raw_positions = [
    mcmed_raw_feature_position_lookup[
        feature_name
    ]
    for feature_name
    in LOCKED_BINARY_FEATURES
]


mcmed_grouped_raw_feature_values[
    :,
    binary_raw_positions,
] = mcmed_binary_values


assert mcmed_grouped_raw_feature_values.shape == (
    6707,
    2495,
)



mcmed_mean_absolute_shap = np.mean(
    np.abs(
        mcmed_grouped_raw_shap_values
    ),
    axis=0,
)


mcmed_mean_signed_shap = np.mean(
    mcmed_grouped_raw_shap_values,
    axis=0,
)


mcmed_positive_shap_fraction = np.mean(
    mcmed_grouped_raw_shap_values
    >
    0,
    axis=0,
)


mcmed_nonzero_shap_fraction = np.mean(
    ~np.isclose(
        mcmed_grouped_raw_shap_values,
        0.0,
        atol=1e-12,
        rtol=0.0,
    ),
    axis=0,
)


mcmed_grouped_raw_feature_importance = pd.DataFrame(
    {
        "raw_feature_position": np.arange(
            2495,
            dtype=int,
        ),
        "source_raw_feature": (
            LOCKED_RAW_FEATURE_ORDER
        ),
        "mean_absolute_shap": (
            mcmed_mean_absolute_shap
        ),
        "mean_signed_shap": (
            mcmed_mean_signed_shap
        ),
        "positive_shap_fraction": (
            mcmed_positive_shap_fraction
        ),
        "nonzero_shap_fraction": (
            mcmed_nonzero_shap_fraction
        ),
    }
)


mcmed_grouped_raw_feature_importance = (
    mcmed_grouped_raw_feature_importance
    .merge(
        locked_feature_metadata,
        on="source_raw_feature",
        how="left",
        validate="one_to_one",
    )
)


mcmed_prepared_column_counts = (
    mcmed_shap_prepared_feature_mapping
    .groupby(
        "source_raw_feature",
        observed=True,
    )
    .size()
    .rename(
        "prepared_columns_grouped"
    )
)


mcmed_grouped_raw_feature_importance = (
    mcmed_grouped_raw_feature_importance
    .merge(
        mcmed_prepared_column_counts,
        left_on="source_raw_feature",
        right_index=True,
        how="left",
        validate="one_to_one",
    )
)


mcmed_grouped_raw_feature_importance = (
    mcmed_grouped_raw_feature_importance
    .sort_values(
        "mean_absolute_shap",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


mcmed_grouped_raw_feature_importance.insert(
    0,
    "raw_feature_rank",
    np.arange(
        1,
        2496,
    ),
)


mcmed_total_mean_absolute_shap = float(
    mcmed_grouped_raw_feature_importance[
        "mean_absolute_shap"
    ].sum()
)


mcmed_grouped_raw_feature_importance[
    "global_importance_percent"
] = (
    100
    *
    mcmed_grouped_raw_feature_importance[
        "mean_absolute_shap"
    ]
    /
    mcmed_total_mean_absolute_shap
)


assert len(
    mcmed_grouped_raw_feature_importance
) == 2495



mcmed_shap_feature_family_summary = (
    mcmed_grouped_raw_feature_importance
    .groupby(
        [
            "family",
            "timing_stage",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        raw_features=(
            "source_raw_feature",
            "count",
        ),
        total_mean_absolute_shap=(
            "mean_absolute_shap",
            "sum",
        ),
    )
    .sort_values(
        "total_mean_absolute_shap",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


mcmed_shap_feature_family_summary[
    "importance_share_percent"
] = (
    100
    *
    mcmed_shap_feature_family_summary[
        "total_mean_absolute_shap"
    ]
    /
    mcmed_shap_feature_family_summary[
        "total_mean_absolute_shap"
    ].sum()
)



TOP_MCMED_SHAP_FEATURES = 30


mcmed_top_raw_features = (
    mcmed_grouped_raw_feature_importance
    .head(
        TOP_MCMED_SHAP_FEATURES
    )[
        "source_raw_feature"
    ]
    .tolist()
)


mcmed_top_raw_positions = np.asarray(
    [
        mcmed_raw_feature_position_lookup[
            feature_name
        ]
        for feature_name
        in mcmed_top_raw_features
    ],
    dtype=int,
)


mcmed_top_shap_values = (
    mcmed_grouped_raw_shap_values[
        :,
        mcmed_top_raw_positions,
    ]
)


mcmed_top_feature_values = (
    mcmed_grouped_raw_feature_values[
        :,
        mcmed_top_raw_positions,
    ]
)


mcmed_top_display_names = [
    (
        f"{feature_name} [grouped categorical]"
        if feature_name
        in LOCKED_S0_CATEGORICAL_FEATURES
        else feature_name
    )
    for feature_name
    in mcmed_top_raw_features
]


mcmed_grouped_shap_explanation = shap.Explanation(
    values=(
        mcmed_top_shap_values
    ),
    base_values=(
        mcmed_shap_bias
    ),
    data=(
        mcmed_top_feature_values
    ),
    feature_names=(
        mcmed_top_display_names
    ),
)



MCMED_SHAP_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook3"
)


MCMED_SHAP_ARRAY_DIR = (
    PROJECT_ROOT
    / "results"
    / "shap"
    / "notebook3"
)


MCMED_SHAP_FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "notebook3"
)


for directory in [
    MCMED_SHAP_TABLE_DIR,
    MCMED_SHAP_ARRAY_DIR,
    MCMED_SHAP_FIGURE_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


mcmed_static_shap_paths = {
    "prepared_mapping": (
        MCMED_SHAP_TABLE_DIR
        / "mcmed_shap_prepared_feature_mapping.csv"
    ),
    "categorical_code_mapping": (
        MCMED_SHAP_TABLE_DIR
        / "mcmed_shap_categorical_beeswarm_code_mapping.csv"
    ),
    "grouped_importance": (
        MCMED_SHAP_TABLE_DIR
        / "mcmed_shap_grouped_raw_feature_importance.csv"
    ),
    "family_summary": (
        MCMED_SHAP_TABLE_DIR
        / "mcmed_shap_feature_family_summary.csv"
    ),
    "prepared_values": (
        MCMED_SHAP_ARRAY_DIR
        / "mcmed_prepared_shap_values.npz"
    ),
    "grouped_values": (
        MCMED_SHAP_ARRAY_DIR
        / "mcmed_grouped_raw_shap_values.npz"
    ),
    "analysis_bundle": (
        MCMED_SHAP_ARRAY_DIR
        / "mcmed_grouped_shap_analysis_bundle.npz"
    ),
    "importance_png": (
        MCMED_SHAP_FIGURE_DIR
        / "07_mcmed_shap_grouped_feature_importance.png"
    ),
    "importance_pdf": (
        MCMED_SHAP_FIGURE_DIR
        / "07_mcmed_shap_grouped_feature_importance.pdf"
    ),
    "beeswarm_png": (
        MCMED_SHAP_FIGURE_DIR
        / "08_mcmed_shap_grouped_beeswarm.png"
    ),
    "beeswarm_pdf": (
        MCMED_SHAP_FIGURE_DIR
        / "08_mcmed_shap_grouped_beeswarm.pdf"
    ),
    "audit": (
        MCMED_SHAP_TABLE_DIR
        / "mcmed_shap_grouping_audit.csv"
    ),
}



mcmed_shap_prepared_feature_mapping.to_csv(
    mcmed_static_shap_paths[
        "prepared_mapping"
    ],
    index=False,
)


mcmed_categorical_beeswarm_code_mapping.to_csv(
    mcmed_static_shap_paths[
        "categorical_code_mapping"
    ],
    index=False,
)


mcmed_grouped_raw_feature_importance.to_csv(
    mcmed_static_shap_paths[
        "grouped_importance"
    ],
    index=False,
)


mcmed_shap_feature_family_summary.to_csv(
    mcmed_static_shap_paths[
        "family_summary"
    ],
    index=False,
)


mcmed_row_ids = np.asarray(
    mcmed_external.index.astype(str),
    dtype=str,
)


np.savez_compressed(
    mcmed_static_shap_paths[
        "prepared_values"
    ],
    shap_values=(
        mcmed_prepared_shap_values
    ),
    bias=(
        mcmed_shap_bias
    ),
    prepared_feature_names=np.asarray(
        mcmed_prepared_feature_order,
        dtype=str,
    ),
    row_ids=(
        mcmed_row_ids
    ),
)


np.savez_compressed(
    mcmed_static_shap_paths[
        "grouped_values"
    ],
    shap_values=(
        mcmed_grouped_raw_shap_values
    ),
    raw_feature_names=np.asarray(
        LOCKED_RAW_FEATURE_ORDER,
        dtype=str,
    ),
    row_ids=(
        mcmed_row_ids
    ),
)


np.savez_compressed(
    mcmed_static_shap_paths[
        "analysis_bundle"
    ],
    shap_values=(
        mcmed_grouped_raw_shap_values
    ),
    feature_values=(
        mcmed_grouped_raw_feature_values
    ),
    bias=(
        mcmed_shap_bias
    ),
    raw_feature_names=np.asarray(
        LOCKED_RAW_FEATURE_ORDER,
        dtype=str,
    ),
    row_ids=(
        mcmed_row_ids
    ),
    outcome_binary=(
        y_mcmed_external.to_numpy(
            dtype=np.int8
        )
    ),
)



mcmed_top_importance_plot = (
    mcmed_grouped_raw_feature_importance
    .head(
        TOP_MCMED_SHAP_FEATURES
    )
    .sort_values(
        "mean_absolute_shap",
        ascending=True,
    )
)


figure, axis = plt.subplots(
    figsize=(
        11,
        10,
    )
)


axis.barh(
    mcmed_top_importance_plot[
        "source_raw_feature"
    ],
    mcmed_top_importance_plot[
        "mean_absolute_shap"
    ],
)


axis.set_xlabel(
    "Mean absolute SHAP value"
)


axis.set_ylabel(
    "Grouped raw feature"
)


axis.set_title(
    "MC-MED external validation: global grouped SHAP importance"
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


figure.savefig(
    mcmed_static_shap_paths[
        "importance_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


figure.savefig(
    mcmed_static_shap_paths[
        "importance_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    figure
)



shap.plots.beeswarm(
    mcmed_grouped_shap_explanation,
    max_display=(
        TOP_MCMED_SHAP_FEATURES
    ),
    show=False,
)


mcmed_beeswarm_figure = plt.gcf()


mcmed_beeswarm_axis = plt.gca()


mcmed_beeswarm_axis.set_title(
    "MC-MED external validation: grouped raw-feature SHAP distribution"
)


mcmed_beeswarm_figure.tight_layout()


mcmed_beeswarm_figure.savefig(
    mcmed_static_shap_paths[
        "beeswarm_png"
    ],
    dpi=600,
    bbox_inches="tight",
)


mcmed_beeswarm_figure.savefig(
    mcmed_static_shap_paths[
        "beeswarm_pdf"
    ],
    bbox_inches="tight",
)


plt.show()


plt.close(
    mcmed_beeswarm_figure
)



mcmed_shap_elapsed_seconds = (
    time.perf_counter()
    -
    mcmed_shap_start_time
)


mcmed_shap_grouping_audit = pd.DataFrame(
    [
        {
            "dataset": "MC-MED",
            "rows": 6707,
            "prepared_model_columns": 2498,
            "grouped_raw_features": 2495,
            "numeric_features": 7,
            "one_hot_columns": 5,
            "grouped_categorical_features": 2,
            "binary_features": 2486,
            "shap_method": (
                "XGBoost native pred_contribs"
            ),
            "maximum_additivity_error": float(
                mcmed_shap_additivity_error.max()
            ),
            "maximum_probability_difference": float(
                mcmed_shap_probability_difference.max()
            ),
            "maximum_grouping_error": float(
                mcmed_grouping_error.max()
            ),
            "shap_seconds": float(
                mcmed_shap_elapsed_seconds
            ),
            "model_refitted": False,
            "preprocessor_refitted": False,
            "calibrator_used_for_shap": False,
            "threshold_used_for_shap": False,
        }
    ]
)


mcmed_shap_grouping_audit.to_csv(
    mcmed_static_shap_paths[
        "audit"
    ],
    index=False,
)



reloaded_mcmed_shap_bundle = np.load(
    mcmed_static_shap_paths[
        "analysis_bundle"
    ],
    allow_pickle=False,
)


assert reloaded_mcmed_shap_bundle[
    "shap_values"
].shape == (
    6707,
    2495,
)


assert reloaded_mcmed_shap_bundle[
    "feature_values"
].shape == (
    6707,
    2495,
)


assert (
    reloaded_mcmed_shap_bundle[
        "raw_feature_names"
    ].tolist()
    ==
    LOCKED_RAW_FEATURE_ORDER
)


assert (
    reloaded_mcmed_shap_bundle[
        "outcome_binary"
    ]
    ==
    y_mcmed_external.to_numpy(
        dtype=np.int8
    )
).all()


reloaded_mcmed_shap_bundle.close()



print(
    "Top 30 MC-MED grouped raw features by mean absolute SHAP:"
)


display(
    mcmed_grouped_raw_feature_importance[
        [
            "raw_feature_rank",
            "source_raw_feature",
            "prepared_columns_grouped",
            "family",
            "timing_stage",
            "mean_absolute_shap",
            "global_importance_percent",
            "mean_signed_shap",
            "positive_shap_fraction",
            "nonzero_shap_fraction",
        ]
    ]
    .head(
        30
    )
    .round(
        6
    )
)


print(
    "\nGrouped categorical feature audit:"
)


display(
    mcmed_grouped_raw_feature_importance.loc[
        mcmed_grouped_raw_feature_importance[
            "source_raw_feature"
        ].isin(
            LOCKED_S0_CATEGORICAL_FEATURES
        ),
        [
            "raw_feature_rank",
            "source_raw_feature",
            "prepared_columns_grouped",
            "mean_absolute_shap",
            "global_importance_percent",
            "mean_signed_shap",
        ],
    ].round(
        6
    )
)


print(
    "\nMC-MED SHAP grouping audit:"
)


display(
    mcmed_shap_grouping_audit.round(
        10
    )
)



print(
    "\nMC-MED STATIC GROUPED SHAP ANALYSIS COMPLETED"
)


print(
    "External rows explained:",
    "6,707",
)


print(
    "Prepared XGBoost columns:",
    "2,498",
)


print(
    "Grouped clinical features:",
    "2,495",
)


print(
    "sex_gender grouped:",
    "YES — 2 one-hot columns → 1 feature",
)


print(
    "arrival_transport_grouped grouped:",
    "YES — 3 one-hot columns → 1 feature",
)


print(
    "Patient-level SHAP totals preserved:",
    "YES",
)


print(
    "Native SHAP additivity verified:",
    "YES",
)


print(
    "Raw probability reproduction verified:",
    "YES",
)


print(
    "\nComplete 6,707 × 2,495 SHAP matrix saved:",
    "YES",
)


print(
    "Complete grouped feature-value matrix saved:",
    "YES",
)


print(
    "Self-contained robust-analysis bundle saved:",
    "YES",
)


print(
    "Full 2,495-feature importance table saved:",
    "YES",
)


print(
    "Static feature-importance figure saved:",
    "YES",
)


print(
    "Static grouped beeswarm saved:",
    "YES",
)


print(
    "\nModel refitted:",
    "NO",
)


print(
    "Preprocessor refitted:",
    "NO",
)


print(
    "Calibration used for SHAP:",
    "NO",
)


print(
    "Threshold used for SHAP:",
    "NO",
)

### Interactive MC-MED grouped SHAP visualisations

Interactive SHAP visualisations are generated from the previously calculated MC-MED grouped raw-feature SHAP matrix.

Numeric and binary features use the conventional SHAP-style continuous colour interpretation, with low feature values shown in blue and high feature values shown in red. The grouped categorical features `sex_gender` and `arrival_transport_grouped` use discrete category-specific colours because their category codes do not represent an ordered low-to-high scale.

All categorical SHAP contributions remain aggregated patient by patient from their corresponding one-hot encoded model columns. No SHAP values, predictions, model parameters, calibration parameters or thresholds are recalculated or modified.

In [ ]:



required_interactive_mcmed_shap_objects = [
    "PROJECT_ROOT",
    "mcmed_external",
    "X_mcmed_external_standardised",
    "mcmed_grouped_raw_shap_values",
    "mcmed_grouped_raw_feature_values",
    "mcmed_grouped_raw_feature_importance",
    "mcmed_external_predictions",
    "LOCKED_RAW_FEATURE_ORDER",
    "LOCKED_S0_CATEGORICAL_FEATURES",
]


missing_interactive_mcmed_shap_objects = [
    object_name
    for object_name
    in required_interactive_mcmed_shap_objects
    if object_name not in globals()
]


if missing_interactive_mcmed_shap_objects:

    raise RuntimeError(
        "Complete the static grouped MC-MED SHAP section "
        "before generating interactive SHAP plots.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_interactive_mcmed_shap_objects
        )
    )


assert mcmed_grouped_raw_shap_values.shape == (
    6707,
    2495,
)


assert mcmed_grouped_raw_feature_values.shape == (
    6707,
    2495,
)


assert len(
    LOCKED_RAW_FEATURE_ORDER
) == 2495


assert {
    "sex_gender",
    "arrival_transport_grouped",
}.issubset(
    set(
        LOCKED_RAW_FEATURE_ORDER
    )
)



SHAP_LOW_BLUE = "#008BFB"

SHAP_HIGH_RED = "#FF0051"


SHAP_CONTINUOUS_COLOUR_SCALE = [
    [
        0.0,
        SHAP_LOW_BLUE,
    ],
    [
        1.0,
        SHAP_HIGH_RED,
    ],
]



arrival_transport_colour_map = {
    "EMS": "#009E73",
    "OTHER_OR_UNKNOWN": "#E69F00",
    "SELF": "#8E44AD",
    "__MISSING__": "#7F8C8D",
    "UNKNOWN_TO_TRAINING_ENCODER": "#7F8C8D",
}


sex_gender_colour_map = {
    "F": "#7B2CBF",
    "M": "#F4A261",
    "__MISSING__": "#7F8C8D",
    "UNKNOWN_TO_TRAINING_ENCODER": "#7F8C8D",
}


categorical_colour_maps = {
    "arrival_transport_grouped": (
        arrival_transport_colour_map
    ),
    "sex_gender": (
        sex_gender_colour_map
    ),
}



MCMED_INTERACTIVE_SHAP_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "notebook3"
    / "interactive"
)


MCMED_INTERACTIVE_SHAP_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


mcmed_interactive_shap_paths = {
    "top30_summary": (
        MCMED_INTERACTIVE_SHAP_DIR
        / "09_mcmed_interactive_grouped_shap_top30.html"
    ),

    "arrival_transport": (
        MCMED_INTERACTIVE_SHAP_DIR
        / "10_mcmed_interactive_shap_arrival_transport.html"
    ),

    "sex_gender": (
        MCMED_INTERACTIVE_SHAP_DIR
        / "11_mcmed_interactive_shap_sex_gender.html"
    ),

    "colour_config": (
        MCMED_INTERACTIVE_SHAP_DIR
        / "mcmed_interactive_shap_colour_config.json"
    ),

    "categorical_plot_data": (
        MCMED_INTERACTIVE_SHAP_DIR
        / "mcmed_interactive_categorical_shap_plot_data.parquet"
    ),

    "artifact_manifest": (
        MCMED_INTERACTIVE_SHAP_DIR
        / "mcmed_interactive_shap_artifact_manifest.csv"
    ),
}



mcmed_interactive_colour_config = {
    "continuous_features": {
        "low_colour": SHAP_LOW_BLUE,
        "high_colour": SHAP_HIGH_RED,
        "interpretation": (
            "Blue = relatively low feature value; "
            "red = relatively high feature value"
        ),
    },

    "arrival_transport_grouped": (
        arrival_transport_colour_map
    ),

    "sex_gender": (
        sex_gender_colour_map
    ),
}


with mcmed_interactive_shap_paths[
    "colour_config"
].open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        mcmed_interactive_colour_config,
        file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )



mcmed_interactive_feature_position_lookup = {
    feature_name: feature_position
    for feature_position, feature_name
    in enumerate(
        LOCKED_RAW_FEATURE_ORDER
    )
}



N_MCMED_INTERACTIVE_ROWS = 6707


mcmed_hover_row_position = np.arange(
    N_MCMED_INTERACTIVE_ROWS,
    dtype=int,
)


mcmed_hover_outcome = np.where(
    mcmed_external_predictions[
        "outcome_binary"
    ]
    .astype(int)
    .to_numpy()
    ==
    1,
    "ADMITTED",
    "HOME",
)


mcmed_hover_raw_probability = (
    pd.to_numeric(
        mcmed_external_predictions[
            "raw_probability"
        ],
        errors="raise",
    )
    .to_numpy(
        dtype=float
    )
)


mcmed_hover_calibrated_probability = (
    pd.to_numeric(
        mcmed_external_predictions[
            "calibrated_probability"
        ],
        errors="raise",
    )
    .to_numpy(
        dtype=float
    )
)


assert len(
    mcmed_hover_outcome
) == 6707



MCMED_INTERACTIVE_SHAP_RANDOM_SEED = 42


mcmed_interactive_rng = np.random.default_rng(
    MCMED_INTERACTIVE_SHAP_RANDOM_SEED
)


def mcmed_create_vertical_jitter(
    size,
    width=0.28,
):

    return mcmed_interactive_rng.uniform(
        low=-width,
        high=width,
        size=size,
    )



def mcmed_normalise_feature_values(
    values,
):

    values = np.asarray(
        values,
        dtype=float,
    )


    finite_mask = np.isfinite(
        values
    )


    normalised_values = np.full(
        len(
            values
        ),
        0.5,
        dtype=float,
    )


    if not finite_mask.any():

        return normalised_values


    finite_values = values[
        finite_mask
    ]


    lower_value = float(
        np.percentile(
            finite_values,
            5,
        )
    )


    upper_value = float(
        np.percentile(
            finite_values,
            95,
        )
    )


    if np.isclose(
        lower_value,
        upper_value,
    ):

        minimum_value = float(
            finite_values.min()
        )


        maximum_value = float(
            finite_values.max()
        )


        if np.isclose(
            minimum_value,
            maximum_value,
        ):

            normalised_values[
                finite_mask
            ] = 0.5

        else:

            normalised_values[
                finite_mask
            ] = (
                finite_values
                -
                minimum_value
            ) / (
                maximum_value
                -
                minimum_value
            )


    else:

        clipped_values = np.clip(
            finite_values,
            lower_value,
            upper_value,
        )


        normalised_values[
            finite_mask
        ] = (
            clipped_values
            -
            lower_value
        ) / (
            upper_value
            -
            lower_value
        )


    return normalised_values



TOP_MCMED_INTERACTIVE_FEATURES = 30


mcmed_interactive_top_features = (
    mcmed_grouped_raw_feature_importance
    .sort_values(
        "mean_absolute_shap",
        ascending=False,
    )
    .head(
        TOP_MCMED_INTERACTIVE_FEATURES
    )[
        "source_raw_feature"
    ]
    .tolist()
)


assert len(
    mcmed_interactive_top_features
) == 30



mcmed_interactive_summary_figure = go.Figure()


continuous_colourbar_added = False


categorical_legend_entries_added = set()


for (
    feature_rank,
    feature_name,
) in enumerate(
    mcmed_interactive_top_features
):

    feature_position = (
        mcmed_interactive_feature_position_lookup[
            feature_name
        ]
    )


    feature_shap_values = (
        mcmed_grouped_raw_shap_values[
            :,
            feature_position,
        ]
    )


    feature_y_position = (
        TOP_MCMED_INTERACTIVE_FEATURES
        -
        1
        -
        feature_rank
    )


    jittered_y = (
        feature_y_position
        +
        mcmed_create_vertical_jitter(
            N_MCMED_INTERACTIVE_ROWS,
            width=0.27,
        )
    )



    if feature_name in categorical_colour_maps:

        categorical_values = (
            X_mcmed_external_standardised[
                feature_name
            ]
            .astype(
                "string"
            )
            .fillna(
                "__MISSING__"
            )
            .astype(str)
            .to_numpy()
        )


        preferred_category_order = list(
            categorical_colour_maps[
                feature_name
            ].keys()
        )


        observed_categories = [
            category_name
            for category_name
            in preferred_category_order
            if np.any(
                categorical_values
                ==
                category_name
            )
        ]


        additional_categories = sorted(
            set(
                categorical_values
            )
            -
            set(
                observed_categories
            )
        )


        observed_categories.extend(
            additional_categories
        )


        for category_name in observed_categories:

            category_mask = (
                categorical_values
                ==
                category_name
            )


            if not category_mask.any():

                continue


            category_colour = (
                categorical_colour_maps[
                    feature_name
                ]
                .get(
                    category_name,
                    "#7F8C8D",
                )
            )


            legend_key = (
                feature_name,
                category_name,
            )


            show_category_legend = (
                legend_key
                not in categorical_legend_entries_added
            )


            categorical_legend_entries_added.add(
                legend_key
            )


            category_custom_data = np.column_stack(
                [
                    np.full(
                        category_mask.sum(),
                        feature_name,
                        dtype=object,
                    ),

                    categorical_values[
                        category_mask
                    ],

                    mcmed_hover_outcome[
                        category_mask
                    ],

                    mcmed_hover_raw_probability[
                        category_mask
                    ],

                    mcmed_hover_calibrated_probability[
                        category_mask
                    ],
                ]
            )


            mcmed_interactive_summary_figure.add_trace(
                go.Scattergl(
                    x=feature_shap_values[
                        category_mask
                    ],

                    y=jittered_y[
                        category_mask
                    ],

                    mode="markers",

                    name=(
                        f"{feature_name}: "
                        f"{category_name}"
                    ),

                    legendgroup=(
                        f"{feature_name}_"
                        f"{category_name}"
                    ),

                    showlegend=(
                        show_category_legend
                    ),

                    marker={
                        "size": 5,
                        "opacity": 0.70,
                        "color": category_colour,
                    },

                    customdata=(
                        category_custom_data
                    ),

                    hovertemplate=(
                        "<b>%{customdata[0]}</b>"
                        "<br>Category: %{customdata[1]}"
                        "<br>Grouped SHAP: %{x:.5f}"
                        "<br>Outcome: %{customdata[2]}"
                        "<br>Raw probability: "
                        "%{customdata[3]:.4f}"
                        "<br>Calibrated probability: "
                        "%{customdata[4]:.4f}"
                        "<extra></extra>"
                    ),
                )
            )



    else:

        feature_values = (
            mcmed_grouped_raw_feature_values[
                :,
                feature_position,
            ]
        )


        normalised_colour_values = (
            mcmed_normalise_feature_values(
                feature_values
            )
        )


        continuous_custom_data = np.column_stack(
            [
                np.full(
                    N_MCMED_INTERACTIVE_ROWS,
                    feature_name,
                    dtype=object,
                ),

                feature_values,

                mcmed_hover_outcome,

                mcmed_hover_raw_probability,

                mcmed_hover_calibrated_probability,
            ]
        )


        marker_configuration = {
            "size": 5,

            "opacity": 0.70,

            "color": (
                normalised_colour_values
            ),

            "colorscale": (
                SHAP_CONTINUOUS_COLOUR_SCALE
            ),

            "cmin": 0.0,

            "cmax": 1.0,

            "showscale": (
                not continuous_colourbar_added
            ),
        }


        if not continuous_colourbar_added:

            marker_configuration[
                "colorbar"
            ] = {
                "title": {
                    "text": (
                        "Feature value"
                    ),
                },

                "tickvals": [
                    0.0,
                    1.0,
                ],

                "ticktext": [
                    "Low",
                    "High",
                ],

                "len": 0.45,

                "y": 0.75,
            }


            continuous_colourbar_added = True


        mcmed_interactive_summary_figure.add_trace(
            go.Scattergl(
                x=feature_shap_values,

                y=jittered_y,

                mode="markers",

                name=feature_name,

                showlegend=False,

                marker=(
                    marker_configuration
                ),

                customdata=(
                    continuous_custom_data
                ),

                hovertemplate=(
                    "<b>%{customdata[0]}</b>"
                    "<br>Feature value: "
                    "%{customdata[1]:.5g}"
                    "<br>Grouped SHAP: %{x:.5f}"
                    "<br>Outcome: %{customdata[2]}"
                    "<br>Raw probability: "
                    "%{customdata[3]:.4f}"
                    "<br>Calibrated probability: "
                    "%{customdata[4]:.4f}"
                    "<extra></extra>"
                ),
            )
        )


mcmed_interactive_summary_figure.add_vline(
    x=0.0,
    line_width=1.2,
    line_dash="dash",
    line_color="grey",
)


mcmed_interactive_summary_figure.update_layout(
    title={
        "text": (
            "MC-MED external validation: "
            "interactive grouped SHAP summary"
        ),
        "x": 0.5,
    },

    xaxis={
        "title": (
            "Grouped SHAP value "
            "(positive → ADMITTED, negative → HOME)"
        ),

        "zeroline": False,

        "showgrid": True,

        "gridcolor": (
            "rgba(128,128,128,0.15)"
        ),
    },

    yaxis={
        "title": "",

        "tickmode": "array",

        "tickvals": list(
            range(
                TOP_MCMED_INTERACTIVE_FEATURES
            )
        ),

        "ticktext": (
            mcmed_interactive_top_features[
                ::-1
            ]
        ),

        "range": [
            -1,
            TOP_MCMED_INTERACTIVE_FEATURES,
        ],

        "showgrid": False,
    },

    height=1050,

    width=1500,

    template="plotly_white",

    hovermode="closest",

    legend={
        "title": {
            "text": (
                "Categorical levels"
            ),
        },

        "orientation": "v",

        "x": 1.02,

        "y": 0.48,
    },

    margin={
        "l": 350,
        "r": 330,
        "t": 90,
        "b": 80,
    },
)


mcmed_interactive_summary_figure.write_html(
    mcmed_interactive_shap_paths[
        "top30_summary"
    ],

    include_plotlyjs=True,

    full_html=True,

    auto_open=False,
)


mcmed_interactive_summary_figure.show()



def create_mcmed_interactive_categorical_shap_plot(
    feature_name,
    colour_map,
    title,
    output_path,
):

    feature_position = (
        mcmed_interactive_feature_position_lookup[
            feature_name
        ]
    )


    feature_shap_values = (
        mcmed_grouped_raw_shap_values[
            :,
            feature_position,
        ]
    )


    categorical_values = (
        X_mcmed_external_standardised[
            feature_name
        ]
        .astype(
            "string"
        )
        .fillna(
            "__MISSING__"
        )
        .astype(str)
        .to_numpy()
    )


    categorical_figure = go.Figure()


    category_order = [
        category_name
        for category_name
        in colour_map.keys()
        if np.any(
            categorical_values
            ==
            category_name
        )
    ]


    additional_categories = sorted(
        set(
            categorical_values
        )
        -
        set(
            category_order
        )
    )


    category_order.extend(
        additional_categories
    )


    for category_name in category_order:

        category_mask = (
            categorical_values
            ==
            category_name
        )


        if not category_mask.any():

            continue


        category_colour = colour_map.get(
            category_name,
            "#7F8C8D",
        )


        category_jitter = (
            mcmed_create_vertical_jitter(
                category_mask.sum(),
                width=0.23,
            )
        )


        category_custom_data = np.column_stack(
            [
                mcmed_hover_outcome[
                    category_mask
                ],

                mcmed_hover_raw_probability[
                    category_mask
                ],

                mcmed_hover_calibrated_probability[
                    category_mask
                ],
            ]
        )


        categorical_figure.add_trace(
            go.Scattergl(
                x=feature_shap_values[
                    category_mask
                ],

                y=category_jitter,

                mode="markers",

                name=category_name,

                marker={
                    "size": 8,
                    "opacity": 0.72,
                    "color": category_colour,
                },

                customdata=(
                    category_custom_data
                ),

                hovertemplate=(
                    f"<b>{feature_name}</b>"
                    f"<br>Category: {category_name}"
                    "<br>Grouped SHAP: %{x:.5f}"
                    "<br>Outcome: %{customdata[0]}"
                    "<br>Raw probability: "
                    "%{customdata[1]:.4f}"
                    "<br>Calibrated probability: "
                    "%{customdata[2]:.4f}"
                    "<extra></extra>"
                ),
            )
        )


    categorical_figure.add_vline(
        x=0.0,
        line_width=1.2,
        line_dash="dash",
        line_color="grey",
    )


    categorical_figure.update_layout(
        title={
            "text": title,
            "x": 0.5,
        },

        xaxis={
            "title": (
                "Grouped SHAP value "
                "(positive → ADMITTED, negative → HOME)"
            ),

            "zeroline": False,

            "showgrid": True,

            "gridcolor": (
                "rgba(128,128,128,0.15)"
            ),
        },

        yaxis={
            "title": "",

            "showticklabels": False,

            "showgrid": False,

            "range": [
                -0.45,
                0.45,
            ],
        },

        template="plotly_white",

        hovermode="closest",

        height=560,

        width=1150,

        legend={
            "title": {
                "text": "Category",
            },

            "orientation": "h",

            "x": 0.5,

            "xanchor": "center",

            "y": -0.16,
        },

        margin={
            "l": 70,
            "r": 60,
            "t": 90,
            "b": 120,
        },
    )


    categorical_figure.write_html(
        output_path,

        include_plotlyjs=True,

        full_html=True,

        auto_open=False,
    )


    categorical_figure.show()


    return categorical_figure



mcmed_interactive_arrival_transport_figure = (
    create_mcmed_interactive_categorical_shap_plot(
        feature_name=(
            "arrival_transport_grouped"
        ),

        colour_map=(
            arrival_transport_colour_map
        ),

        title=(
            "MC-MED external validation: "
            "grouped SHAP for arrival transport"
        ),

        output_path=(
            mcmed_interactive_shap_paths[
                "arrival_transport"
            ]
        ),
    )
)



mcmed_interactive_sex_gender_figure = (
    create_mcmed_interactive_categorical_shap_plot(
        feature_name=(
            "sex_gender"
        ),

        colour_map=(
            sex_gender_colour_map
        ),

        title=(
            "MC-MED external validation: "
            "grouped SHAP for sex/gender"
        ),

        output_path=(
            mcmed_interactive_shap_paths[
                "sex_gender"
            ]
        ),
    )
)



mcmed_interactive_categorical_records = []


for feature_name in [
    "arrival_transport_grouped",
    "sex_gender",
]:

    feature_position = (
        mcmed_interactive_feature_position_lookup[
            feature_name
        ]
    )


    feature_categories = (
        X_mcmed_external_standardised[
            feature_name
        ]
        .astype(
            "string"
        )
        .fillna(
            "__MISSING__"
        )
        .astype(str)
        .to_numpy()
    )


    feature_shap_values = (
        mcmed_grouped_raw_shap_values[
            :,
            feature_position,
        ]
    )


    for row_position in range(
        N_MCMED_INTERACTIVE_ROWS
    ):

        mcmed_interactive_categorical_records.append(
            {
                "shap_row_position": int(
                    row_position
                ),

                "outcome": (
                    mcmed_hover_outcome[
                        row_position
                    ]
                ),

                "feature": (
                    feature_name
                ),

                "category": (
                    feature_categories[
                        row_position
                    ]
                ),

                "grouped_shap_value": float(
                    feature_shap_values[
                        row_position
                    ]
                ),

                "raw_probability": float(
                    mcmed_hover_raw_probability[
                        row_position
                    ]
                ),

                "calibrated_probability": float(
                    mcmed_hover_calibrated_probability[
                        row_position
                    ]
                ),
            }
        )


mcmed_interactive_categorical_shap_data = (
    pd.DataFrame(
        mcmed_interactive_categorical_records
    )
)


mcmed_interactive_categorical_shap_data.to_parquet(
    mcmed_interactive_shap_paths[
        "categorical_plot_data"
    ],

    index=False,

    engine="pyarrow",
)



def mcmed_interactive_shap_sha256(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(
        path
    )


    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                chunk_size
            ),
            b"",
        ):

            digest.update(
                chunk
            )


    return digest.hexdigest()


mcmed_interactive_artifact_names = [
    "top30_summary",
    "arrival_transport",
    "sex_gender",
    "colour_config",
    "categorical_plot_data",
]


mcmed_interactive_manifest_records = []


for artifact_name in (
    mcmed_interactive_artifact_names
):

    artifact_path = (
        mcmed_interactive_shap_paths[
            artifact_name
        ]
    )


    assert artifact_path.exists()


    mcmed_interactive_manifest_records.append(
        {
            "artifact": (
                artifact_name
            ),

            "filename": (
                artifact_path.name
            ),

            "directory": str(
                artifact_path.parent
            ),

            "size_bytes": int(
                artifact_path
                .stat()
                .st_size
            ),

            "sha256": (
                mcmed_interactive_shap_sha256(
                    artifact_path
                )
            ),
        }
    )


mcmed_interactive_shap_artifact_manifest = (
    pd.DataFrame(
        mcmed_interactive_manifest_records
    )
)


mcmed_interactive_shap_artifact_manifest.to_csv(
    mcmed_interactive_shap_paths[
        "artifact_manifest"
    ],

    index=False,
)



print(
    "MC-MED interactive SHAP artifact manifest:"
)


display(
    mcmed_interactive_shap_artifact_manifest
)


print(
    "\nMC-MED INTERACTIVE GROUPED SHAP COMPLETED"
)


print(
    "Rows visualised:",
    "6,707",
)


print(
    "Grouped raw-feature universe:",
    "2,495",
)


print(
    "Top interactive features:",
    TOP_MCMED_INTERACTIVE_FEATURES,
)


print(
    "\nContinuous feature colours:"
)


print(
    "  Low feature value:",
    SHAP_LOW_BLUE,
    "(blue)",
)


print(
    "  High feature value:",
    SHAP_HIGH_RED,
    "(red)",
)


print(
    "\nCategorical grouped colours:"
)


print(
    "arrival_transport_grouped:"
)


for category_name, colour_value in (
    arrival_transport_colour_map.items()
):

    print(
        f"  {category_name}: {colour_value}"
    )


print(
    "\nsex_gender:"
)


for category_name, colour_value in (
    sex_gender_colour_map.items()
):

    print(
        f"  {category_name}: {colour_value}"
    )


print(
    "\nsex_gender one-hot SHAP values grouped:",
    "YES",
)


print(
    "arrival_transport one-hot SHAP values grouped:",
    "YES",
)


print(
    "Interactive categorical plotting data saved:",
    "YES",
)


print(
    "Standalone HTML files saved:",
    "YES",
)


print(
    "Checksums saved:",
    "YES",
)


print(
    "\nSHAP recalculated:",
    "NO",
)


print(
    "Model refitted:",
    "NO",
)


print(
    "Preprocessor refitted:",
    "NO",
)


print(
    "Calibration changed:",
    "NO",
)


print(
    "Threshold changed:",
    "NO",
)

### MC-MED external calibration intercept and slope

External calibration is additionally quantified using the locked Platt-calibrated MC-MED probabilities.

Calibration-in-the-large and logistic recalibration parameters are estimated solely as external-validation diagnostics. An ideally calibrated model has calibration intercept 0 and calibration slope 1.

These estimates do not replace the transported Platt calibrator, do not alter predicted probabilities and are not used to modify the locked model or decision threshold.

In [ ]:



required_external_calibration_objects = [
    "PROJECT_ROOT",
    "y_mcmed_external",
    "mcmed_external_raw_probability",
    "mcmed_external_calibrated_probability",
    "LOCKED_DECISION_THRESHOLD",
]


missing_external_calibration_objects = [
    object_name
    for object_name
    in required_external_calibration_objects
    if object_name not in globals()
]


if missing_external_calibration_objects:

    raise RuntimeError(
        "Complete the locked MC-MED external prediction "
        "before calculating calibration diagnostics.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_external_calibration_objects
        )
    )



external_calibration_y = (
    y_mcmed_external
    .to_numpy(
        dtype=np.float64
    )
)


external_raw_probability_for_calibration = np.asarray(
    mcmed_external_raw_probability,
    dtype=np.float64,
)


external_calibrated_probability_for_calibration = np.asarray(
    mcmed_external_calibrated_probability,
    dtype=np.float64,
)


assert external_calibration_y.shape == (
    6707,
)


assert (
    external_raw_probability_for_calibration.shape
    ==
    (
        6707,
    )
)


assert (
    external_calibrated_probability_for_calibration.shape
    ==
    (
        6707,
    )
)


assert set(
    np.unique(
        external_calibration_y
    )
) == {
    0.0,
    1.0,
}



EXTERNAL_CALIBRATION_EPSILON = 1e-6


external_calibration_probability_clipped = np.clip(
    external_calibrated_probability_for_calibration,
    EXTERNAL_CALIBRATION_EPSILON,
    1.0
    -
    EXTERNAL_CALIBRATION_EPSILON,
)


external_calibration_logit = np.log(
    external_calibration_probability_clipped
    /
    (
        1.0
        -
        external_calibration_probability_clipped
    )
)


assert np.isfinite(
    external_calibration_logit
).all()



def stable_logistic(
    linear_predictor,
):

    linear_predictor = np.asarray(
        linear_predictor,
        dtype=np.float64,
    )


    clipped_predictor = np.clip(
        linear_predictor,
        -50.0,
        50.0,
    )


    return (
        1.0
        /
        (
            1.0
            +
            np.exp(
                -clipped_predictor
            )
        )
    )



def fit_calibration_in_the_large(
    outcome,
    predicted_logit,
    tolerance=1e-10,
    max_iterations=100,
):

    calibration_intercept = 0.0


    converged = False


    for iteration in range(
        1,
        max_iterations
        +
        1,
    ):

        linear_predictor = (
            predicted_logit
            +
            calibration_intercept
        )


        fitted_probability = stable_logistic(
            linear_predictor
        )


        score = float(
            np.sum(
                outcome
                -
                fitted_probability
            )
        )


        information = float(
            np.sum(
                fitted_probability
                *
                (
                    1.0
                    -
                    fitted_probability
                )
            )
        )


        if (
            not np.isfinite(
                information
            )
            or
            information
            <=
            0
        ):

            raise RuntimeError(
                "Invalid information matrix while fitting "
                "calibration-in-the-large."
            )


        parameter_update = (
            score
            /
            information
        )


        calibration_intercept += (
            parameter_update
        )


        if abs(
            parameter_update
        ) < tolerance:

            converged = True

            break


    if not converged:

        raise RuntimeError(
            "Calibration-in-the-large Newton fitting did "
            "not converge."
        )


    return {
        "intercept": float(
            calibration_intercept
        ),
        "iterations": int(
            iteration
        ),
        "converged": True,
    }


external_citl_fit = (
    fit_calibration_in_the_large(
        outcome=external_calibration_y,
        predicted_logit=external_calibration_logit,
    )
)


external_calibration_in_the_large = float(
    external_citl_fit[
        "intercept"
    ]
)



def fit_calibration_intercept_and_slope(
    outcome,
    predicted_logit,
    tolerance=1e-10,
    max_iterations=100,
):

    outcome = np.asarray(
        outcome,
        dtype=np.float64,
    )


    predicted_logit = np.asarray(
        predicted_logit,
        dtype=np.float64,
    )


    design_matrix = np.column_stack(
        [
            np.ones(
                len(
                    outcome
                ),
                dtype=np.float64,
            ),
            predicted_logit,
        ]
    )


    parameters = np.asarray(
        [
            0.0,
            1.0,
        ],
        dtype=np.float64,
    )


    converged = False


    for iteration in range(
        1,
        max_iterations
        +
        1,
    ):

        linear_predictor = (
            design_matrix
            @
            parameters
        )


        fitted_probability = stable_logistic(
            linear_predictor
        )


        residual = (
            outcome
            -
            fitted_probability
        )


        weights = (
            fitted_probability
            *
            (
                1.0
                -
                fitted_probability
            )
        )


        score = (
            design_matrix.T
            @
            residual
        )


        information_matrix = (
            design_matrix.T
            @
            (
                weights[
                    :,
                    None
                ]
                *
                design_matrix
            )
        )


        assert np.isfinite(
            information_matrix
        ).all()


        assert np.isfinite(
            score
        ).all()


        try:

            parameter_update = np.linalg.solve(
                information_matrix,
                score,
            )

        except np.linalg.LinAlgError as error:

            raise RuntimeError(
                "Calibration intercept/slope information "
                "matrix was singular."
            ) from error


        parameters = (
            parameters
            +
            parameter_update
        )


        if np.max(
            np.abs(
                parameter_update
            )
        ) < tolerance:

            converged = True

            break


    if not converged:

        raise RuntimeError(
            "Calibration intercept/slope Newton fitting "
            "did not converge."
        )


    return {
        "intercept": float(
            parameters[
                0
            ]
        ),
        "slope": float(
            parameters[
                1
            ]
        ),
        "iterations": int(
            iteration
        ),
        "converged": True,
    }


external_calibration_model_fit = (
    fit_calibration_intercept_and_slope(
        outcome=external_calibration_y,
        predicted_logit=external_calibration_logit,
    )
)


external_calibration_intercept = float(
    external_calibration_model_fit[
        "intercept"
    ]
)


external_calibration_slope = float(
    external_calibration_model_fit[
        "slope"
    ]
)


assert np.isfinite(
    external_calibration_in_the_large
)


assert np.isfinite(
    external_calibration_intercept
)


assert np.isfinite(
    external_calibration_slope
)



mcmed_external_calibration_diagnostics = pd.DataFrame(
    [
        {
            "dataset": (
                "MC-MED external validation"
            ),

            "rows": 6707,

            "observed_prevalence": float(
                external_calibration_y.mean()
            ),

            "mean_raw_probability": float(
                external_raw_probability_for_calibration.mean()
            ),

            "mean_calibrated_probability": float(
                external_calibrated_probability_for_calibration.mean()
            ),

            "mean_calibrated_probability_minus_prevalence": float(
                external_calibrated_probability_for_calibration.mean()
                -
                external_calibration_y.mean()
            ),

            "calibration_in_the_large": (
                external_calibration_in_the_large
            ),

            "diagnostic_calibration_intercept": (
                external_calibration_intercept
            ),

            "diagnostic_calibration_slope": (
                external_calibration_slope
            ),

            "ideal_calibration_in_the_large": 0.0,

            "ideal_intercept": 0.0,

            "ideal_slope": 1.0,

            "citl_iterations": int(
                external_citl_fit[
                    "iterations"
                ]
            ),

            "intercept_slope_iterations": int(
                external_calibration_model_fit[
                    "iterations"
                ]
            ),

            "predictions_modified": False,

            "platt_calibrator_refitted": False,

            "threshold_changed": False,
        }
    ]
)



(
    external_raw_fraction_positive,
    external_raw_mean_predicted,
) = calibration_curve(
    external_calibration_y,
    external_raw_probability_for_calibration,
    n_bins=10,
    strategy="quantile",
)


(
    external_calibrated_fraction_positive,
    external_calibrated_mean_predicted,
) = calibration_curve(
    external_calibration_y,
    external_calibrated_probability_for_calibration,
    n_bins=10,
    strategy="quantile",
)


mcmed_external_calibration_curve_data = pd.concat(
    [
        pd.DataFrame(
            {
                "probability_type": (
                    "Raw XGBoost"
                ),

                "bin": np.arange(
                    1,
                    len(
                        external_raw_mean_predicted
                    )
                    +
                    1,
                ),

                "mean_predicted_probability": (
                    external_raw_mean_predicted
                ),

                "observed_admission_proportion": (
                    external_raw_fraction_positive
                ),
            }
        ),

        pd.DataFrame(
            {
                "probability_type": (
                    "Platt calibrated"
                ),

                "bin": np.arange(
                    1,
                    len(
                        external_calibrated_mean_predicted
                    )
                    +
                    1,
                ),

                "mean_predicted_probability": (
                    external_calibrated_mean_predicted
                ),

                "observed_admission_proportion": (
                    external_calibrated_fraction_positive
                ),
            }
        ),
    ],

    ignore_index=True,
)



NOTEBOOK3_FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "notebook3"
)


NOTEBOOK3_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook3"
)


NOTEBOOK3_FIGURE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


NOTEBOOK3_TABLE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


CALIBRATION_DIAGNOSTIC_PNG = (
    NOTEBOOK3_FIGURE_DIR
    / "12_mcmed_external_calibration_diagnostics.png"
)


CALIBRATION_DIAGNOSTIC_PDF = (
    NOTEBOOK3_FIGURE_DIR
    / "12_mcmed_external_calibration_diagnostics.pdf"
)


CALIBRATION_DIAGNOSTIC_TABLE = (
    NOTEBOOK3_TABLE_DIR
    / "mcmed_external_calibration_diagnostics.csv"
)


CALIBRATION_CURVE_DATA_TABLE = (
    NOTEBOOK3_TABLE_DIR
    / "mcmed_external_calibration_curve_data.csv"
)



figure, axis = plt.subplots(
    figsize=(
        8,
        7,
    )
)


axis.plot(
    [
        0,
        1,
    ],
    [
        0,
        1,
    ],
    linestyle="--",
    linewidth=1.2,
    label="Perfect calibration",
)


axis.plot(
    external_raw_mean_predicted,
    external_raw_fraction_positive,
    marker="o",
    linewidth=2,
    label="Raw XGBoost",
)


axis.plot(
    external_calibrated_mean_predicted,
    external_calibrated_fraction_positive,
    marker="o",
    linewidth=2,
    label="Platt calibrated",
)


diagnostic_text = (
    "External calibration diagnostics\n"
    f"Calibration-in-the-large = "
    f"{external_calibration_in_the_large:.3f}\n"
    f"Calibration intercept = "
    f"{external_calibration_intercept:.3f}\n"
    f"Calibration slope = "
    f"{external_calibration_slope:.3f}"
)


axis.text(
    0.04,
    0.96,
    diagnostic_text,
    transform=axis.transAxes,
    va="top",
    ha="left",
)


axis.set_xlim(
    0,
    1,
)


axis.set_ylim(
    0,
    1,
)


axis.set_xlabel(
    "Mean predicted admission probability"
)


axis.set_ylabel(
    "Observed admission proportion"
)


axis.set_title(
    "MC-MED external calibration"
)


axis.legend(
    frameon=False,
    loc="lower right",
)


axis.spines[
    "top"
].set_visible(
    False
)


axis.spines[
    "right"
].set_visible(
    False
)


figure.tight_layout()


figure.savefig(
    CALIBRATION_DIAGNOSTIC_PNG,
    dpi=600,
    bbox_inches="tight",
)


figure.savefig(
    CALIBRATION_DIAGNOSTIC_PDF,
    bbox_inches="tight",
)


plt.show()


plt.close(
    figure
)



mcmed_external_calibration_diagnostics.to_csv(
    CALIBRATION_DIAGNOSTIC_TABLE,
    index=False,
)


mcmed_external_calibration_curve_data.to_csv(
    CALIBRATION_CURVE_DATA_TABLE,
    index=False,
)



print(
    "MC-MED external calibration diagnostics:"
)


display(
    mcmed_external_calibration_diagnostics.round(
        6
    )
)


print(
    "\nMC-MED EXTERNAL CALIBRATION DIAGNOSTIC COMPLETED"
)


print(
    "Calibration-in-the-large:",
    round(
        external_calibration_in_the_large,
        6,
    ),
)


print(
    "Diagnostic calibration intercept:",
    round(
        external_calibration_intercept,
        6,
    ),
)


print(
    "Diagnostic calibration slope:",
    round(
        external_calibration_slope,
        6,
    ),
)


print(
    "\nIdeal calibration-in-the-large:",
    0,
)


print(
    "Ideal calibration intercept:",
    0,
)


print(
    "Ideal calibration slope:",
    1,
)


print(
    "\nCalibration curve plotted:",
    "YES",
)


print(
    "Raw XGBoost curve plotted:",
    "YES",
)


print(
    "Platt-calibrated curve plotted:",
    "YES",
)


print(
    "Perfect-calibration diagonal plotted:",
    "YES",
)


print(
    "Slope/intercept annotated on figure:",
    "YES",
)


print(
    "\nPredictions modified:",
    "NO",
)


print(
    "Platt calibrator refitted:",
    "NO",
)


print(
    "Threshold changed:",
    "NO",
)


print(
    "Calibration diagnostics table saved:",
    "YES",
)


print(
    "Calibration-curve data saved:",
    "YES",
)


print(
    "600-dpi PNG saved:",
    "YES",
)


print(
    "Vector PDF saved:",
    "YES",
)

## Final Notebook 3 completion and Notebook 4 hand-off

Notebook 3 is closed by verifying the complete external-validation artifact set and constructing a reproducible hand-off for subsequent cross-dataset SHAP and clinician-comparison analyses.

The hand-off verifies that MIMIC-IV and MC-MED use the identical ordered universe of 2,495 grouped clinically interpretable features. Their patient-level grouped SHAP matrices, corresponding feature-value matrices, outcomes, predictions, model provenance, external-validation metrics, bootstrap confidence intervals and calibration diagnostics are referenced by explicit paths and SHA-256 checksums.

The canonical trained XGBoost pipeline and Platt calibrator remain the locked Notebook 2 artifacts and are not duplicated. Notebook 3 contains only external-validation results and references to those immutable model artifacts.

No model fitting, preprocessing fitting, recalibration, threshold modification, feature selection or SHAP recalculation is performed in this final section.

In [ ]:



required_final_notebook3_objects = [
    "PROJECT_ROOT",
    "LOCKED_RAW_FEATURE_ORDER",
    "LOCKED_FEATURE_ORDER_SHA256",
    "LOCKED_DECISION_THRESHOLD",
    "LOCKED_CALIBRATOR_INTERCEPT",
    "LOCKED_CALIBRATOR_SLOPE",
    "notebook2_handoff_paths",
]


missing_final_notebook3_objects = [
    object_name
    for object_name
    in required_final_notebook3_objects
    if object_name not in globals()
]


if missing_final_notebook3_objects:

    raise RuntimeError(
        "Notebook 3 cannot be closed because required "
        "handoff objects are missing.\n\n"
        "Missing objects:\n"
        +
        "\n".join(
            missing_final_notebook3_objects
        )
    )


assert len(
    LOCKED_RAW_FEATURE_ORDER
) == 2495


assert len(
    set(
        LOCKED_RAW_FEATURE_ORDER
    )
) == 2495


assert np.isclose(
    LOCKED_DECISION_THRESHOLD,
    0.282,
    rtol=0.0,
    atol=1e-12,
)



NOTEBOOK3_TABLE_DIR = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook3"
)


NOTEBOOK3_PREDICTION_DIR = (
    PROJECT_ROOT
    / "results"
    / "predictions"
    / "notebook3"
)


NOTEBOOK3_BOOTSTRAP_DIR = (
    PROJECT_ROOT
    / "results"
    / "bootstrap"
    / "notebook3"
)


NOTEBOOK3_FIGURE_DIR = (
    PROJECT_ROOT
    / "results"
    / "figures"
    / "notebook3"
)


NOTEBOOK3_SHAP_DIR = (
    PROJECT_ROOT
    / "results"
    / "shap"
    / "notebook3"
)


NOTEBOOK3_CONFIG_DIR = (
    PROJECT_ROOT
    / "results"
    / "config"
    / "notebook3"
)


NOTEBOOK3_HANDOFF_DIR = (
    PROJECT_ROOT
    / "results"
    / "handoff"
    / "notebook3"
)


NOTEBOOK3_HANDOFF_DIR.mkdir(
    parents=True,
    exist_ok=True,
)



NOTEBOOK2_MIMIC_SHAP_BUNDLE = (
    PROJECT_ROOT
    / "results"
    / "shap"
    / "notebook2"
    / "internal_test_grouped_shap_analysis_bundle.npz"
)


NOTEBOOK2_CLINICIAN_UNIVERSE = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook2"
    / "clinician_comparison_feature_universe.csv"
)


NOTEBOOK2_SHAP_HANDOFF_CONFIG = (
    PROJECT_ROOT
    / "results"
    / "shap"
    / "notebook2"
    / "notebook2_shap_handoff_config.json"
)


NOTEBOOK2_SHAP_HANDOFF_MANIFEST = (
    PROJECT_ROOT
    / "results"
    / "tables"
    / "notebook2"
    / "notebook2_shap_handoff_manifest.csv"
)



NOTEBOOK3_MCMED_PREDICTIONS = (
    NOTEBOOK3_PREDICTION_DIR
    / "mcmed_external_predictions.parquet"
)


NOTEBOOK3_MCMED_METRICS_CI = (
    NOTEBOOK3_TABLE_DIR
    / "mcmed_external_metrics_with_ci.csv"
)


NOTEBOOK3_BOOTSTRAP_DISTRIBUTION = (
    NOTEBOOK3_BOOTSTRAP_DIR
    / "mcmed_external_bootstrap_distribution.parquet"
)


NOTEBOOK3_PERFORMANCE_COMPARISON = (
    NOTEBOOK3_TABLE_DIR
    / "internal_vs_external_performance_comparison.csv"
)


NOTEBOOK3_PUBLICATION_COMPARISON = (
    NOTEBOOK3_TABLE_DIR
    / "internal_vs_external_publication_table.csv"
)


NOTEBOOK3_PROBABILITY_TRANSPORT = (
    NOTEBOOK3_TABLE_DIR
    / "internal_vs_external_probability_transport.csv"
)


NOTEBOOK3_CALIBRATION_DIAGNOSTICS = (
    NOTEBOOK3_TABLE_DIR
    / "mcmed_external_calibration_diagnostics.csv"
)


NOTEBOOK3_CALIBRATION_CURVE_DATA = (
    NOTEBOOK3_TABLE_DIR
    / "mcmed_external_calibration_curve_data.csv"
)


NOTEBOOK3_MCMED_SHAP_BUNDLE = (
    NOTEBOOK3_SHAP_DIR
    / "mcmed_grouped_shap_analysis_bundle.npz"
)


NOTEBOOK3_MCMED_GROUPED_SHAP = (
    NOTEBOOK3_SHAP_DIR
    / "mcmed_grouped_raw_shap_values.npz"
)


NOTEBOOK3_MCMED_PREPARED_SHAP = (
    NOTEBOOK3_SHAP_DIR
    / "mcmed_prepared_shap_values.npz"
)


NOTEBOOK3_MCMED_SHAP_IMPORTANCE = (
    NOTEBOOK3_TABLE_DIR
    / "mcmed_shap_grouped_raw_feature_importance.csv"
)


NOTEBOOK3_MCMED_SHAP_MAPPING = (
    NOTEBOOK3_TABLE_DIR
    / "mcmed_shap_prepared_feature_mapping.csv"
)


NOTEBOOK3_MCMED_CATEGORY_MAPPING = (
    NOTEBOOK3_TABLE_DIR
    / "mcmed_shap_categorical_beeswarm_code_mapping.csv"
)


NOTEBOOK3_MCMED_SHAP_AUDIT = (
    NOTEBOOK3_TABLE_DIR
    / "mcmed_shap_grouping_audit.csv"
)


NOTEBOOK3_MODEL_REFERENCE = (
    NOTEBOOK3_TABLE_DIR
    / "notebook3_locked_model_reference.csv"
)


NOTEBOOK3_EXTERNAL_CONFIG = (
    NOTEBOOK3_CONFIG_DIR
    / "notebook3_external_validation_config.json"
)



critical_handoff_paths = {
    "locked_pipeline": (
        notebook2_handoff_paths[
            "locked_pipeline"
        ]
    ),

    "platt_calibrator": (
        notebook2_handoff_paths[
            "platt_calibrator"
        ]
    ),

    "locked_model_package": (
        notebook2_handoff_paths[
            "locked_model_package"
        ]
    ),

    "notebook2_model_config": (
        notebook2_handoff_paths[
            "model_config"
        ]
    ),

    "mimic_grouped_shap_bundle": (
        NOTEBOOK2_MIMIC_SHAP_BUNDLE
    ),

    "clinician_comparison_feature_universe": (
        NOTEBOOK2_CLINICIAN_UNIVERSE
    ),

    "notebook2_shap_handoff_config": (
        NOTEBOOK2_SHAP_HANDOFF_CONFIG
    ),

    "notebook2_shap_handoff_manifest": (
        NOTEBOOK2_SHAP_HANDOFF_MANIFEST
    ),

    "mcmed_external_predictions": (
        NOTEBOOK3_MCMED_PREDICTIONS
    ),

    "mcmed_external_metrics_with_ci": (
        NOTEBOOK3_MCMED_METRICS_CI
    ),

    "mcmed_external_bootstrap_distribution": (
        NOTEBOOK3_BOOTSTRAP_DISTRIBUTION
    ),

    "internal_vs_external_performance": (
        NOTEBOOK3_PERFORMANCE_COMPARISON
    ),

    "internal_vs_external_publication_table": (
        NOTEBOOK3_PUBLICATION_COMPARISON
    ),

    "internal_vs_external_probability_transport": (
        NOTEBOOK3_PROBABILITY_TRANSPORT
    ),

    "mcmed_external_calibration_diagnostics": (
        NOTEBOOK3_CALIBRATION_DIAGNOSTICS
    ),

    "mcmed_external_calibration_curve_data": (
        NOTEBOOK3_CALIBRATION_CURVE_DATA
    ),

    "mcmed_grouped_shap_bundle": (
        NOTEBOOK3_MCMED_SHAP_BUNDLE
    ),

    "mcmed_grouped_shap_values": (
        NOTEBOOK3_MCMED_GROUPED_SHAP
    ),

    "mcmed_prepared_shap_values": (
        NOTEBOOK3_MCMED_PREPARED_SHAP
    ),

    "mcmed_grouped_shap_importance": (
        NOTEBOOK3_MCMED_SHAP_IMPORTANCE
    ),

    "mcmed_shap_prepared_mapping": (
        NOTEBOOK3_MCMED_SHAP_MAPPING
    ),

    "mcmed_shap_category_mapping": (
        NOTEBOOK3_MCMED_CATEGORY_MAPPING
    ),

    "mcmed_shap_grouping_audit": (
        NOTEBOOK3_MCMED_SHAP_AUDIT
    ),

    "notebook3_locked_model_reference": (
        NOTEBOOK3_MODEL_REFERENCE
    ),

    "notebook3_external_validation_config": (
        NOTEBOOK3_EXTERNAL_CONFIG
    ),
}


critical_handoff_existence_records = []


for artifact_name, artifact_path in (
    critical_handoff_paths.items()
):

    critical_handoff_existence_records.append(
        {
            "artifact": artifact_name,

            "exists": (
                artifact_path.exists()
            ),

            "path": str(
                artifact_path
            ),
        }
    )


notebook3_critical_handoff_audit = pd.DataFrame(
    critical_handoff_existence_records
)


assert notebook3_critical_handoff_audit[
    "exists"
].all(), (
    "One or more critical Notebook 3 / Notebook 4 "
    "hand-off artifacts are missing."
)



def notebook3_final_sha256(
    path,
    chunk_size=1024 * 1024,
):

    path = Path(
        path
    )


    digest = hashlib.sha256()


    with path.open(
        "rb"
    ) as file:

        for chunk in iter(
            lambda: file.read(
                chunk_size
            ),
            b"",
        ):

            digest.update(
                chunk
            )


    return digest.hexdigest()



mimic_shap_bundle = np.load(
    NOTEBOOK2_MIMIC_SHAP_BUNDLE,
    allow_pickle=False,
)


mcmed_shap_bundle = np.load(
    NOTEBOOK3_MCMED_SHAP_BUNDLE,
    allow_pickle=False,
)


mimic_shap_values_shape = (
    mimic_shap_bundle[
        "shap_values"
    ].shape
)


mimic_feature_values_shape = (
    mimic_shap_bundle[
        "feature_values"
    ].shape
)


mcmed_shap_values_shape = (
    mcmed_shap_bundle[
        "shap_values"
    ].shape
)


mcmed_feature_values_shape = (
    mcmed_shap_bundle[
        "feature_values"
    ].shape
)


assert mimic_shap_values_shape == (
    1851,
    2495,
)


assert mimic_feature_values_shape == (
    1851,
    2495,
)


assert mcmed_shap_values_shape == (
    6707,
    2495,
)


assert mcmed_feature_values_shape == (
    6707,
    2495,
)


mimic_shap_feature_names = (
    mimic_shap_bundle[
        "raw_feature_names"
    ]
    .astype(str)
    .tolist()
)


mcmed_shap_feature_names = (
    mcmed_shap_bundle[
        "raw_feature_names"
    ]
    .astype(str)
    .tolist()
)


assert len(
    mimic_shap_feature_names
) == 2495


assert len(
    mcmed_shap_feature_names
) == 2495


assert (
    mimic_shap_feature_names
    ==
    LOCKED_RAW_FEATURE_ORDER
), (
    "MIMIC grouped SHAP feature order does not match the "
    "locked raw-feature order."
)


assert (
    mcmed_shap_feature_names
    ==
    LOCKED_RAW_FEATURE_ORDER
), (
    "MC-MED grouped SHAP feature order does not match the "
    "locked raw-feature order."
)


assert (
    mimic_shap_feature_names
    ==
    mcmed_shap_feature_names
), (
    "MIMIC and MC-MED grouped SHAP feature orders differ."
)


handoff_feature_order_sha256 = hashlib.sha256(
    "\n".join(
        mimic_shap_feature_names
    )
    .encode(
        "utf-8"
    )
).hexdigest()


assert (
    handoff_feature_order_sha256
    ==
    LOCKED_FEATURE_ORDER_SHA256
)


mimic_handoff_outcome = (
    mimic_shap_bundle[
        "outcome_binary"
    ]
)


mcmed_handoff_outcome = (
    mcmed_shap_bundle[
        "outcome_binary"
    ]
)


assert mimic_handoff_outcome.shape == (
    1851,
)


assert mcmed_handoff_outcome.shape == (
    6707,
)


assert set(
    np.unique(
        mimic_handoff_outcome
    )
) == {
    0,
    1,
}


assert set(
    np.unique(
        mcmed_handoff_outcome
    )
) == {
    0,
    1,
}


mimic_shap_bundle.close()

mcmed_shap_bundle.close()



notebook4_clinician_feature_universe = (
    pd.read_csv(
        NOTEBOOK2_CLINICIAN_UNIVERSE
    )
)


assert len(
    notebook4_clinician_feature_universe
) == 2495


assert (
    notebook4_clinician_feature_universe[
        "source_raw_feature"
    ]
    .astype(str)
    .tolist()
    ==
    LOCKED_RAW_FEATURE_ORDER
)



notebook3_shap_cross_dataset_handoff_audit = (
    pd.DataFrame(
        [
            {
                "mimic_rows": 1851,

                "mcmed_rows": 6707,

                "mimic_grouped_features": 2495,

                "mcmed_grouped_features": 2495,

                "same_feature_names": True,

                "same_feature_order": True,

                "feature_order_sha256": (
                    handoff_feature_order_sha256
                ),

                "mimic_feature_values_saved": True,

                "mcmed_feature_values_saved": True,

                "mimic_outcomes_saved": True,

                "mcmed_outcomes_saved": True,

                "clinician_comparison_universe_rows": 2495,

                "ready_for_distance_from_zero": True,

                "ready_for_min_max_shap": True,

                "ready_for_percentile_filtering": True,

                "ready_for_support_filtering": True,

                "ready_for_stability_analysis": True,

                "ready_for_cross_dataset_shap_comparison": True,

                "ready_for_clinician_comparison": True,
            }
        ]
    )
)



notebook3_artifact_directories = [
    NOTEBOOK3_TABLE_DIR,
    NOTEBOOK3_PREDICTION_DIR,
    NOTEBOOK3_BOOTSTRAP_DIR,
    NOTEBOOK3_FIGURE_DIR,
    NOTEBOOK3_SHAP_DIR,
    NOTEBOOK3_CONFIG_DIR,
]


notebook3_artifact_records = []


for artifact_directory in (
    notebook3_artifact_directories
):

    if not artifact_directory.exists():

        continue


    for artifact_path in sorted(
        path
        for path
        in artifact_directory.rglob(
            "*"
        )
        if path.is_file()
    ):

        notebook3_artifact_records.append(
            {
                "relative_path": str(
                    artifact_path.relative_to(
                        PROJECT_ROOT
                    )
                ),

                "filename": (
                    artifact_path.name
                ),

                "directory": str(
                    artifact_path.parent
                ),

                "suffix": (
                    artifact_path.suffix.lower()
                ),

                "size_bytes": int(
                    artifact_path
                    .stat()
                    .st_size
                ),

                "sha256": (
                    notebook3_final_sha256(
                        artifact_path
                    )
                ),
            }
        )


notebook3_final_artifact_manifest = (
    pd.DataFrame(
        notebook3_artifact_records
    )
)


assert not notebook3_final_artifact_manifest.empty


assert notebook3_final_artifact_manifest[
    "relative_path"
].is_unique



notebook2_reference_records = []


for artifact_name, artifact_path in {
    "locked_xgboost_pipeline": (
        notebook2_handoff_paths[
            "locked_pipeline"
        ]
    ),

    "platt_calibrator": (
        notebook2_handoff_paths[
            "platt_calibrator"
        ]
    ),

    "locked_model_package": (
        notebook2_handoff_paths[
            "locked_model_package"
        ]
    ),

    "notebook2_model_config": (
        notebook2_handoff_paths[
            "model_config"
        ]
    ),

    "mimic_grouped_shap_bundle": (
        NOTEBOOK2_MIMIC_SHAP_BUNDLE
    ),

    "clinician_comparison_feature_universe": (
        NOTEBOOK2_CLINICIAN_UNIVERSE
    ),

    "notebook2_shap_handoff_config": (
        NOTEBOOK2_SHAP_HANDOFF_CONFIG
    ),
}.items():

    assert artifact_path.exists()


    notebook2_reference_records.append(
        {
            "artifact": (
                artifact_name
            ),

            "canonical_path": str(
                artifact_path
            ),

            "relative_path": str(
                artifact_path.relative_to(
                    PROJECT_ROOT
                )
            ),

            "size_bytes": int(
                artifact_path
                .stat()
                .st_size
            ),

            "sha256": (
                notebook3_final_sha256(
                    artifact_path
                )
            ),

            "source_notebook": (
                "Notebook 2"
            ),

            "duplicated_in_notebook3": False,
        }
    )


notebook3_notebook2_reference_manifest = (
    pd.DataFrame(
        notebook2_reference_records
    )
)



notebook4_handoff_config = {
    "handoff_source": (
        "Notebook 3 — external validation"
    ),

    "next_stage": (
        "Notebook 4 — robust cross-dataset SHAP and "
        "clinician comparison"
    ),

    "model": {
        "algorithm": (
            "XGBoost"
        ),

        "calibration": (
            "Platt scaling"
        ),

        "locked_threshold": float(
            LOCKED_DECISION_THRESHOLD
        ),

        "platt_intercept": float(
            LOCKED_CALIBRATOR_INTERCEPT
        ),

        "platt_slope": float(
            LOCKED_CALIBRATOR_SLOPE
        ),

        "canonical_pipeline": str(
            notebook2_handoff_paths[
                "locked_pipeline"
            ]
        ),

        "canonical_calibrator": str(
            notebook2_handoff_paths[
                "platt_calibrator"
            ]
        ),

        "canonical_model_package": str(
            notebook2_handoff_paths[
                "locked_model_package"
            ]
        ),

        "model_source_notebook": (
            "Notebook 2"
        ),
    },

    "feature_universe": {
        "clinically_interpretable_features": 2495,

        "technical_prepared_model_columns": 2498,

        "primary_analysis_representation": (
            "grouped raw features"
        ),

        "feature_order_sha256": (
            handoff_feature_order_sha256
        ),

        "categorical_grouping": {
            "sex_gender": (
                "2 one-hot columns grouped to 1 raw feature"
            ),

            "arrival_transport_grouped": (
                "3 one-hot columns grouped to 1 raw feature"
            ),
        },
    },

    "mimic_internal_test": {
        "rows": 1851,

        "grouped_features": 2495,

        "grouped_shap_bundle": str(
            NOTEBOOK2_MIMIC_SHAP_BUNDLE
        ),

        "predictions": str(
            PROJECT_ROOT
            / "results"
            / "predictions"
            / "notebook2"
            / "internal_test_predictions.parquet"
        ),

        "metrics_with_ci": str(
            PROJECT_ROOT
            / "results"
            / "tables"
            / "notebook2"
            / "internal_test_metrics_with_ci.csv"
        ),
    },

    "mcmed_external": {
        "rows": 6707,

        "home": 4872,

        "admitted": 1835,

        "grouped_features": 2495,

        "grouped_shap_bundle": str(
            NOTEBOOK3_MCMED_SHAP_BUNDLE
        ),

        "predictions": str(
            NOTEBOOK3_MCMED_PREDICTIONS
        ),

        "metrics_with_ci": str(
            NOTEBOOK3_MCMED_METRICS_CI
        ),

        "bootstrap_distribution": str(
            NOTEBOOK3_BOOTSTRAP_DISTRIBUTION
        ),

        "calibration_diagnostics": str(
            NOTEBOOK3_CALIBRATION_DIAGNOSTICS
        ),
    },

    "cross_dataset": {
        "performance_comparison": str(
            NOTEBOOK3_PERFORMANCE_COMPARISON
        ),

        "publication_table": str(
            NOTEBOOK3_PUBLICATION_COMPARISON
        ),

        "probability_transport": str(
            NOTEBOOK3_PROBABILITY_TRANSPORT
        ),

        "same_ordered_2495_feature_universe": True,
    },

    "clinician_comparison": {
        "feature_universe": str(
            NOTEBOOK2_CLINICIAN_UNIVERSE
        ),

        "features": 2495,

        "comparison_representation": (
            "grouped raw features"
        ),
    },

    "supported_notebook4_analyses": [
        "Patient-level SHAP distance from zero",

        "Feature-level minimum and maximum SHAP distance",

        "SHAP percentile filtering",

        "Positive and negative SHAP direction analysis",

        "Minimum-support filtering",

        "SHAP stability analysis",

        "MIMIC-IV versus MC-MED SHAP comparison",

        "Common-important-feature analysis",

        "Dataset-specific important-feature analysis",

        "Clinician-versus-model importance comparison",

        "Novel-feature identification",
    ],

    "analysis_boundaries": {
        "model_development_complete": True,

        "internal_test_spent": True,

        "mcmed_external_evaluated": True,

        "model_refitting_allowed": False,

        "recalibration_allowed": False,

        "threshold_reselection_allowed": False,

        "feature_reselection_for_primary_model_allowed": False,

        "shap_analysis_is_descriptive": True,
    },

    "created_utc": (
        datetime.now(
            timezone.utc
        ).isoformat()
    ),
}



NOTEBOOK4_HANDOFF_CONFIG_PATH = (
    NOTEBOOK3_HANDOFF_DIR
    / "notebook3_to_notebook4_handoff.json"
)


NOTEBOOK3_FINAL_ARTIFACT_MANIFEST_PATH = (
    NOTEBOOK3_HANDOFF_DIR
    / "notebook3_final_artifact_manifest.csv"
)


NOTEBOOK3_NOTEBOOK2_REFERENCE_MANIFEST_PATH = (
    NOTEBOOK3_HANDOFF_DIR
    / "notebook3_notebook2_reference_manifest.csv"
)


NOTEBOOK3_SHAP_CROSS_DATASET_AUDIT_PATH = (
    NOTEBOOK3_HANDOFF_DIR
    / "notebook3_shap_cross_dataset_handoff_audit.csv"
)


NOTEBOOK3_COMPLETION_AUDIT_PATH = (
    NOTEBOOK3_HANDOFF_DIR
    / "notebook3_completion_audit.csv"
)


NOTEBOOK3_HANDOFF_CHECKSUM_MANIFEST_PATH = (
    NOTEBOOK3_HANDOFF_DIR
    / "notebook3_handoff_checksum_manifest.csv"
)



with NOTEBOOK4_HANDOFF_CONFIG_PATH.open(
    "w",
    encoding="utf-8",
) as file:

    json.dump(
        notebook4_handoff_config,
        file,
        indent=2,
        ensure_ascii=False,
        sort_keys=True,
    )


notebook3_final_artifact_manifest.to_csv(
    NOTEBOOK3_FINAL_ARTIFACT_MANIFEST_PATH,
    index=False,
)


notebook3_notebook2_reference_manifest.to_csv(
    NOTEBOOK3_NOTEBOOK2_REFERENCE_MANIFEST_PATH,
    index=False,
)


notebook3_shap_cross_dataset_handoff_audit.to_csv(
    NOTEBOOK3_SHAP_CROSS_DATASET_AUDIT_PATH,
    index=False,
)



notebook3_completion_audit = pd.DataFrame(
    [
        {
            "notebook": (
                "03_external_validation.ipynb"
            ),

            "algorithm": (
                "XGBoost"
            ),

            "calibration": (
                "Platt scaling"
            ),

            "locked_threshold": float(
                LOCKED_DECISION_THRESHOLD
            ),

            "raw_feature_universe": 2495,

            "prepared_model_columns": 2498,

            "mimic_internal_rows": 1851,

            "mcmed_external_rows": 6707,

            "mcmed_home": 4872,

            "mcmed_admitted": 1835,

            "mcmed_predictions_completed": True,

            "bootstrap_95_ci_completed": True,

            "internal_external_comparison_completed": True,

            "external_calibration_completed": True,

            "external_calibration_slope_intercept_completed": True,

            "static_mcmed_shap_completed": True,

            "interactive_mcmed_shap_completed": True,

            "mimic_mcmed_shap_alignment_verified": True,

            "clinician_comparison_universe_verified": True,

            "notebook4_handoff_created": True,

            "canonical_model_remains_notebook2": True,

            "model_refitted": False,

            "preprocessor_refitted": False,

            "calibrator_refitted": False,

            "threshold_changed": False,

            "notebook3_complete": True,
        }
    ]
)


notebook3_completion_audit.to_csv(
    NOTEBOOK3_COMPLETION_AUDIT_PATH,
    index=False,
)



final_handoff_files = {
    "notebook4_handoff_config": (
        NOTEBOOK4_HANDOFF_CONFIG_PATH
    ),

    "notebook3_final_artifact_manifest": (
        NOTEBOOK3_FINAL_ARTIFACT_MANIFEST_PATH
    ),

    "notebook2_reference_manifest": (
        NOTEBOOK3_NOTEBOOK2_REFERENCE_MANIFEST_PATH
    ),

    "shap_cross_dataset_audit": (
        NOTEBOOK3_SHAP_CROSS_DATASET_AUDIT_PATH
    ),

    "notebook3_completion_audit": (
        NOTEBOOK3_COMPLETION_AUDIT_PATH
    ),
}


final_handoff_checksum_records = []


for artifact_name, artifact_path in (
    final_handoff_files.items()
):

    assert artifact_path.exists()


    final_handoff_checksum_records.append(
        {
            "artifact": artifact_name,

            "filename": (
                artifact_path.name
            ),

            "path": str(
                artifact_path
            ),

            "size_bytes": int(
                artifact_path
                .stat()
                .st_size
            ),

            "sha256": (
                notebook3_final_sha256(
                    artifact_path
                )
            ),
        }
    )


notebook3_handoff_checksum_manifest = pd.DataFrame(
    final_handoff_checksum_records
)


notebook3_handoff_checksum_manifest.to_csv(
    NOTEBOOK3_HANDOFF_CHECKSUM_MANIFEST_PATH,
    index=False,
)



with NOTEBOOK4_HANDOFF_CONFIG_PATH.open(
    "r",
    encoding="utf-8",
) as file:

    reloaded_notebook4_handoff = (
        json.load(
            file
        )
    )


assert (
    reloaded_notebook4_handoff[
        "feature_universe"
    ][
        "clinically_interpretable_features"
    ]
    ==
    2495
)


assert (
    reloaded_notebook4_handoff[
        "feature_universe"
    ][
        "feature_order_sha256"
    ]
    ==
    LOCKED_FEATURE_ORDER_SHA256
)


assert (
    reloaded_notebook4_handoff[
        "mimic_internal_test"
    ][
        "rows"
    ]
    ==
    1851
)


assert (
    reloaded_notebook4_handoff[
        "mcmed_external"
    ][
        "rows"
    ]
    ==
    6707
)


assert (
    reloaded_notebook4_handoff[
        "cross_dataset"
    ][
        "same_ordered_2495_feature_universe"
    ]
    is True
)


assert (
    reloaded_notebook4_handoff[
        "analysis_boundaries"
    ][
        "model_refitting_allowed"
    ]
    is False
)



print(
    "Cross-dataset SHAP hand-off audit:"
)


display(
    notebook3_shap_cross_dataset_handoff_audit
)


print(
    "\nNotebook 3 completion audit:"
)


display(
    notebook3_completion_audit
)


print(
    "\nFinal Notebook 3 hand-off checksum manifest:"
)


display(
    notebook3_handoff_checksum_manifest
)



print(
    "\n=================================================="
)


print(
    "NOTEBOOK 3 COMPLETED"
)


print(
    "=================================================="
)


print(
    "\nLOCKED MODEL"
)


print(
    "Algorithm:",
    "XGBoost",
)


print(
    "Calibration:",
    "Platt scaling",
)


print(
    "Threshold:",
    round(
        float(
            LOCKED_DECISION_THRESHOLD
        ),
        6,
    ),
)


print(
    "Model source:",
    "Notebook 2",
)


print(
    "Model duplicated in Notebook 3:",
    "NO",
)


print(
    "\nEXTERNAL VALIDATION"
)


print(
    "MC-MED rows:",
    "6,707",
)


print(
    "HOME:",
    "4,872",
)


print(
    "ADMITTED:",
    "1,835",
)


print(
    "Bootstrap 95% CI:",
    "COMPLETED — 2,000 replicates",
)


print(
    "External calibration:",
    "COMPLETED",
)


print(
    "Calibration intercept/slope:",
    "COMPLETED",
)


print(
    "\nSHAP"
)


print(
    "MIMIC grouped SHAP:",
    "1,851 × 2,495",
)


print(
    "MC-MED grouped SHAP:",
    "6,707 × 2,495",
)


print(
    "Feature names identical:",
    "YES",
)


print(
    "Feature order identical:",
    "YES",
)


print(
    "Feature-order SHA-256:",
    handoff_feature_order_sha256,
)


print(
    "sex_gender grouped:",
    "YES",
)


print(
    "arrival_transport_grouped grouped:",
    "YES",
)


print(
    "\nNOTEBOOK 4 READINESS"
)


print(
    "Distance-from-zero analysis ready:",
    "YES",
)


print(
    "Min/max SHAP analysis ready:",
    "YES",
)


print(
    "Percentile filtering ready:",
    "YES",
)


print(
    "Support filtering ready:",
    "YES",
)


print(
    "Stability analysis ready:",
    "YES",
)


print(
    "Cross-dataset SHAP comparison ready:",
    "YES",
)


print(
    "Clinician comparison ready:",
    "YES",
)


print(
    "\nPRIMARY NOTEBOOK 4 HAND-OFF:"
)


print(
    NOTEBOOK4_HANDOFF_CONFIG_PATH
)


print(
    "\nFinal Notebook 3 artifact manifest:"
)


print(
    NOTEBOOK3_FINAL_ARTIFACT_MANIFEST_PATH
)


print(
    "\nNotebook 3 model refitted:",
    "NO",
)


print(
    "Preprocessor refitted:",
    "NO",
)


print(
    "Calibrator refitted:",
    "NO",
)


print(
    "Threshold changed:",
    "NO",
)


print(
    "\nNOTEBOOK 3 STATUS:",
    "COMPLETE",
)